# Material Entity Resolution & Duplicate Detection — Cleaned Version

Bu sürümde açık kod hataları, eski/başarısız deneme blokları ve birebir tekrarlar temizlendi. PLM kimlikleri case-sensitive tutuldu; yalnızca tamamen sayısal Excel kodlarındaki `.0` son eki normalize edildi. PLM mal grubu eşlemesindeki eski yanlış yaklaşım kaldırıldı, duplicate-detection akışı tek bir güvenli implementasyona indirildi ve INCONCEPT/status odaklı yan analiz çıkarıldı.

> Not: Kaynak Excel dosyaları bu notebook ile birlikte yüklenmediği için çıktı hücreleri temizlendi. Notebook'u kendi veri klasörünüzde **Restart Kernel + Run All** ile çalıştırın.


In [1]:
#Upload dataset
import pandas as pd
import numpy as np
from pathlib import Path

#define the folder containing the project files
#data_folder = Path(".") choses the current path
data_folder = Path(r"C:\Users\EMRE.YILMAZ\Desktop\Kumaş PLM Tespit")

#list all excel and csv files in the data folder
files = list(data_folder.glob("*.xlsx")) + list(data_folder.glob("*.csv"))

for file in files:
    print(file.name)



Fabric_Mandatory_Fields.xlsx
Kum.xlsx
MaterialAttributes_full.xlsx
PLM Çalışması - Mara.xlsx
PLM_Codes.xlsx
PLM_MULTI_VAL_CHAR.xlsx
SATNR.xlsx
MaterialAttributes_full.csv


In [2]:
material_attributes_path = data_folder / "MaterialAttributes_full.xlsx"
sap_plm_mapping_path = data_folder / "PLM Çalışması - Mara.xlsx"
plm_codes_path = data_folder / "PLM_Codes.xlsx"

#check sheet names in each excel file
print(pd.ExcelFile(material_attributes_path).sheet_names)
print(pd.ExcelFile(sap_plm_mapping_path).sheet_names)
print(pd.ExcelFile(plm_codes_path).sheet_names)

['Data']
['GenericArticle']
['Data']


In [3]:
#Load datasets
material_attributes = pd.read_excel(
    material_attributes_path,
    sheet_name="Data"
)
sap_plm_mapping = pd.read_excel(
    sap_plm_mapping_path,
    sheet_name="GenericArticle"
)
plm_codes = pd.read_excel(
    plm_codes_path,
    sheet_name="Data"
)

In [4]:
#Check data set dimensions
print("Material attributes:", material_attributes.shape)
print("SAP-PLM Mapping:", sap_plm_mapping.shape)
print("PLM Codes :", plm_codes.shape)

Material attributes: (98248, 4)
SAP-PLM Mapping: (21205, 6)
PLM Codes : (32361, 27)


In [5]:
#display column names
print("\nMaterial Attributes columns:")
print(material_attributes.columns.tolist())

print("\nSAP-PLM Mapping columns:")
print(sap_plm_mapping.columns.tolist())

print("\nPLM Codes columns:")
print(plm_codes.columns.tolist())


Material Attributes columns:
['Malzeme', 'Dahili krkt.no.', 'Sayaç', 'Karakteristik değeri']

SAP-PLM Mapping columns:
['Malzeme', 'Mal grubu', 'Temel ölçü birimi', 'Türkçe malzeme açıklaması', 'Türkçe malzeme Uzun açıklaması', 'PLM Kodu']

PLM Codes columns:
['PLM Kodu', 'Örme alt tipi', 'İlmek uzunluğu/50 iğne', 'Malzeme statüs', '1.İplik numarası örme', '2.İplik numarası örm', '3.İplik numarası örme', 'Kumaş ağırlığı', 'Kumaş ağırlığı birimi', 'Kumaş eni', 'Kumaş eni birimi', 'Pus', 'Fine', 'Mal grubu', 'Türkçe malzeme açıklaması', 'Kumaş tipi', 'Çözgü sıklığı (tel /in', 'Atkı sıklığı (tel /inç)', 'Dokuma tipi', '1.Çözgü iplik numarası', '1.Atkı iplik numarası', '2.Çözgü iplik numarası', '2.Atkı iplik numarası', '3.Çözgü iplik numarası', '3.Atkı iplik numaras', 'Malzeme ingilizce adı', 'Malzeme Türkçe Adı']


In [6]:
display(material_attributes.head())
display(sap_plm_mapping.head())
display(plm_codes.head())

,Malzeme,Dahili krkt.no.,Sayaç,Karakteristik değeri
0,1020000012,FABRICSTRUCTURETNAME,1,TWILL
1,1020000012,WEAVETYPE,1,LCWWEAVETYPELIST8
2,1020000012,WARPDENSITY,1,44
3,1020000012,WEFTDENSITY,1,33
4,1020000012,WARPYARN1TYPEID,1,RING


,Malzeme,Mal grubu,Temel ölçü birimi,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu
0,1020000000,DOKUMA,M,GABARD.30/20 HAM,NaN,NaN
1,1020000001,DOKUMA,M,40/1 80 TEL HAM POPLIN,NaN,NaN
2,1020000002,ORME,KG,30/1 SUPREM FU YA,30/1 SUPREM 180CM 60PAM 40PES 1LS-PETROL,NaN
3,1020000003,ORME,KG,30/1 PENYE INTERLOK HAM,30/1 PENYE INTERLOK HAM,NaN
4,1020000004,ORME,KG,30/1 LYC RIBANA HAM,30/1 LYC RIBANA HAM,NaN


,PLM Kodu,Örme alt tipi,İlmek uzunluğu/50 iğne,Malzeme statüs,1.İplik numarası örme,2.İplik numarası örm,3.İplik numarası örme,Kumaş ağırlığı,Kumaş ağırlığı birimi,Kumaş eni,...,Atkı sıklığı (tel /inç),Dokuma tipi,1.Çözgü iplik numarası,1.Atkı iplik numarası,2.Çözgü iplik numarası,2.Atkı iplik numarası,3.Çözgü iplik numarası,3.Atkı iplik numaras,Malzeme ingilizce adı,Malzeme Türkçe Adı
0,300001,NaN,NaN,INCONCEPT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,300011,CIRCULAR,NaN,INCONCEPT,NaN,NaN,NaN,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,043021 MATERIAL NAME 1-1 (FOR,FLAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,LCWWEAVETYPELIST1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,043021 MATERIAL NUMBER 1-1 (FO,FLAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,LCWWEAVETYPELIST1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100000000,NaN,NaN,INCONCEPT,8,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Check unique key counts
print("Unique SAP materials in attributes:",
      material_attributes["Malzeme"].nunique())

print("Unique SAP materials in mapping:",
      sap_plm_mapping["Malzeme"].nunique())

print("Unique PLM codes in mapping:",
      sap_plm_mapping["PLM Kodu"].nunique())

print("Unique PLM codes in PLM master:",
      plm_codes["PLM Kodu"].nunique())

Unique SAP materials in attributes: 9544
Unique SAP materials in mapping: 21205
Unique PLM codes in mapping: 4012
Unique PLM codes in PLM master: 32361


In [8]:
# Calculate PLM mapping coverage
total_materials = sap_plm_mapping["Malzeme"].nunique()
materials_with_plm = (
    sap_plm_mapping
    .dropna(subset=["PLM Kodu"])["Malzeme"]
    .nunique()
)

materials_without_plm = total_materials - materials_with_plm

print("Total SAP materials:", total_materials)
print("SAP materials with PLM:", materials_with_plm)
print("SAP materials without PLM:", materials_without_plm)

print(
    "PLM coverage:",
    round(materials_with_plm / total_materials * 100, 2),
    "%"
)


Total SAP materials: 21205
SAP materials with PLM: 5646
SAP materials without PLM: 15559
PLM coverage: 26.63 %


In [9]:
# Count how many SAP materials are linked to the same PLM code
sap_count_per_plm = (
    sap_plm_mapping
    .dropna(subset=["PLM Kodu"])
    .groupby("PLM Kodu")["Malzeme"]
    .nunique()
    .sort_values(ascending=False)
)

display(sap_count_per_plm.head(20))

plm_sap_distribution = (
    sap_count_per_plm
    .value_counts()
    .sort_index()
    .rename_axis("sap_material_count")
    .reset_index(name="plm_count")
)

display(plm_sap_distribution.head(20))


PLM Kodu
12075.0     130
4667.0       78
2822.0       52
4823.0       45
2529.0       40
2533.0       32
2574.0       32
3859.0       25
171694.0     24
352974.0     23
26823.0      21
16678.0      20
21683.0      18
354310.0     17
123575.0     17
23061.0      15
4641.0       14
29694.0      14
12063.0      13
172235.0     12
Name: Malzeme, dtype: int64

,sap_material_count,plm_count
0,1,3541
1,2,226
2,3,102
3,4,53
4,5,22
5,6,14
6,7,4
7,8,15
8,9,6
9,10,4


In [10]:
# Standardize SAP material IDs before comparing datasets
sap_plm_mapping["material_id"] = (
    pd.to_numeric(sap_plm_mapping["Malzeme"],errors="coerce")
    .astype("Int64")
    .astype("string")
)

material_attributes["material_id"] = (
    pd.to_numeric(material_attributes["Malzeme"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

# Compare SAP material coverage between datasets
mapping_materials = set(
    sap_plm_mapping["material_id"].dropna()
)

attribute_materials = set(
    material_attributes["material_id"].dropna()
)

common_materials = mapping_materials & attribute_materials

print("SAP materials in mapping:", len(mapping_materials))
print("SAP materials with attributes:", len(attribute_materials))
print("Common SAP materials:", len(common_materials))

print(
    "Attribute coverage:",
    round(len(common_materials)/len(mapping_materials)*100,2),"%"
)

SAP materials in mapping: 21205
SAP materials with attributes: 9544
Common SAP materials: 9510
Attribute coverage: 44.85 %


In [11]:
# Check attribute coverage separately for materials with and without PLM codes
# First collapse the mapping table to one row per SAP material so that repeated
# mapping rows cannot inflate the number of materials with attributes.
material_attribute_flags = (
    sap_plm_mapping
    .assign(
        has_plm=lambda df: df["PLM Kodu"].notna(),
        has_attributes=lambda df: df["material_id"].isin(attribute_materials)
    )
    .groupby("material_id", as_index=False)
    .agg(
        has_plm=("has_plm", "max"),
        has_attributes=("has_attributes", "max")
    )
)

attribute_coverage = (
    material_attribute_flags
    .groupby("has_plm")
    .agg(
        material_count=("material_id", "nunique"),
        materials_with_attributes=("has_attributes", "sum")
    )
)

attribute_coverage["attribute_coverage_pct"] = (
    attribute_coverage["materials_with_attributes"]
    / attribute_coverage["material_count"]
    * 100
).round(2)

display(attribute_coverage)


,material_count,materials_with_attributes,attribute_coverage_pct
has_plm,,,
False,15559,5990,38.50
True,5646,3520,62.35


In [12]:
#Identify SAP materials that do not exist in the attribute dataset
missing_attribute_materials = (
    sap_plm_mapping[
        ~sap_plm_mapping["material_id"].isin(attribute_materials)
    ].copy()
)

print(
    "SAP materials without attribute records:",
    missing_attribute_materials["material_id"].nunique()
)

display(missing_attribute_materials.sort_values("Malzeme", ascending=False).head(20))

SAP materials without attribute records: 11695


,Malzeme,Mal grubu,Temel ölçü birimi,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu,material_id
16526,1020017322,DOKUMA,M,FLİP FLOP SAYA,NaN,NaN,1020017322
16512,1020017308,DOKUMA,M,4MM SUNGER + POLIBOND LAMINASYON,NaN,NaN,1020017308
16502,1020017297,DOKUMA,M,110 GR KUM BEJİ +4 MM EVA,NaN,NaN,1020017297
16499,1020017294,DENIM,M,90 GR KIRIK BEYAZ LACOSTE+4 MM EVA,NaN,NaN,1020017294
16490,1020017285,DOKUMA,M,500 GR PELUŞ,NaN,NaN,1020017285
16489,1020017284,DOKUMA,M,"110GR LACOSTE KUM BEJİ+3,26 DNS SNG+TELA",NaN,NaN,1020017284
16485,1020017280,DOKUMA,M,PVC ASTAR,NaN,NaN,1020017280
16458,1020017243,DOKUMA,M,FLİP FLOP SAYA,NaN,NaN,1020017243
16456,1020017241,DOKUMA,M,6815 YEŞİL VELAR VİTAŞ,NaN,NaN,1020017241
16445,1020017230,DENIM,M,90 GR BEYAZ LACOSTE+4 MM EVA,NaN,NaN,1020017230


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

#combine short and long descriptions

descriptions = (
    missing_attribute_materials["Türkçe malzeme açıklaması"]
    .fillna("")
    .astype(str)
    + " "
    + missing_attribute_materials["Türkçe malzeme Uzun açıklaması"]
    .fillna("")
    .astype(str)
).str.upper()

# extract the most frequent 1-3 word expressions
vectorizer = CountVectorizer(
    ngram_range=(1,3),
    min_df=5
)

ngram_matrix = vectorizer.fit_transform(descriptions)

ngram_counts = ngram_matrix.sum(axis=0).A1
ngram_terms = vectorizer.get_feature_names_out()

frequent_expressions = (
    pd.DataFrame(
        {
            "expression": ngram_terms,
            "count": ngram_counts
        }
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(frequent_expressions.head(100))

,expression,count
0,30,7778
1,pes,7511
2,pam,6161
3,100,6128
4,duz,4466
...,...,...
95,vual,394
96,popl,392
97,pam sup,391
98,gabard,391


In [14]:
# Remove expressions that contain only numbers
frequent_expressions_clean = frequent_expressions[
    ~frequent_expressions["expression"].str.fullmatch(r"[\d\s]+")
].copy()

# Separate single-word terms
top_unigrams = frequent_expressions_clean[
    frequent_expressions_clean["expression"].str.split().str.len() == 1
].head(100)
# Show up to 100 rows without truncation
pd.set_option("display.max_rows", 100)

display(top_unigrams)


,expression,count
1,pes,7511
2,pam,6161
4,duz,4466
5,penye,2998
6,lyc,2584
10,co,1984
11,gr,1925
13,sup,1727
15,50d,1440
16,75d,1421


In [15]:
# Extract two- and three-word expressions
top_phrases = frequent_expressions_clean[
    frequent_expressions_clean["expression"].str.split().str.len().between(2, 3)
].head(100)
# Show up to 100 rows without truncation
pd.set_option("display.max_rows", 100)

display(top_phrases)

,expression,count
7,100 pes,2533
8,30 penye,2283
14,pam pes,1630
17,100 pam,1307
19,duz bya,1293
21,sup duz,1072
24,100 co,985
25,pam lyc,976
31,30 pny,857
32,pes lyc,812


In [16]:
# Define keywords that may indicate non-fabric materials
non_fabric_search_terms = [
    "SAYA",
    "AYAKKABI",
    "DERI",
    "DERİ",
    "TABAN",
    "TOKA",
    "FERMUAR",
    "DUGME",
    "DÜĞME",
    "AKSESUAR",
    "LASTIK",
    "LASTİK",
    "KORDON",
    "IP",
    "İP",
    "BANT"
]

# Count descriptions containing each keyword
keyword_counts = []

for term in non_fabric_search_terms:
    count = descriptions.str.contains(
        term,
        case=False,
        regex=False
    ).sum()

    keyword_counts.append({
        "keyword": term,
        "material_count": count
    })

keyword_summary = (
    pd.DataFrame(keyword_counts)
    .sort_values("material_count", ascending=False)
)

display(keyword_summary)

,keyword,material_count
13,IP,857
14,İP,317
15,BANT,91
4,TABAN,77
3,DERİ,63
2,DERI,56
0,SAYA,46
1,AYAKKABI,10
11,LASTİK,9
7,DUGME,0


In [17]:
# Inspect materials containing "SAYA"
display(
    missing_attribute_materials[
        descriptions.str.contains(
            "SAYA",
            case=False,
            regex=False
        )
    ][
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması",
            "PLM Kodu"
        ]
    ].head(50)
)

,Malzeme,Mal grubu,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu
10237,1020010496,DOKUMA,COLOMBES BEIGE SAYA TABAN MOSTRA SALPA,NaN,NaN
10239,1020010498,DOKUMA,BARBAROS SAYA TABAN MOSTRA SALPA,NaN,NaN
10240,1020010499,DOKUMA,GORAY SAYA TABAN MOSTRA SALPA,NaN,NaN
10249,1020010510,DOKUMA,PESTEN SAYA TABAN MOSTRA SALPA,NaN,NaN
10250,1020010511,DOKUMA,FORTE NAVY SAYA TABAN MOSTRA SALPA,NaN,NaN
10251,1020010512,DOKUMA,FORTE GREY SAYA TABAN MOSTRA SALPA,NaN,NaN
10252,1020010513,DOKUMA,NİCE SAYA TABAN MOSTRA SALPA,NaN,NaN
10253,1020010514,DOKUMA,KANO SAYA TABAN MOSTRA SALPA,NaN,NaN
10254,1020010515,DOKUMA,OLİVER SAYA TABAN MOSTRA SALPA,NaN,NaN
10255,1020010516,DOKUMA,SCOLA SAYA TABAN MOSTRA SALPA,NaN,NaN


In [18]:
import re
import unicodedata

# Combine material descriptions
missing_attribute_materials["combined_description"] = (
    missing_attribute_materials["Türkçe malzeme açıklaması"]
    .fillna("")
    .astype(str)
    + " "
    + missing_attribute_materials["Türkçe malzeme Uzun açıklaması"]
    .fillna("")
    .astype(str)
).str.upper()

In [19]:
# Define strong footwear upper indicators
saya_keywords = [
    "SAYA",
    "AYAKKABI",
    "MOSTRA",
    "SALPA"
]

saya_pattern = r"\b(?:" + "|".join(saya_keywords) + r")\b"

saya_candidates = missing_attribute_materials[
    missing_attribute_materials["combined_description"]
    .str.contains(saya_pattern, regex=True, na=False)
].copy()

print("Saya candidates:", len(saya_candidates))

display(
    saya_candidates[
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "PLM Kodu"
        ]
    ].head(50)
)

Saya candidates: 66


,Malzeme,Mal grubu,Türkçe malzeme açıklaması,PLM Kodu
10237,1020010496,DOKUMA,COLOMBES BEIGE SAYA TABAN MOSTRA SALPA,NaN
10239,1020010498,DOKUMA,BARBAROS SAYA TABAN MOSTRA SALPA,NaN
10240,1020010499,DOKUMA,GORAY SAYA TABAN MOSTRA SALPA,NaN
10249,1020010510,DOKUMA,PESTEN SAYA TABAN MOSTRA SALPA,NaN
10250,1020010511,DOKUMA,FORTE NAVY SAYA TABAN MOSTRA SALPA,NaN
10251,1020010512,DOKUMA,FORTE GREY SAYA TABAN MOSTRA SALPA,NaN
10252,1020010513,DOKUMA,NİCE SAYA TABAN MOSTRA SALPA,NaN
10253,1020010514,DOKUMA,KANO SAYA TABAN MOSTRA SALPA,NaN
10254,1020010515,DOKUMA,OLİVER SAYA TABAN MOSTRA SALPA,NaN
10255,1020010516,DOKUMA,SCOLA SAYA TABAN MOSTRA SALPA,NaN


In [20]:
# Assign preliminary scope category
saya_candidates["scope_category"] = "saya"

In [21]:
# Define strong trim indicators
trim_keywords = [
    "FERMUAR",
    "DÜĞME",
    "DUGME",
    "TOKA",
    "KORDON",
    "ŞERİT",
    "SERIT",
    "BİYE",
    "BIYE",
    "LASTİK",
    "LASTIK",
    "AKSESUAR"
]

# Create a regex pattern using whole words
trim_pattern = r"\b(?:" + "|".join(trim_keywords) + r")\b"

# Identify potential trim materials
trim_candidates = missing_attribute_materials[
    missing_attribute_materials["combined_description"]
    .str.contains(trim_pattern, regex=True, na=False)
].copy()

print("Trim candidates:", len(trim_candidates))

display(
    trim_candidates[
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması",
            "PLM Kodu"
        ]
    ].head(50)
)

Trim candidates: 11


,Malzeme,Mal grubu,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu
8734,1020008939,ORME,ŞERİT DNTL DUZ BYA,P3S PINK 100 PES DNTL DUZ BYA,98492.0
9177,1020009406,DOKUMA,BRD E5X BYZ 60/1 60/1 %65 PAM %3,PAMUK VUAL ÜZERI ŞERİT BRODE(300,NaN
10330,1020010594,DOKUMA,SERIT-ANIMAL KUMAŞ,NaN,NaN
10493,1020010761,DOKUMA,RABAT HI- 16 MM SERIT,NaN,NaN
10494,1020010762,DOKUMA,RABAT HI- 10 MM SERIT,NaN,NaN
11683,1020012047,DENIM,PATARA BEJ ŞERİT,NaN,NaN
11684,1020012048,DENIM,PATARA BEJ LASTİK,NaN,NaN
11696,1020012060,DENIM,İDA SİYAH ŞERİT,NaN,NaN
11697,1020012061,DENIM,İDA SİYAH LASTİK,NaN,NaN
11915,1020012319,ORME,İNTERLOK BİYE KUMAŞ KIRMIZI,NaN,NaN


In [22]:
# Combine short and long material descriptions
sap_plm_mapping["combined_description"] = (
    sap_plm_mapping["Türkçe malzeme açıklaması"]
    .fillna("")
    .astype(str)
    + " "
    + sap_plm_mapping["Türkçe malzeme Uzun açıklaması"]
    .fillna("")
    .astype(str)
).str.upper()

# Define strong footwear-related indicators
saya_keywords = [
    "SAYA",
    "AYAKKABI",
    "MOSTRA",
    "SALPA"
]

saya_pattern = r"\b(?:" + "|".join(saya_keywords) + r")\b"

# Flag footwear-related materials
sap_plm_mapping["is_saya"] = (
    sap_plm_mapping["combined_description"]
    .str.contains(saya_pattern, regex=True, na=False)
)

print(sap_plm_mapping["is_saya"].value_counts())

is_saya
False    21094
True       111
Name: count, dtype: int64


In [23]:
# Create a separate list for potential material group corrections
material_group_update_candidates = (
    sap_plm_mapping[
        sap_plm_mapping["is_saya"]
    ]
    .copy()
)

display(
    material_group_update_candidates[
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "PLM Kodu"
        ]
    ]
)

,Malzeme,Mal grubu,Türkçe malzeme açıklaması,PLM Kodu
10237,1020010496,DOKUMA,COLOMBES BEIGE SAYA TABAN MOSTRA SALPA,NaN
10239,1020010498,DOKUMA,BARBAROS SAYA TABAN MOSTRA SALPA,NaN
10240,1020010499,DOKUMA,GORAY SAYA TABAN MOSTRA SALPA,NaN
10249,1020010510,DOKUMA,PESTEN SAYA TABAN MOSTRA SALPA,NaN
10250,1020010511,DOKUMA,FORTE NAVY SAYA TABAN MOSTRA SALPA,NaN
...,...,...,...,...
19958,1020020938,DOKUMA,"1,5 MM SALPA + SÜNGER + LAMİNASYON",NaN
20405,1020021400,DOKUMA,90 GR ALKANTRA+2.5 MM EVA+1.5 MM SALPA,NaN
20704,1020021704,DOKUMA,VANDA DANA ASTAR+MOSTRA,NaN
20963,1020021975,DOKUMA,"ALKANTRA + 2,5MM EVA + 1,5 MM SALPA",NaN


In [24]:
# Collect material IDs that should be excluded from the fabric modeling scope
excluded_materials = set(
    material_group_update_candidates["material_id"]
) | set(
    trim_candidates["material_id"]
)

# Keep saya candidates separately
saya_materials = material_group_update_candidates.copy()

# Keep trim candidates separately
trim_materials = trim_candidates.copy()

# Create the final fabric modeling dataset
fabric_materials = (
    sap_plm_mapping[
        ~sap_plm_mapping["material_id"].isin(excluded_materials)
    ]
    .copy()
)

print("Original SAP materials:", sap_plm_mapping["material_id"].nunique())
print("Fabric materials:", fabric_materials["material_id"].nunique())
print("Saya materials:", saya_materials["material_id"].nunique())
print("Trim materials:", trim_materials["material_id"].nunique())

Original SAP materials: 21205
Fabric materials: 21083
Saya materials: 111
Trim materials: 11


In [25]:
#Calculate PLM coverage within the fabric modeling scope
total_fabric_materials = fabric_materials["material_id"].nunique()
fabric_materials_with_plm = (
    fabric_materials
    .dropna(subset=["PLM Kodu"])["material_id"]
    .nunique()
)

fabric_materials_without_plm = (
    total_materials - fabric_materials_with_plm
)
print("Total fabric materials:", total_fabric_materials)
print("Fabric materials with PLM:", fabric_materials_with_plm)
print("Fabric materials without PLM:", fabric_materials_without_plm)
print(
    "PLM coverage:",
    round(fabric_materials_with_plm/total_fabric_materials*100,2),"%"

)

Total fabric materials: 21083
Fabric materials with PLM: 5645
Fabric materials without PLM: 15560
PLM coverage: 26.78 %


In [27]:
# Check attribute coverage for fabric materials

# PLM var/yok bilgisini oluştur
fabric_materials["has_plm"] = (
    fabric_materials["plm_code"].notna()
)

# Attribute var/yok bilgisini oluştur
fabric_materials["has_attributes"] = (
    fabric_materials["material_id"].isin(attribute_materials)
)

# Önce material_id seviyesine indir:
# aynı malzemenin birden fazla satırı varsa çift sayılmasını engelle
fabric_material_flags = (
    fabric_materials
    .groupby("material_id", as_index=False)
    .agg(
        has_plm=("has_plm", "max"),
        has_attributes=("has_attributes", "max")
    )
)

# PLM durumuna göre attribute coverage hesapla
fabric_attribute_coverage = (
    fabric_material_flags
    .groupby("has_plm")
    .agg(
        materials_count=("material_id", "nunique"),
        materials_with_attributes=("has_attributes", "sum")
    )
)

fabric_attribute_coverage["attribute_coverage_pct"] = (
    fabric_attribute_coverage["materials_with_attributes"]
    / fabric_attribute_coverage["materials_count"]
    * 100
).round(2)

display(fabric_attribute_coverage)

KeyError: 'plm_code'

In [ ]:
#keep attribute records only for materials in the fabric modeling scope
fabric_attributes = material_attributes[
    material_attributes["material_id"].isin(
        set(fabric_materials["material_id"])
    )
].copy()

#summarize attribute availability
attribute_summary = (
    fabric_attributes
    .groupby("Dahili krkt.no.")
    .agg(
        material_count = ("material_id", "nunique"),
        unique_value_count = ("Karakteristik değeri", "nunique"),
        row_count = ("material_id", "size")
    ).sort_values("material_count", ascending=False)
)

attribute_summary["coverage_pct"] = (
    attribute_summary["material_count"] / total_fabric_materials * 100
).round(2)
print("Unique attribute types:", len(attribute_summary))

display(attribute_summary.head(40))

In [ ]:
display(
    pd.DataFrame({
        "attribute_name": sorted(
            fabric_attributes["Dahili krkt.no."]
            .dropna()
            .unique()
        )
    })
)

In [ ]:
# Define candidate mappings between SAP attributes and PLM master fields
attribute_mapping = {
    "WEIGHT": "Kumaş ağırlığı",
    "WEIGHTUOMNAME": "Kumaş ağırlığı birimi",
    "WIDTH": "Kumaş eni",
    "WIDTHUOMNAME": "Kumaş eni birimi",

    # Knitted fabric attributes
    "KNITSUBTYPEID": "Örme alt tipi",
    "LOOPLENGTH50NEEDLE": "İlmek uzunluğu/50 iğne",
    "YARNCOUNT1KNITSID": "1.İplik numarası örme",
    "YARNCOUNT2KNITSID": "2.İplik numarası örm",
    "YARNCOUNT3KNITSID": "3.İplik numarası örme",
    "PUS": "Pus",
    "FINE": "Fine",

    # Woven fabric attributes
    "WEAVETYPE": "Dokuma tipi",
    "WARPDENSITY": "Çözgü sıklığı (tel /in",
    "WEFTDENSITY": "Atkı sıklığı (tel /inç)",
    "WARPYARNCOUNT1ID": "1.Çözgü iplik numarası",
    "WEFTYARNCOUNT1ID": "1.Atkı iplik numarası",
    "WARPYARNCOUNT2ID": "2.Çözgü iplik numarası",
    "WEFTYARNCOUNT2ID": "2.Atkı iplik numarası",
    "WARPYARNCOUNT3ID": "3.Çözgü iplik numarası",
    "WEFTYARNCOUNT3ID": "3.Atkı iplik numaras"
}

In [ ]:
# Keep only SAP materials with an existing PLM code
known_plm_materials = set(
    fabric_materials.loc[
        fabric_materials["has_plm"],
        "material_id"
    ]
)

# Calculate coverage of candidate attributes within known PLM materials
candidate_attribute_coverage = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(attribute_mapping.keys())
        & fabric_attributes["material_id"].isin(known_plm_materials)
    ]
    .groupby("Dahili krkt.no.")["material_id"]
    .nunique()
    .to_frame("material_count")
)

candidate_attribute_coverage["coverage_pct"] = (
    candidate_attribute_coverage["material_count"]
    / len(known_plm_materials)
    * 100
).round(2)

candidate_attribute_coverage["plm_field"] = (
    candidate_attribute_coverage.index.map(attribute_mapping)
)

display(
    candidate_attribute_coverage
    .sort_values("coverage_pct", ascending=False)
)

In [ ]:
# Compare sample values between SAP attributes and PLM master fields
for sap_attribute, plm_field in {
    "WEIGHT": "Kumaş ağırlığı",
    "WIDTH": "Kumaş eni",
    "WEAVETYPE": "Dokuma tipi",
    "KNITSUBTYPEID": "Örme alt tipi"
}.items():

    print(f"\n--- {sap_attribute} <-> {plm_field} ---")

    print("\nSAP sample values:")
    print(
        fabric_attributes.loc[
            fabric_attributes["Dahili krkt.no."] == sap_attribute,
            "Karakteristik değeri"
        ]
        .dropna()
        .drop_duplicates()
        .head(15)
        .tolist()
    )

    print("\nPLM sample values:")
    print(
        plm_codes[plm_field]
        .dropna()
        .drop_duplicates()
        .head(15)
        .tolist()
    )

In [ ]:
# Standardize PLM codes while preserving identifier identity
import re

def normalize_plm_code(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    # Remove Excel-style ".0" only when the entire identifier is numeric.
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = re.sub(r"\.0+$", "", value)

    return value


fabric_materials["plm_code"] = (
    fabric_materials["PLM Kodu"]
    .apply(normalize_plm_code)
)

plm_codes["plm_code"] = (
    plm_codes["PLM Kodu"]
    .apply(normalize_plm_code)
)


In [ ]:
# Compare existing SAP-PLM mappings with the PLM master
mapped_plm_codes = set(
    fabric_materials["plm_code"].dropna()
)

master_plm_codes = set(
    plm_codes["plm_code"].dropna()
)

common_plm_codes = mapped_plm_codes & master_plm_codes
missing_from_master = mapped_plm_codes - master_plm_codes

print("PLM codes used in SAP mapping:", len(mapped_plm_codes))
print("PLM codes available in PLM master:", len(master_plm_codes))
print("Common PLM codes:", len(common_plm_codes))
print("Mapped PLM codes missing from master:", len(missing_from_master))

In [ ]:
#Inspect weight values togetger with their units
weight_values = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(
            ["WEIGHT", "WEIGHTUOMNAME"]
        )
    ].pivot_table(
        index = "material_id",
        columns = "Dahili krkt.no.",
        values = "Karakteristik değeri",
        aggfunc= "first"
    ).reset_index()
)

display(weight_values.head(30))

display(
    weight_values["WEIGHTUOMNAME"]
    .value_counts(dropna=False).head(30)
)

In [ ]:
weight_values["sap_weight_raw"] = pd.to_numeric(
    weight_values["WEIGHT"],
    errors="coerce"
)

plm_weight = plm_codes[
    ["plm_code","Kumaş ağırlığı"]
].copy()

plm_weight["plm_weight"] = pd.to_numeric(
    plm_weight["Kumaş ağırlığı"],
    errors="coerce"
)

known_weight_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        weight_values[
            ["material_id","sap_weight_raw","WEIGHTUOMNAME"]
        ],
        on="material_id",
        how="inner"
    ).merge(
        plm_weight[
            ["plm_code","plm_weight"]
             ],
             on="plm_code",
             how="inner"
        )
)

#keep valid positive values only
known_weight_pairs = known_weight_pairs[
    (known_weight_pairs["sap_weight_raw"]>0)&
    (known_weight_pairs["plm_weight"]>0)
].copy()

display(known_weight_pairs.head(30))

In [ ]:
# Calculate the scale ratio between SAP and PLM weight values
known_weight_pairs["weight_ratio"] = (
    known_weight_pairs["sap_weight_raw"]
    / known_weight_pairs["plm_weight"]
)

display(
    known_weight_pairs[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "plm_weight",
            "weight_ratio",
            "WEIGHTUOMNAME"
        ]
    ]
    .sort_values("weight_ratio", ascending=False)
    .head(50)
)

In [ ]:
# Classify common scaling patterns
def classify_weight_scale(ratio):
    if 0.95 <= ratio <= 1.05:
        return "same_scale"
    elif 950 <= ratio <= 1050:
        return "x1000"
    else:
        return "other"


known_weight_pairs["scale_pattern"] = (
    known_weight_pairs["weight_ratio"]
    .apply(classify_weight_scale)
)

display(
    known_weight_pairs["scale_pattern"]
    .value_counts()
    .to_frame("record_count")
)

In [ ]:
# Normalize SAP weight values
def normalize_weight(value):
    if pd.isna(value):
        return np.nan

    # Values above 1000 appear to be stored with a x1000 scale
    if value >= 1000:
        return value / 1000

    return value


known_weight_pairs["sap_weight_normalized"] = (
    known_weight_pairs["sap_weight_raw"]
    .apply(normalize_weight)
)

# Calculate the difference after normalization
known_weight_pairs["weight_difference"] = abs(
    known_weight_pairs["sap_weight_normalized"]
    - known_weight_pairs["plm_weight"]
)

known_weight_pairs["weight_difference_pct"] = (
    known_weight_pairs["weight_difference"]
    / known_weight_pairs["plm_weight"]
    * 100
)
# Evaluate weight normalization quality
print(
    "Exact matches:",
    (known_weight_pairs["weight_difference"] == 0).sum()
)

print(
    "Within ±5%:",
    (known_weight_pairs["weight_difference_pct"] <= 5).sum()
)

print(
    "Within ±10%:",
    (known_weight_pairs["weight_difference_pct"] <= 10).sum()
)

print(
    "Total pairs:",
    len(known_weight_pairs)
)

# Calculate match percentages after normalization
weight_validation = pd.Series({
    "exact_match_pct":
        (known_weight_pairs["weight_difference"] == 0).mean() * 100,

    "within_5_pct":
        (known_weight_pairs["weight_difference_pct"] <= 5).mean() * 100,

    "within_10_pct":
        (known_weight_pairs["weight_difference_pct"] <= 10).mean() * 100
}).round(2)

display(weight_validation)

In [ ]:
# Inspect large differences after normalization
weight_outliers = (
    known_weight_pairs[
        known_weight_pairs["weight_difference_pct"] > 10
    ]
    .sort_values("weight_difference_pct", ascending=False)
)

display(
    weight_outliers[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "sap_weight_normalized",
            "plm_weight",
            "weight_difference_pct",
            "WEIGHTUOMNAME"
        ]
    ].head(30)
)

In [ ]:
# Prepare PLM weight values together with their units
plm_weight = plm_codes[
    [
        "plm_code",
        "Kumaş ağırlığı",
        "Kumaş ağırlığı birimi"
    ]
].copy()

plm_weight["plm_weight"] = pd.to_numeric(
    plm_weight["Kumaş ağırlığı"],
    errors="coerce"
)

plm_weight = plm_weight.rename(
    columns={
        "Kumaş ağırlığı birimi": "plm_weight_unit"
    }
)

In [ ]:
# Add the PLM weight unit to the validation dataset
known_weight_pairs = known_weight_pairs.drop(
    columns=["plm_weight"],
    errors="ignore"
)

known_weight_pairs = known_weight_pairs.merge(
    plm_weight[
        [
            "plm_code",
            "plm_weight",
            "plm_weight_unit"
        ]
    ],
    on="plm_code",
    how="left"
)

In [ ]:
# Normalize SAP weight values
known_weight_pairs["sap_weight_normalized"] = (
    known_weight_pairs["sap_weight_raw"]
    .apply(normalize_weight)
)

known_weight_pairs["weight_difference_pct"] = (
    abs(
        known_weight_pairs["sap_weight_normalized"]
        - known_weight_pairs["plm_weight"]
    )
    / known_weight_pairs["plm_weight"]
    * 100
)

In [ ]:
# Inspect remaining weight outliers together with measurement units
weight_outliers = (
    known_weight_pairs[
        known_weight_pairs["weight_difference_pct"] > 10
    ]
    .sort_values(
        "weight_difference_pct",
        ascending=False
    )
)

display(
    weight_outliers[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "sap_weight_normalized",
            "WEIGHTUOMNAME",
            "plm_weight",
            "plm_weight_unit",
            "weight_difference_pct"
        ]
    ].head(40)
)

In [ ]:
# Summarize weight outliers by SAP and PLM unit combinations
weight_outlier_units = (
    weight_outliers
    .groupby(
        ["WEIGHTUOMNAME", "plm_weight_unit"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

display(weight_outlier_units)

In [ ]:
# Identify weight discrepancies where both systems use the same unit
same_unit_outliers = weight_outliers[
    weight_outliers["WEIGHTUOMNAME"].fillna("UNKNOWN")
    ==
    weight_outliers["plm_weight_unit"].fillna("UNKNOWN")
].copy()

display(
    same_unit_outliers[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "WEIGHTUOMNAME",
            "plm_weight",
            "plm_weight_unit",
            "weight_difference_pct"
        ]
    ]
)

In [ ]:
# Prepare SAP width values
width_values = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(
            ["WIDTH", "WIDTHUOMNAME"]
        )
    ]
    .pivot_table(
        index="material_id",
        columns="Dahili krkt.no.",
        values="Karakteristik değeri",
        aggfunc="first"
    )
    .reset_index()
)

display(width_values.head(30))

In [ ]:
# Check the most common SAP width units
display(
    width_values["WIDTHUOMNAME"]
    .value_counts(dropna=False)
    .head(30)
)

In [ ]:
# Prepare PLM width data
plm_width = plm_codes[
    [
        "plm_code",
        "Kumaş eni",
        "Kumaş eni birimi"
    ]
].copy()

plm_width["plm_width"] = pd.to_numeric(
    plm_width["Kumaş eni"],
    errors="coerce"
)

plm_width = plm_width.rename(
    columns={
        "Kumaş eni birimi": "plm_width_unit"
    }
)

In [ ]:
# Convert SAP width values to numeric
width_values["sap_width"] = pd.to_numeric(
    width_values["WIDTH"],
    errors="coerce"
)

In [ ]:
# Create known SAP-PLM width pairs
known_width_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        width_values[
            ["material_id", "sap_width", "WIDTHUOMNAME"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        plm_width[
            ["plm_code", "plm_width", "plm_width_unit"]
        ],
        on="plm_code",
        how="inner"
    )
)

# Keep valid positive width values
known_width_pairs = known_width_pairs[
    (known_width_pairs["sap_width"] > 0) &
    (known_width_pairs["plm_width"] > 0)
].copy()

known_width_pairs["width_difference_pct"] = (
    abs(
        known_width_pairs["sap_width"]
        - known_width_pairs["plm_width"]
    )
    / known_width_pairs["plm_width"]
    * 100
)

print("Comparable width pairs:", len(known_width_pairs))

print(
    "Exact matches:",
    (known_width_pairs["width_difference_pct"] == 0).sum()
)

print(
    "Within ±5%:",
    (known_width_pairs["width_difference_pct"] <= 5).sum()
)

print(
    "Within ±10%:",
    (known_width_pairs["width_difference_pct"] <= 10).sum()
)

In [ ]:
# Evaluate SAP-PLM width consistency
width_validation = pd.Series({
    "exact_match_pct":
        (known_width_pairs["width_difference_pct"] == 0).mean() * 100,

    "within_5_pct":
        (known_width_pairs["width_difference_pct"] <= 5).mean() * 100,

    "within_10_pct":
        (known_width_pairs["width_difference_pct"] <= 10).mean() * 100
}).round(2)

display(width_validation)

In [ ]:
# Check width availability on both sides
print(
    "SAP materials with numeric width:",
    width_values["sap_width"].gt(0).sum()
)

print(
    "PLM records with numeric width:",
    plm_width["plm_width"].gt(0).sum()
)

print(
    "Known SAP-PLM materials with comparable width:",
    len(known_width_pairs)
)

In [ ]:
# Define categorical SAP-PLM attribute pairs
categorical_mapping = {
    "WEAVETYPE": "Dokuma tipi",
    "KNITSUBTYPEID": "Örme alt tipi"
}

categorical_results = []

for sap_attribute, plm_field in categorical_mapping.items():

    sap_values = (
        fabric_attributes[
            fabric_attributes["Dahili krkt.no."] == sap_attribute
        ][["material_id", "Karakteristik değeri"]]
        .drop_duplicates(subset=["material_id"])
        .rename(columns={"Karakteristik değeri": "sap_value"})
    )

    comparison = (
        fabric_materials[
            fabric_materials["plm_code"].notna()
        ][["material_id", "plm_code"]]
        .merge(sap_values, on="material_id", how="inner")
        .merge(
            plm_codes[["plm_code", plm_field]],
            on="plm_code",
            how="inner"
        )
        .dropna(subset=["sap_value", plm_field])
    )

    comparison["is_match"] = (
        comparison["sap_value"]
        .astype(str)
        .str.strip()
        .str.upper()
        ==
        comparison[plm_field]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    categorical_results.append({
        "attribute": sap_attribute,
        "comparable_pairs": len(comparison),
        "exact_matches": comparison["is_match"].sum(),
        "match_pct": round(comparison["is_match"].mean() * 100, 2)
    })

categorical_validation = pd.DataFrame(categorical_results)

display(categorical_validation)

In [ ]:
# Check width availability only for PLM codes currently linked to SAP materials
mapped_plm_width = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["plm_code"]]
    .drop_duplicates()
    .merge(
        plm_width[["plm_code", "plm_width", "plm_width_unit"]],
        on="plm_code",
        how="left"
    )
)

print(
    "Mapped PLM codes:",
    len(mapped_plm_width)
)

print(
    "Mapped PLM codes with positive width:",
    mapped_plm_width["plm_width"].gt(0).sum()
)

In [ ]:
# Compare SAP and PLM knit subtype values
knit_subtype_pairs = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."] == "KNITSUBTYPEID"
    ][["material_id", "Karakteristik değeri"]]
    .drop_duplicates(subset=["material_id"])
    .rename(columns={"Karakteristik değeri": "sap_knit_subtype"})
    .merge(
        fabric_materials[
            fabric_materials["plm_code"].notna()
        ][["material_id", "plm_code"]],
        on="material_id",
        how="inner"
    )
    .merge(
        plm_codes[
            ["plm_code", "Örme alt tipi"]
        ],
        on="plm_code",
        how="inner"
    )
    .dropna(
        subset=["sap_knit_subtype", "Örme alt tipi"]
    )
)

# Show the most common SAP-PLM value combinations
knit_subtype_combinations = (
    knit_subtype_pairs
    .groupby(
        ["sap_knit_subtype", "Örme alt tipi"]
    )
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

display(knit_subtype_combinations.head(30))

In [ ]:
# Normalize SAP knit subtype values
knit_subtype_mapping = {
    "C": "CIRCULAR",
    "F": "FLAT",
    "WARP": "WARP",
    "CIRCULAR": "CIRCULAR"
}

knit_subtype_pairs["sap_knit_subtype_normalized"] = (
    knit_subtype_pairs["sap_knit_subtype"]
    .replace(knit_subtype_mapping)
)

knit_subtype_pairs["is_match_normalized"] = (
    knit_subtype_pairs["sap_knit_subtype_normalized"]
    ==
    knit_subtype_pairs["Örme alt tipi"]
)

print(
    "Normalized match rate:",
    round(
        knit_subtype_pairs["is_match_normalized"].mean() * 100,
        2
    ),
    "%"
)

In [ ]:
# Check width availability step by step for known PLM materials
known_materials = fabric_materials[
    fabric_materials["plm_code"].notna()
][["material_id", "plm_code"]].copy()

width_check = (
    known_materials
    .merge(
        width_values[["material_id", "sap_width"]],
        on="material_id",
        how="left"
    )
    .merge(
        plm_width[["plm_code", "plm_width"]],
        on="plm_code",
        how="left"
    )
)

print("Known SAP-PLM materials:", len(width_check))
print("With SAP numeric width:", width_check["sap_width"].gt(0).sum())
print("With PLM numeric width:", width_check["plm_width"].gt(0).sum())
print(
    "With width on both sides:",
    (
        width_check["sap_width"].gt(0)
        & width_check["plm_width"].gt(0)
    ).sum()
)

In [ ]:
# Inspect raw WIDTH values for SAP materials with known PLM codes
known_width_raw = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        width_values[
            ["material_id", "WIDTH", "sap_width", "WIDTHUOMNAME"]
        ],
        on="material_id",
        how="inner"
    )
)

print("Known materials with WIDTH:", len(known_width_raw))
print("Numeric WIDTH values:", known_width_raw["sap_width"].notna().sum())
print(
    "Non-numeric WIDTH values:",
    known_width_raw["sap_width"].isna().sum()
)

# Show the most common raw WIDTH values that could not be converted to numeric
display(
    known_width_raw[
        known_width_raw["sap_width"].isna()
    ]["WIDTH"]
    .value_counts()
    .head(30)
)

In [ ]:
# Check WIDTH data type
print(width_values["sap_width"].dtype)

# Rebuild the width comparison dataset
width_check = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        width_values[["material_id", "sap_width"]],
        on="material_id",
        how="left"
    )
    .merge(
        plm_width[["plm_code", "plm_width"]],
        on="plm_code",
        how="left"
    )
)

print("Known SAP-PLM materials:", len(width_check))
print("With SAP numeric width:", width_check["sap_width"].notna().sum())
print("With positive SAP width:", width_check["sap_width"].gt(0).sum())
print("With positive PLM width:", width_check["plm_width"].gt(0).sum())

print(
    "With positive width on both sides:",
    (
        width_check["sap_width"].gt(0) &
        width_check["plm_width"].gt(0)
    ).sum()
)

In [ ]:
# Inspect the distribution of SAP width values for known PLM materials
known_sap_widths = width_check["sap_width"].dropna()

print("Numeric width records:", len(known_sap_widths))
print("Zero width records:", (known_sap_widths == 0).sum())
print("Positive width records:", (known_sap_widths > 0).sum())

display(
    known_sap_widths
    .value_counts()
    .head(20)
)

In [ ]:
# Display summary statistics for SAP width values
display(
    known_sap_widths.describe()
)

In [ ]:
# Treat zero width values as missing
width_values["sap_width_clean"] = (
    width_values["sap_width"]
    .replace(0, np.nan)
)

In [ ]:
# Inspect fiber content records
fiber_content = fabric_attributes[
    fabric_attributes["Dahili krkt.no."] == "FIBERCONTENTLISTID"
].copy()

print("Materials with fiber content:",
      fiber_content["material_id"].nunique())

print("Total fiber content rows:",
      len(fiber_content))

display(fiber_content.head(30))

In [ ]:
# Count fiber content records per material
fiber_count_per_material = (
    fiber_content
    .groupby("material_id")
    .size()
)

display(
    fiber_count_per_material
    .value_counts()
    .sort_index()
    .to_frame("material_count")
)
# Display the most common fiber content values
display(
    fiber_content["Karakteristik değeri"]
    .value_counts()
    .head(40)
)

In [ ]:
# Define the PLM multi-value characteristic file path
plm_multi_value_char_path = (
    data_folder / "PLM_MULTI_VAL_CHAR.xlsx"
)

# Check sheet names
print(
    pd.ExcelFile(plm_multi_value_char_path).sheet_names
)

In [ ]:
plm_multi_value_char = pd.read_excel(
    plm_multi_value_char_path,
    sheet_name="Data"
)

print("Shape:", plm_multi_value_char.shape)
print("\nColumns:")
print(plm_multi_value_char.columns.tolist())

display(plm_multi_value_char.head(20))

In [ ]:
#Normalize PLM codes
plm_multi_value_char["plm_code"] = (
    plm_multi_value_char["plm_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

#keep only plm codes that exist in the plm master

valid_plm_codes = set(
    plm_codes["plm_code"].dropna()
)

plm_multi_value_valid = (
    plm_multi_value_char[
        plm_multi_value_char["plm_code"].isin(valid_plm_codes)
    ].copy()
)
print("Total rows:", len(plm_multi_value_char))
print("Valid PLM characteristic rows:", len(plm_multi_value_valid))
print(
    "Unique valid PLM codes:",
    plm_multi_value_valid["plm_code"].nunique()
)

In [ ]:
# Inspect available multi-value PLM charachteristics
plm_characteristic_summary = (
    plm_multi_value_valid
    .groupby("PLM Karakteristik Tanımı")
    .agg(
        plm_count = ("plm_code", "nunique"),
        row_count = ("plm_code", "size"),
        unique_value_count = ("PLM Karakteristik Değeri", "nunique")
    ).sort_values("plm_count", ascending=False)
)

display(plm_characteristic_summary.head(30))

# Keep PLM fiber content records
plm_fiber_content = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "FIBERCONTENTLISTID"
    ].copy()
)

print(
    "PLM codes with fiber content:",
    plm_fiber_content["plm_code"].nunique()
)

print(
    "Total fiber content rows:",
    len(plm_fiber_content)
)

display(plm_fiber_content.head(20))

In [ ]:
# Check whether PLM codes have multiple versions
version_summary = ( #version summary
    plm_multi_value_valid
    .groupby("plm_code")["version"] #plm_code değerine göre grupluyor. sonra her plm'in version değerine bakıyor
    .nunique() #benzersiz versiyon sayısını buluyor.
    .value_counts() #kaç plm kodunda kaç verisyon var bunu hesaplıyor, bunu koymazsak plm kodu bazında benzersiz versiyon sayısı döner
    .sort_index() #sonucu index'e göre küçükten büyüğe sıralıyor.
)

display(version_summary)

In [ ]:
# Check the version column
print("Version data type:", plm_multi_value_valid["version"].dtype)

display(version_summary)

# Show PLM codes that have more than one version
multi_version_plm = (
    plm_multi_value_valid
    .groupby("plm_code")["version"]
    .nunique()
    .sort_values(ascending=False)
)

display(
    multi_version_plm[
        multi_version_plm > 1
    ].head(20)
)

In [ ]:
import re

#Normalize obvious fiber name variations
def normalize_fiber_name(fiber_name):
    if pd.isna(fiber_name):
        return pd.NA
    fiber_name = str(fiber_name).strip().upper()

    fiber_name_mapping = {
        "POLIESTER": "POLYESTER",
        "ELASTHANE": "ELASTANE",
        "ELASTANE/SPANDEX": "ELASTANE",
        "POLIAMID6": "POLYAMIDE",
        "NYLON": "POLYAMIDE",
        "ACETAT": "ACETATE",
        "POLYAMIDE6": "POLYAMIDE",
        "TENCEL": "LYOCELL",
    }

    return fiber_name_mapping.get(fiber_name, fiber_name)

#Extract percentage and fiber name from a composition value
def parse_fiber_content(value):
    if pd.isna(value):
        return pd.Series([np.nan, pd.NA])
    value = str(value).strip().upper()
    match = re.match(r"^(\d+(?:\.\d+)?)\s+(.+)$", value)

    if match:
        percentage = float(match.group(1))
        fiber_name = normalize_fiber_name(match.group(2))
        return pd.Series([percentage, fiber_name])

    return pd.Series(
        [
            np.nan,
            normalize_fiber_name(value)
        ]
    )

# Parse SAP fiber content values
fiber_content[
    ["fiber_percentage", "fiber_name"]
] = fiber_content["Karakteristik değeri"].apply(
    parse_fiber_content
)

# Parse PLM fiber content values
plm_fiber_content[
    ["fiber_percentage", "fiber_name"]
] = plm_fiber_content[
    "PLM Karakteristik Değeri"
].apply(
    parse_fiber_content
)

# Check parsing quality
print(
    "SAP rows with percentage:",
    fiber_content["fiber_percentage"].notna().sum(),
    "/",
    len(fiber_content)
)

print(
    "PLM rows with percentage:",
    plm_fiber_content["fiber_percentage"].notna().sum(),
    "/",
    len(plm_fiber_content)
)

# Inspect the most common values without percentage information
print("SAP values without percentage:")
display(
    fiber_content[
        fiber_content["fiber_percentage"].isna()
    ]["Karakteristik değeri"]
    .value_counts()
    .head(30)
)

print("PLM values without percentage:")
display(
    plm_fiber_content[
        plm_fiber_content["fiber_percentage"].isna()
    ]["PLM Karakteristik Değeri"]
    .value_counts()
    .head(30)
)

# Calculate total composition percentage per SAP material
sap_fiber_totals = (
    fiber_content
    .groupby("material_id")["fiber_percentage"]
    .sum(min_count=1)
)

# Calculate total composition percentage per PLM code
plm_fiber_totals = (
    plm_fiber_content
    .groupby("plm_code")["fiber_percentage"]
    .sum(min_count=1)
)

print("SAP composition totals:")
display(sap_fiber_totals.describe())

print("\nPLM composition totals:")
display(plm_fiber_totals.describe())

In [ ]:
# Remove exact duplicate fiber records
sap_fiber_clean = (
    fiber_content[
        ["material_id", "fiber_name", "fiber_percentage"]
    ]
    .dropna(subset=["fiber_name"])
    .drop_duplicates()
    .copy()
)

plm_fiber_clean = (
    plm_fiber_content[
        ["plm_code", "fiber_name", "fiber_percentage"]
    ]
    .dropna(subset=["fiber_name"])
    .drop_duplicates()
    .copy()
)

In [ ]:
# Create fiber sets for SAP materials
sap_fiber_sets = (
    sap_fiber_clean
    .groupby("material_id")["fiber_name"]
    .apply(set)
    .rename("sap_fiber_set")
    .reset_index()
)

# Create fiber sets for PLM codes
plm_fiber_sets = (
    plm_fiber_clean
    .groupby("plm_code")["fiber_name"]
    .apply(set)
    .rename("plm_fiber_set")
    .reset_index()
)

In [ ]:
# Build known SAP-PLM pairs
known_fiber_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .drop_duplicates()
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="left"
    )
    .merge(
        plm_fiber_sets,
        on="plm_code",
        how="left"
    )
)

In [ ]:
# Calculate Jaccard similarity between two fiber sets
def calculate_jaccard_similarity(set_a, set_b):
    if not isinstance(set_a, set) or not isinstance(set_b, set):
        return np.nan

    union = set_a | set_b

    if len(union) == 0:
        return np.nan

    return len(set_a & set_b) / len(union)


known_fiber_pairs["fiber_set_similarity"] = (
    known_fiber_pairs.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_fiber_set"],
            row["plm_fiber_set"]
        ),
        axis=1
    )
)

In [ ]:
print(
    "Comparable SAP-PLM pairs:",
    known_fiber_pairs["fiber_set_similarity"].notna().sum()
)

display(
    known_fiber_pairs[
        "fiber_set_similarity"
    ].describe()
)

display(
    known_fiber_pairs[
        "fiber_set_similarity"
    ].value_counts()
    .sort_index(ascending=False)
    .head(20)
)

In [ ]:
comparable_fiber_pairs = known_fiber_pairs[
    known_fiber_pairs["fiber_set_similarity"].notna()
]

exact_fiber_set_match_rate = (
    comparable_fiber_pairs["fiber_set_similarity"].eq(1).mean()
)

print(
    f"Exact fiber-set match rate: "
    f"{exact_fiber_set_match_rate:.2%}"
)

In [ ]:
# Identify SAP materials with reliable percentage compositions
sap_percentage_quality = (
    fiber_content
    .groupby("material_id")
    .agg(
        total_rows=("fiber_name", "size"),
        parsed_rows=("fiber_percentage", "count"),
        composition_total=(
            "fiber_percentage",
            lambda x: x.sum(min_count=1)
        )
    )
)

sap_percentage_quality["is_reliable"] = (
    (sap_percentage_quality["total_rows"] ==
     sap_percentage_quality["parsed_rows"])
    &
    sap_percentage_quality["composition_total"].between(98, 102)
)

reliable_sap_materials = set(
    sap_percentage_quality[
        sap_percentage_quality["is_reliable"]
    ].index
)

print(
    "SAP materials with reliable percentage composition:",
    len(reliable_sap_materials)
)

In [ ]:
# Identify PLM codes with reliable percentage compositions
plm_percentage_quality = (
    plm_fiber_content
    .groupby("plm_code")["fiber_percentage"]
    .sum()
)

reliable_plm_codes = set(
    plm_percentage_quality[
        plm_percentage_quality.between(98, 102)
    ].index
)

print(
    "PLM codes with reliable percentage composition:",
    len(reliable_plm_codes)
)

In [ ]:
# Build SAP percentage profiles
sap_composition = (
    fiber_content[
        fiber_content["material_id"].isin(reliable_sap_materials)
    ]
    .groupby(
        ["material_id", "fiber_name"],
        as_index=False
    )["fiber_percentage"]
    .sum()
)

# Build PLM percentage profiles
plm_composition = (
    plm_fiber_content[
        plm_fiber_content["plm_code"].isin(reliable_plm_codes)
    ]
    .groupby(
        ["plm_code", "fiber_name"],
        as_index=False
    )["fiber_percentage"]
    .sum()
)

In [ ]:
# Normalize compositions to 100%
sap_composition["fiber_percentage_normalized"] = (
    sap_composition["fiber_percentage"]
    /
    sap_composition.groupby("material_id")[
        "fiber_percentage"
    ].transform("sum")
    * 100
)

plm_composition["fiber_percentage_normalized"] = (
    plm_composition["fiber_percentage"]
    /
    plm_composition.groupby("plm_code")[
        "fiber_percentage"
    ].transform("sum")
    * 100
)

In [ ]:
# Convert compositions to dictionaries
sap_composition_profiles = (
    sap_composition
    .groupby("material_id")
    .apply(
        lambda group: dict(
            zip(
                group["fiber_name"],
                group["fiber_percentage_normalized"]
            )
        ),
        include_groups=False
    )
    .rename("sap_composition")
    .reset_index()
)

plm_composition_profiles = (
    plm_composition
    .groupby("plm_code")
    .apply(
        lambda group: dict(
            zip(
                group["fiber_name"],
                group["fiber_percentage_normalized"]
            )
        ),
        include_groups=False
    )
    .rename("plm_composition")
    .reset_index()
)

In [ ]:
# Calculate percentage-based composition similarity
def calculate_composition_similarity(sap_profile, plm_profile):
    if not isinstance(sap_profile, dict) or not isinstance(plm_profile, dict):
        return np.nan

    fibers = set(sap_profile) | set(plm_profile)

    total_difference = sum(
        abs(
            sap_profile.get(fiber, 0)
            - plm_profile.get(fiber, 0)
        )
        for fiber in fibers
    )

    return 1 - (total_difference / 200)

In [ ]:
# Compare reliable compositions for known SAP-PLM mappings
known_composition_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .drop_duplicates()
    .merge(
        sap_composition_profiles,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_composition_profiles,
        on="plm_code",
        how="inner"
    )
)

known_composition_pairs["fiber_composition_similarity"] = (
    known_composition_pairs.apply(
        lambda row: calculate_composition_similarity(
            row["sap_composition"],
            row["plm_composition"]
        ),
        axis=1
    )
)

print(
    "Comparable reliable composition pairs:",
    len(known_composition_pairs)
)

display(
    known_composition_pairs[
        "fiber_composition_similarity"
    ].describe()
)

print(
    "Exact composition match:",
    f"{known_composition_pairs['fiber_composition_similarity'].eq(1).mean():.2%}"
)

print(
    "Similarity >= 0.95:",
    f"{known_composition_pairs['fiber_composition_similarity'].ge(0.95).mean():.2%}"
)

print(
    "Similarity >= 0.90:",
    f"{known_composition_pairs['fiber_composition_similarity'].ge(0.90).mean():.2%}"
)

In [ ]:
# Inspect composition mismatches
composition_outliers = (
    known_composition_pairs[
        known_composition_pairs["fiber_composition_similarity"] < 0.95
    ]
    .sort_values("fiber_composition_similarity")
    .copy()
)

print("Composition outliers:", len(composition_outliers))

display(
    composition_outliers[
        [
            "material_id",
            "plm_code",
            "sap_composition",
            "plm_composition",
            "fiber_composition_similarity"
        ]
    ].head(50)
)

In [ ]:
# Add SAP material descriptions for error analysis
composition_outliers = (
    composition_outliers
    .merge(
        fabric_materials[
            [
                "material_id",
                "Türkçe malzeme açıklaması",
                "Türkçe malzeme Uzun açıklaması"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
)

display(
    composition_outliers[
        [
            "material_id",
            "plm_code",
            "Türkçe malzeme açıklaması",
            "sap_composition",
            "plm_composition",
            "fiber_composition_similarity"
        ]
    ].head(50)
)

Yüksek fiber similarity, aynı malzeme olduğuna dair çok güçlü pozitif kanıttır. Düşük similarity ise tek başına farklı malzeme anlamına gelmez; temporal attribute drift, yanlış mapping veya master-data problemi olabilir.

In [ ]:
# Define the material-group characteristic rules file
mandatory_fields_path = (
    data_folder / "Fabric_Mandatory_Fields.xlsx"
)

# Load material-group characteristic rules
mandatory_fields = pd.read_excel(
    mandatory_fields_path,
    sheet_name="Data"
)

print("Shape:", mandatory_fields.shape)
display(mandatory_fields.head(20))

In [ ]:
# Rename columns for easier analysis
mandatory_fields = mandatory_fields.rename(
    columns={
        "Mal grubu": "material_group",
        "Tanım Tipi (Kısa/Uzun)": "rule_type",
        "Özellik": "rule_description",
        "Field name": "field_name",
        "Sıra": "sequence"
    }
)

mandatory_fields["material_group"] = (
    mandatory_fields["material_group"]
    .astype(str)
    .str.strip()
)

mandatory_fields["field_name"] = (
    mandatory_fields["field_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

mandatory_fields["sequence"] = pd.to_numeric(
    mandatory_fields["sequence"],
    errors="coerce"
)

In [ ]:
# Mandatory characteristics
mandatory_policy = (
    mandatory_fields[
        mandatory_fields["rule_type"] == "Z"
    ][["material_group", "field_name"]]
    .drop_duplicates()
    .assign(is_mandatory=True)
)

# Characteristics used in short descriptions
short_description_policy = (
    mandatory_fields[
        mandatory_fields["rule_type"] == "K"
    ][
        [
            "material_group",
            "field_name",
            "sequence"
        ]
    ]
    .rename(
        columns={
            "sequence": "short_description_order"
        }
    )
    .drop_duplicates()
    .assign(in_short_description=True)
)

# Characteristics used in long descriptions
long_description_policy = (
    mandatory_fields[
        mandatory_fields["rule_type"] == "U"
    ][
        [
            "material_group",
            "field_name",
            "sequence"
        ]
    ]
    .rename(
        columns={
            "sequence": "long_description_order"
        }
    )
    .drop_duplicates()
    .assign(in_long_description=True)
)

In [ ]:
# Create one row per material group and characteristic
feature_policy = (
    mandatory_fields[
        ["material_group", "field_name"]
    ]
    .drop_duplicates()
    .merge(
        mandatory_policy,
        on=["material_group", "field_name"],
        how="left"
    )
    .merge(
        short_description_policy,
        on=["material_group", "field_name"],
        how="left"
    )
    .merge(
        long_description_policy,
        on=["material_group", "field_name"],
        how="left"
    )
)

boolean_columns = [
    "is_mandatory",
    "in_short_description",
    "in_long_description"
]

feature_policy[boolean_columns] = (
    feature_policy[boolean_columns]
    .eq(True)
)

feature_policy = feature_policy.sort_values(
    [
        "material_group",
        "is_mandatory",
        "short_description_order",
        "long_description_order"
    ],
    ascending=[True, False, True, True]
)

display(feature_policy)


In [ ]:
# Normalize material group in the SAP material dataset
fabric_materials["material_group"] = (
    fabric_materials["Mal grubu"]
    .astype(str)
    .str.strip()
)

policy_material_groups = set(
    feature_policy["material_group"]
)

fabric_materials["has_feature_policy"] = (
    fabric_materials["material_group"]
    .isin(policy_material_groups)
)

print(
    "Materials covered by feature policy:",
    fabric_materials["has_feature_policy"].sum(),
    "/",
    len(fabric_materials)
)

display(
    fabric_materials[
        "material_group"
    ].value_counts()
)

In [ ]:
# Normalize policy material-group codes
feature_policy["material_group"] = (
    pd.to_numeric(
        feature_policy["material_group"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


In [ ]:
# Normalize SAP material families
fabric_materials["material_family"] = (
    fabric_materials["Mal grubu"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [ ]:
# Inspect raw PLM material-group values
display(
    plm_codes["Mal grubu"]
    .value_counts(dropna=False)
    .head(20)
)

In [ ]:
# Map current PLM material-group labels to policy codes
material_group_code_mapping = {
    "ORME": "1020001",
    "DOKUMA": "1020002",
    "DENIM": "1030004"
}

plm_codes["material_group_name"] = (
    plm_codes["Mal grubu"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_codes["material_group_code"] = (
    plm_codes["material_group_name"]
    .map(material_group_code_mapping)
    .astype("string")
)


In [ ]:
display(
    plm_codes[
        ["material_group_name", "material_group_code"]
    ]
    .value_counts(dropna=False)
)

In [ ]:
# Add actual PLM material group to known SAP-PLM mappings
known_material_groups = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][
        [
            "material_id",
            "plm_code",
            "material_family"
        ]
    ]
    .drop_duplicates()
    .merge(
        plm_codes[
            [
                "plm_code",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("plm_code"),
        on="plm_code",
        how="left"
    )
)

display(
    known_material_groups[
        "material_group_code"
    ].value_counts(dropna=False)
)

In [ ]:
# Inspect known mappings without a material-group policy
unmapped_known_materials = (
    known_material_groups[
        known_material_groups["material_group_code"].isna()
    ]
    .copy()
)

display(unmapped_known_materials)

In [ ]:
# Standardize policy material-group codes
feature_policy["material_group"] = (
    feature_policy["material_group"]
    .astype("string")
    .str.strip()
)

# Keep mandatory characteristics only
mandatory_features = (
    feature_policy[
        feature_policy["is_mandatory"]
    ][
        ["material_group", "field_name"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)

display(
    mandatory_features[
        "material_group_code"
    ].value_counts()
)

In [ ]:
# Standardize SAP characteristic names
fabric_attributes["field_name"] = (
    fabric_attributes["Dahili krkt.no."]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify characteristics with an actual value
sap_attribute_presence = (
    fabric_attributes[
        fabric_attributes["Karakteristik değeri"].notna()
        & fabric_attributes["Karakteristik değeri"]
        .astype("string")
        .str.strip()
        .ne("")
    ][
        ["material_id", "field_name"]
    ]
    .drop_duplicates()
    .assign(is_present=True)
)

In [ ]:
# Generate expected mandatory characteristics for known mappings
mandatory_coverage_detail = (
    known_material_groups[
        known_material_groups["material_group_code"].notna()
    ]
    .merge(
        mandatory_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"]
    .eq(True)
)

print(
    "Materials included:",
    mandatory_coverage_detail["material_id"].nunique()
)

print(
    "Mandatory fields evaluated:",
    mandatory_coverage_detail["field_name"].nunique()
)


In [ ]:
# Calculate mandatory-field coverage
mandatory_coverage_summary = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        material_count=("material_id", "nunique"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

mandatory_coverage_summary["coverage_rate"] = (
    mandatory_coverage_summary["present_count"]
    / mandatory_coverage_summary["material_count"]
)

display(
    mandatory_coverage_summary
    .sort_values(
        ["material_group_code", "coverage_rate"],
        ascending=[True, False]
    )
)

In [ ]:
# Calculate mandatory completeness per material
material_level_completeness = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    )
    .agg(
        mandatory_field_count=("field_name", "nunique"),
        mandatory_field_present=("is_present", "sum")
    )
    .reset_index()
)

material_level_completeness["mandatory_completeness"] = (
    material_level_completeness["mandatory_field_present"]
    / material_level_completeness["mandatory_field_count"]
)

display(
    material_level_completeness
    .groupby(
        ["material_group_code", "material_group_name"]
    )["mandatory_completeness"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
)

In [ ]:
# Identify materials that have at least one SAP attribute
materials_with_attributes = set(
    fabric_attributes["material_id"].dropna()
)

mandatory_coverage_detail["has_any_attribute"] = (
    mandatory_coverage_detail["material_id"]
    .isin(materials_with_attributes)
)

# Avoid the fillna downcasting warning
mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"].eq(True)
)
# Calculate both absolute and conditional mandatory-field coverage
mandatory_coverage_audit = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        total_materials=("material_id", "nunique"),
        materials_with_attributes=(
            "has_any_attribute",
            "sum"
        ),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

# Coverage among all known SAP-PLM mappings
mandatory_coverage_audit["absolute_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["total_materials"]
)

# Coverage only where SAP attribute data exists
mandatory_coverage_audit["conditional_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["materials_with_attributes"]
)

display(
    mandatory_coverage_audit.sort_values(
        [
            "material_group_code",
            "conditional_coverage"
        ],
        ascending=[True, False]
    )
)

In [ ]:
# Define business-validated policy corrections
policy_overrides = pd.DataFrame(
    [
        {
            "material_group": "1020001",
            "field_name": "WEAVETYPE",
            "is_mandatory_override": False,
            "reason": "WEAVETYPE is not applicable to knitted fabrics."
        },
        {
            "material_group": "1020002",
            "field_name": "YARNCOUNT1KNITSID",
            "is_mandatory_override": False,
            "reason": "Knitting yarn count is not applicable to woven fabrics."
        },
        {
            "material_group": "1030004",
            "field_name": "YARNCOUNT1KNITSID",
            "is_mandatory_override": False,
            "reason": "Knitting yarn count is not applicable to denim fabrics."
        }
    ]
)

display(policy_overrides)

In [ ]:
# Apply business-validated mandatory-field overrides
feature_policy = feature_policy.merge(
    policy_overrides,
    on=["material_group", "field_name"],
    how="left"
)

feature_policy["is_mandatory_original"] = (
    feature_policy["is_mandatory"]
)

feature_policy["is_mandatory"] = np.where(
    feature_policy["is_mandatory_override"].notna(),
    feature_policy["is_mandatory_override"],
    feature_policy["is_mandatory"]
)

feature_policy["is_mandatory"] = (
    feature_policy["is_mandatory"]
    .astype(bool)
)

In [ ]:
# Verify corrected mandatory-field policies
display(
    feature_policy[
        feature_policy["material_group"].isin(
            ["1020001", "1020002", "1030004"]
        )
    ][
        [
            "material_group",
            "field_name",
            "is_mandatory_original",
            "is_mandatory",
            "reason"
        ]
    ]
    .sort_values(
        ["material_group", "field_name"]
    )
)

In [ ]:
# Rebuild mandatory characteristics after policy corrections
mandatory_features = (
    feature_policy[
        feature_policy["is_mandatory"]
    ][
        ["material_group", "field_name"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)

display(
    mandatory_features[
        "material_group_code"
    ].value_counts()
)

In [ ]:
# Rebuild mandatory coverage using the corrected policy
mandatory_coverage_detail = (
    known_material_groups[
        known_material_groups["material_group_code"].notna()
    ]
    .merge(
        mandatory_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"]
    .eq(True)
)

mandatory_coverage_detail["has_any_attribute"] = (
    mandatory_coverage_detail["material_id"]
    .isin(materials_with_attributes)
)

In [ ]:
# Recalculate field-level coverage
mandatory_coverage_audit = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        total_materials=("material_id", "nunique"),
        materials_with_attributes=("has_any_attribute", "sum"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

mandatory_coverage_audit["absolute_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["total_materials"]
)

mandatory_coverage_audit["conditional_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["materials_with_attributes"]
)

display(
    mandatory_coverage_audit
    .sort_values(
        ["material_group_code", "conditional_coverage"],
        ascending=[True, False]
    )
)

In [ ]:
# Recalculate material-level mandatory completeness
material_level_completeness = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    )
    .agg(
        mandatory_field_count=("field_name", "nunique"),
        mandatory_field_present=("is_present", "sum")
    )
    .reset_index()
)

material_level_completeness["mandatory_completeness"] = (
    material_level_completeness["mandatory_field_present"]
    / material_level_completeness["mandatory_field_count"]
)

display(
    material_level_completeness
    .groupby(
        ["material_group_code", "material_group_name"]
    )["mandatory_completeness"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
)

In [ ]:
# Get all policy characteristics for the material groups in scope
all_policy_features = (
    feature_policy[
        feature_policy["material_group"].isin(
            ["1020001", "1020002", "1030004"]
        )
    ][
        [
            "material_group",
            "field_name",
            "is_mandatory",
            "in_short_description",
            "short_description_order",
            "in_long_description",
            "long_description_order"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)
# Generate expected policy characteristics for each known material
policy_coverage_detail = (
    known_material_groups[
        known_material_groups["material_group_code"].notna()
    ]
    .merge(
        all_policy_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

policy_coverage_detail["is_present"] = (
    policy_coverage_detail["is_present"].eq(True)
)

policy_coverage_detail["has_any_attribute"] = (
    policy_coverage_detail["material_id"]
    .isin(materials_with_attributes)
)
# Calculate observed coverage for every policy characteristic
policy_coverage_summary = (
    policy_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        total_materials=("material_id", "nunique"),
        materials_with_attributes=("has_any_attribute", "sum"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

policy_coverage_summary["absolute_coverage"] = (
    policy_coverage_summary["present_count"]
    / policy_coverage_summary["total_materials"]
)

policy_coverage_summary["conditional_coverage"] = (
    policy_coverage_summary["present_count"]
    / policy_coverage_summary["materials_with_attributes"]
)

In [ ]:
# Combine business policy with observed SAP coverage
feature_policy_audit = (
    all_policy_features
    .merge(
        policy_coverage_summary[
            [
                "material_group_code",
                "field_name",
                "conditional_coverage"
            ]
        ],
        on=["material_group_code", "field_name"],
        how="left"
    )
)

In [ ]:
# Assign a high-level role to each characteristic
def assign_feature_role(row):
    if row["is_mandatory"] and (
        row["in_short_description"]
        or row["in_long_description"]
    ):
        return "CORE"

    if row["is_mandatory"]:
        return "MANDATORY"

    if (
        row["in_short_description"]
        or row["in_long_description"]
    ):
        return "DESCRIPTION_SUPPORT"

    return "SUPPORTING"


feature_policy_audit["feature_role"] = (
    feature_policy_audit.apply(
        assign_feature_role,
        axis=1
    )
)

In [ ]:
display(
    feature_policy_audit[
        [
            "material_group_code",
            "field_name",
            "feature_role",
            "is_mandatory",
            "in_short_description",
            "short_description_order",
            "in_long_description",
            "long_description_order",
            "conditional_coverage"
        ]
    ]
    .sort_values(
        [
            "material_group_code",
            "feature_role",
            "conditional_coverage"
        ],
        ascending=[True, True, False]
    )
)

In [ ]:
# Assign modeling tiers based on business role and observed data coverage
def assign_modeling_tier(row):
    coverage = row["conditional_coverage"]

    if pd.isna(coverage):
        return "EXCLUDE"

    # Strong business relevance and high observed coverage
    if row["feature_role"] == "CORE" and coverage >= 0.80:
        return "PRIMARY"

    # Mandatory fields with high coverage
    if row["feature_role"] == "MANDATORY" and coverage >= 0.80:
        return "PRIMARY"

    # Relevant fields with moderate coverage
    if (
        row["feature_role"] in [
            "CORE",
            "MANDATORY",
            "DESCRIPTION_SUPPORT"
        ]
        and coverage >= 0.40
    ):
        return "SECONDARY"

    # Low-coverage fields may still provide useful evidence when available
    if coverage >= 0.10:
        return "OPTIONAL"

    return "EXCLUDE"


feature_policy_audit["modeling_tier"] = (
    feature_policy_audit.apply(
        assign_modeling_tier,
        axis=1
    )
)

In [ ]:
display(
    feature_policy_audit[
        [
            "material_group_code",
            "field_name",
            "feature_role",
            "conditional_coverage",
            "modeling_tier"
        ]
    ]
    .sort_values(
        [
            "material_group_code",
            "modeling_tier",
            "conditional_coverage"
        ],
        ascending=[True, True, False]
    )
)

In [ ]:
# Build material-group-specific feature sets
matching_feature_sets = (
    feature_policy_audit[
        feature_policy_audit["modeling_tier"]
        .isin(["PRIMARY", "SECONDARY"])
    ]
    .groupby(
        [
            "material_group_code",
            "modeling_tier"
        ]
    )["field_name"]
    .apply(list)
)

display(matching_feature_sets)

In [ ]:
# Check usable coverage for numeric characteristics
numeric_features = [
    "WEIGHT",
    "WIDTH"
]

numeric_quality_records = []

for material_group_code in ["1020001", "1020002", "1030004"]:
    
    group_materials = set(
        known_material_groups.loc[
            known_material_groups["material_group_code"]
            == material_group_code,
            "material_id"
        ]
    )

    group_attributes = fabric_attributes[
        fabric_attributes["material_id"].isin(group_materials)
    ]

    for field_name in numeric_features:

        field_values = group_attributes[
            group_attributes["field_name"] == field_name
        ].copy()

        field_values["numeric_value"] = pd.to_numeric(
            field_values["Karakteristik değeri"],
            errors="coerce"
        )

        materials_with_any_attributes = len(
            group_materials
            & materials_with_attributes
        )

        materials_with_valid_value = (
            field_values.loc[
                field_values["numeric_value"] > 0,
                "material_id"
            ]
            .nunique()
        )

        numeric_quality_records.append(
            {
                "material_group_code": material_group_code,
                "field_name": field_name,
                "materials_with_attributes":
                    materials_with_any_attributes,
                "materials_with_valid_value":
                    materials_with_valid_value,
                "usable_coverage":
                    materials_with_valid_value
                    / materials_with_any_attributes
                    if materials_with_any_attributes > 0
                    else np.nan
            }
        )


numeric_feature_quality = pd.DataFrame(
    numeric_quality_records
)

display(numeric_feature_quality)

In [ ]:
# Add usable coverage for numeric characteristics
feature_policy_audit = feature_policy_audit.merge(
    numeric_feature_quality[
        [
            "material_group_code",
            "field_name",
            "usable_coverage"
        ]
    ],
    on=["material_group_code", "field_name"],
    how="left"
)

# Use usable coverage when available, otherwise regular conditional coverage
feature_policy_audit["effective_coverage"] = (
    feature_policy_audit["usable_coverage"]
    .fillna(feature_policy_audit["conditional_coverage"])
)

In [ ]:
# Reassign modeling tiers using effective coverage
def assign_modeling_tier(row):
    coverage = row["effective_coverage"]

    if pd.isna(coverage):
        return "EXCLUDE"

    if (
        row["feature_role"] in ["CORE", "MANDATORY"]
        and coverage >= 0.80
    ):
        return "PRIMARY"

    if (
        row["feature_role"] in [
            "CORE",
            "MANDATORY",
            "DESCRIPTION_SUPPORT"
        ]
        and coverage >= 0.40
    ):
        return "SECONDARY"

    if coverage >= 0.10:
        return "OPTIONAL"

    return "EXCLUDE"


feature_policy_audit["modeling_tier"] = (
    feature_policy_audit.apply(
        assign_modeling_tier,
        axis=1
    )
)

In [ ]:
display(
    feature_policy_audit[
        [
            "material_group_code",
            "field_name",
            "feature_role",
            "conditional_coverage",
            "usable_coverage",
            "effective_coverage",
            "modeling_tier"
        ]
    ]
    .sort_values(
        [
            "material_group_code",
            "modeling_tier",
            "effective_coverage"
        ],
        ascending=[True, True, False]
    )
)

In [ ]:
# Select categorical characteristics that may be used for matching
categorical_features = [
    "FABRICSTRUCTURETNAME",
    "COLORINGID",
    "WEIGHTUOMNAME",
    "WEAVETYPE",
    "YARNCOUNT1KNITSID",
    "YARN1TYPEKNITSID",
    "WEFTYARNCOUNT1ID"
]

categorical_value_summary = (
    fabric_attributes[
        fabric_attributes["field_name"].isin(categorical_features)
    ]
    .merge(
        known_material_groups[
            [
                "material_id",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="inner"
    )
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name",
            "Karakteristik değeri"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

categorical_value_summary["rank"] = (
    categorical_value_summary
    .groupby(
        ["material_group_code", "field_name"]
    )["count"]
    .rank(
        method="first",
        ascending=False
    )
)

display(
    categorical_value_summary[
        categorical_value_summary["rank"] <= 20
    ]
    .sort_values(
        [
            "material_group_code",
            "field_name",
            "count"
        ],
        ascending=[True, True, False]
    )
)

In [ ]:
# Measure categorical feature diversity
categorical_feature_profile = (
    fabric_attributes[
        fabric_attributes["field_name"].isin(categorical_features)
    ]
    .merge(
        known_material_groups[
            [
                "material_id",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="inner"
    )
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        material_count=("material_id", "nunique"),
        unique_values=("Karakteristik değeri", "nunique")
    )
    .reset_index()
)

display(categorical_feature_profile)

In [ ]:
# Inspect weight-unit values by material group
weight_unit_distribution = (
    categorical_value_summary[
        categorical_value_summary["field_name"] == "WEIGHTUOMNAME"
    ][
        [
            "material_group_code",
            "material_group_name",
            "Karakteristik değeri",
            "count"
        ]
    ]
    .sort_values(
        ["material_group_code", "count"],
        ascending=[True, False]
    )
)

display(weight_unit_distribution)

In [ ]:
# Inspect PLM weight-unit values by material group
plm_weight_unit_distribution = (
    plm_codes[
        plm_codes["material_group_code"].isin(
            ["1020001", "1020002", "1030004"]
        )
    ]
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "Kumaş ağırlığı birimi"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["material_group_code", "count"],
        ascending=[True, False]
    )
)

display(plm_weight_unit_distribution)

In [ ]:
# Prepare SAP weight information
sap_weight_audit = weight_values[
    [
        "material_id",
        "sap_weight_raw",
        "WEIGHTUOMNAME"
    ]
].copy()

sap_weight_audit = sap_weight_audit.rename(
    columns={
        "WEIGHTUOMNAME": "sap_weight_unit_raw"
    }
)

In [ ]:
# Normalize only clearly equivalent weight-unit labels
def normalize_weight_unit(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    unit_mapping = {
        "GSM": "GSM",
        "G/M2": "GSM",
        "GR/M2": "GSM",
        "VRDOZPERSQYARD": "VRDOZPERSQYARD"
    }

    return unit_mapping.get(value, value)


sap_weight_audit["sap_weight_unit"] = (
    sap_weight_audit["sap_weight_unit_raw"]
    .apply(normalize_weight_unit)
)

In [ ]:
# Prepare PLM weight information
plm_weight_audit = plm_codes[
    [
        "plm_code",
        "Kumaş ağırlığı",
        "Kumaş ağırlığı birimi"
    ]
].copy()

plm_weight_audit["plm_weight"] = pd.to_numeric(
    plm_weight_audit["Kumaş ağırlığı"],
    errors="coerce"
)

plm_weight_audit["plm_weight_unit"] = (
    plm_weight_audit["Kumaş ağırlığı birimi"]
    .apply(normalize_weight_unit)
)

In [ ]:
# Build known SAP-PLM weight pairs
known_weight_audit = (
    known_material_groups[
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_weight_audit,
        on="material_id",
        how="left"
    )
    .merge(
        plm_weight_audit,
        on="plm_code",
        how="left"
    )
)

In [ ]:
# Inspect SAP vs PLM weight-unit combinations
weight_unit_pair_summary = (
    known_weight_audit
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "sap_weight_unit",
            "plm_weight_unit"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["material_group_code", "count"],
        ascending=[True, False]
    )
)

display(weight_unit_pair_summary)

In [ ]:
# Analyze scaling patterns where SAP and PLM use the same unit
same_unit_weight_pairs = known_weight_audit[
    known_weight_audit["sap_weight_raw"].notna()
    & known_weight_audit["plm_weight"].notna()
    & (known_weight_audit["sap_weight_raw"] > 0)
    & (known_weight_audit["plm_weight"] > 0)
    & known_weight_audit["sap_weight_unit"].notna()
    & (
        known_weight_audit["sap_weight_unit"]
        == known_weight_audit["plm_weight_unit"]
    )
].copy()

same_unit_weight_pairs["weight_ratio"] = (
    same_unit_weight_pairs["sap_weight_raw"]
    / same_unit_weight_pairs["plm_weight"]
)

display(
    same_unit_weight_pairs.groupby(
        [
            "material_group_name",
            "sap_weight_unit"
        ]
    )["weight_ratio"].describe()
)

In [ ]:
# Classify common SAP-to-PLM weight scaling patterns
def classify_weight_scale(ratio):
    if pd.isna(ratio):
        return "unknown"

    if np.isclose(ratio, 1, rtol=0.05):
        return "same_scale"

    if np.isclose(ratio, 1000, rtol=0.05):
        return "sap_x1000"

    if np.isclose(ratio, 100, rtol=0.05):
        return "sap_x100"

    if np.isclose(ratio, 0.001, rtol=0.05):
        return "plm_x1000"

    return "other"


same_unit_weight_pairs["scale_pattern"] = (
    same_unit_weight_pairs["weight_ratio"]
    .apply(classify_weight_scale)
)

display(
    same_unit_weight_pairs.groupby(
        [
            "material_group_name",
            "sap_weight_unit",
            "scale_pattern"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["material_group_name", "sap_weight_unit", "count"],
        ascending=[True, True, False]
    )
)

In [ ]:
# Compare SAP and PLM weights while accounting for SAP x1000 scaling
def calculate_weight_similarity(
    sap_weight,
    plm_weight,
    sap_unit=None,
    plm_unit=None
):
    if (
        pd.isna(sap_weight)
        or pd.isna(plm_weight)
        or sap_weight <= 0
        or plm_weight <= 0
    ):
        return pd.Series(
            [np.nan, np.nan, pd.NA]
        )

    # If both units exist and disagree, do not compare them directly
    if (
        pd.notna(sap_unit)
        and pd.notna(plm_unit)
        and sap_unit != plm_unit
    ):
        return pd.Series(
            [np.nan, np.nan, "unit_mismatch"]
        )

    candidates = {
        "same_scale": sap_weight,
        "sap_x1000": sap_weight / 1000
    }

    relative_errors = {
        scale: abs(value - plm_weight) / plm_weight
        for scale, value in candidates.items()
    }

    best_scale = min(
        relative_errors,
        key=relative_errors.get
    )

    best_error = relative_errors[best_scale]

    # 1 = perfect match, approaches 0 as the error increases
    similarity = 1 / (1 + best_error)

    return pd.Series(
        [
            similarity,
            best_error,
            best_scale
        ]
    )

In [ ]:
# Calculate scale-aware weight similarity for known mappings
known_weight_audit[
    [
        "weight_similarity",
        "weight_relative_error",
        "weight_scale"
    ]
] = known_weight_audit.apply(
    lambda row: calculate_weight_similarity(
        row["sap_weight_raw"],
        row["plm_weight"],
        row["sap_weight_unit"],
        row["plm_weight_unit"]
    ),
    axis=1
)

In [ ]:
# Evaluate weight agreement
comparable_weight_pairs = known_weight_audit[
    known_weight_audit["weight_similarity"].notna()
].copy()

print(
    "Comparable weight pairs:",
    len(comparable_weight_pairs)
)

print(
    "Exact / nearly exact (<= 1% error):",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.01).mean():.2%}"
)

print(
    "Within 5%:",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.05).mean():.2%}"
)

print(
    "Within 10%:",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.10).mean():.2%}"
)

display(
    comparable_weight_pairs[
        "weight_scale"
    ].value_counts()
)

In [ ]:
display(
    comparable_weight_pairs
    .groupby("material_group_name")
    .agg(
        comparable_pairs=("material_id", "size"),
        within_5_percent=(
            "weight_relative_error",
            lambda x: (x <= 0.05).mean()
        ),
        within_10_percent=(
            "weight_relative_error",
            lambda x: (x <= 0.10).mean()
        )
    )
)

In [ ]:
# Keep only material groups currently in modeling scope
model_material_groups = [
    "1020001",  # ORME
    "1020002",  # DOKUMA
    "1030004"   # DENIM
]

known_material_groups_model = (
    known_material_groups[
        known_material_groups["material_group_code"]
        .isin(model_material_groups)
    ]
    .copy()
)

In [ ]:
# Prepare SAP fabric structure values
sap_fabric_structure = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FABRICSTRUCTURETNAME"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri":
            "sap_fabric_structure"
        }
    )
)

sap_fabric_structure["sap_fabric_structure"] = (
    sap_fabric_structure["sap_fabric_structure"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Prepare PLM fabric structure values
plm_fabric_structure = (
    plm_codes[
        [
            "plm_code",
            "Kumaş tipi"
        ]
    ]
    .copy()
    .rename(
        columns={
            "Kumaş tipi":
            "plm_fabric_structure"
        }
    )
)

plm_fabric_structure["plm_fabric_structure"] = (
    plm_fabric_structure["plm_fabric_structure"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Compare SAP and PLM fabric structure values
fabric_structure_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_fabric_structure,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_fabric_structure,
        on="plm_code",
        how="inner"
    )
)

fabric_structure_pairs = fabric_structure_pairs[
    fabric_structure_pairs["sap_fabric_structure"].notna()
    & fabric_structure_pairs["plm_fabric_structure"].notna()
].copy()

fabric_structure_pairs["fabric_structure_match"] = (
    fabric_structure_pairs["sap_fabric_structure"]
    == fabric_structure_pairs["plm_fabric_structure"]
)

print(
    "Comparable fabric structure pairs:",
    len(fabric_structure_pairs)
)

print(
    "Exact match rate:",
    f"{fabric_structure_pairs['fabric_structure_match'].mean():.2%}"
)

display(
    fabric_structure_pairs
    .groupby("material_group_name")
    ["fabric_structure_match"]
    .agg(
        comparable_pairs="count",
        exact_match_rate="mean"
    )
)

In [ ]:
# Inspect common fabric-structure mismatches
display(
    fabric_structure_pairs[
        ~fabric_structure_pairs["fabric_structure_match"]
    ]
    .groupby(
        [
            "sap_fabric_structure",
            "plm_fabric_structure"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

In [ ]:
# Normalize only clearly equivalent fabric-structure labels
fabric_structure_mapping = {
    "POLYVISCON": "POLYVISCOSE",
    "BEZAYAĞI": "PLAINWEAVE"
}

fabric_structure_pairs["sap_fabric_structure_normalized"] = (
    fabric_structure_pairs["sap_fabric_structure"]
    .replace(fabric_structure_mapping)
)

fabric_structure_pairs["plm_fabric_structure_normalized"] = (
    fabric_structure_pairs["plm_fabric_structure"]
    .replace(fabric_structure_mapping)
)

fabric_structure_pairs["fabric_structure_match_normalized"] = (
    fabric_structure_pairs["sap_fabric_structure_normalized"]
    == fabric_structure_pairs["plm_fabric_structure_normalized"]
)

print(
    "Normalized exact match rate:",
    f"{fabric_structure_pairs['fabric_structure_match_normalized'].mean():.2%}"
)

In [ ]:
# Prepare SAP coloring values
sap_coloring = (
    fabric_attributes[
        fabric_attributes["field_name"] == "COLORINGID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_coloring["coloring_value"] = (
    sap_coloring["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sap_coloring_sets = (
    sap_coloring
    .groupby("material_id")["coloring_value"]
    .apply(set)
    .rename("sap_coloring_set")
    .reset_index()
)

In [ ]:
# Prepare PLM coloring values from the multi-value characteristic table
plm_coloring = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "COLORINGID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_coloring["coloring_value"] = (
    plm_coloring["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_coloring_sets = (
    plm_coloring
    .groupby("plm_code")["coloring_value"]
    .apply(set)
    .rename("plm_coloring_set")
    .reset_index()
)

In [ ]:
# Compare coloring sets for known SAP-PLM mappings
coloring_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_coloring_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_coloring_sets,
        on="plm_code",
        how="inner"
    )
)

coloring_pairs["coloring_similarity"] = (
    coloring_pairs.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_coloring_set"],
            row["plm_coloring_set"]
        ),
        axis=1
    )
)

print(
    "Comparable coloring pairs:",
    len(coloring_pairs)
)

print(
    "Exact coloring-set match:",
    f"{coloring_pairs['coloring_similarity'].eq(1).mean():.2%}"
)

display(
    coloring_pairs
    .groupby("material_group_name")
    ["coloring_similarity"]
    .agg(
        comparable_pairs="count",
        mean_similarity="mean",
        exact_match_rate=lambda x: x.eq(1).mean()
    )
)

In [ ]:
# Inspect common coloring mismatches
coloring_mismatches = (
    coloring_pairs[
        coloring_pairs["coloring_similarity"] < 1
    ].copy()
)

# Convert sets to hashable tuples for counting
coloring_mismatches["sap_coloring_tuple"] = (
    coloring_mismatches["sap_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

coloring_mismatches["plm_coloring_tuple"] = (
    coloring_mismatches["plm_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

display(
    coloring_mismatches[
        [
            "sap_coloring_tuple",
            "plm_coloring_tuple"
        ]
    ]
    .value_counts()
    .head(30)
)

In [ ]:
display(
    coloring_mismatches[
        [
            "material_group_name",
            "sap_coloring_tuple",
            "plm_coloring_tuple"
        ]
    ]
    .value_counts()
    .head(40)
)

In [ ]:
# Prepare SAP COLORINGID and DYETYPEID values
sap_coloring_dye = (
    fabric_attributes[
        fabric_attributes["field_name"].isin(
            ["COLORINGID", "DYETYPEID"]
        )
    ][
        [
            "material_id",
            "field_name",
            "Karakteristik değeri"
        ]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_coloring_dye["value"] = (
    sap_coloring_dye["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Create SAP value sets by characteristic
sap_coloring_dye_sets = (
    sap_coloring_dye
    .groupby(
        ["material_id", "field_name"]
    )["value"]
    .apply(set)
    .unstack()
    .reset_index()
    .rename(
        columns={
            "COLORINGID": "sap_coloring_set",
            "DYETYPEID": "sap_dye_type_set"
        }
    )
)

In [ ]:
# Prepare PLM COLORINGID and DYETYPEID values
plm_coloring_dye = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ].isin(
            ["COLORINGID", "DYETYPEID"]
        )
    ][
        [
            "plm_code",
            "PLM Karakteristik Tanımı",
            "PLM Karakteristik Değeri"
        ]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_coloring_dye["value"] = (
    plm_coloring_dye["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_coloring_dye_sets = (
    plm_coloring_dye
    .groupby(
        [
            "plm_code",
            "PLM Karakteristik Tanımı"
        ]
    )["value"]
    .apply(set)
    .unstack()
    .reset_index()
    .rename(
        columns={
            "COLORINGID": "plm_coloring_set",
            "DYETYPEID": "plm_dye_type_set"
        }
    )
)

In [ ]:
# Build cross-field comparison table
coloring_semantic_audit = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_coloring_dye_sets,
        on="material_id",
        how="left"
    )
    .merge(
        plm_coloring_dye_sets,
        on="plm_code",
        how="left"
    )
)

In [ ]:
# Compare same-field and cross-field similarities
comparison_pairs = {
    "coloring_to_coloring": (
        "sap_coloring_set",
        "plm_coloring_set"
    ),
    "dye_to_dye": (
        "sap_dye_type_set",
        "plm_dye_type_set"
    ),
    "dye_to_coloring": (
        "sap_dye_type_set",
        "plm_coloring_set"
    ),
    "coloring_to_dye": (
        "sap_coloring_set",
        "plm_dye_type_set"
    )
}

for score_name, (sap_col, plm_col) in comparison_pairs.items():
    coloring_semantic_audit[score_name] = (
        coloring_semantic_audit.apply(
            lambda row: calculate_jaccard_similarity(
                row[sap_col],
                row[plm_col]
            ),
            axis=1
        )
    )

In [ ]:
# Summarize semantic agreement by material group
semantic_summary = []

for material_group, group in (
    coloring_semantic_audit.groupby(
        "material_group_name"
    )
):
    for score_name in comparison_pairs:

        comparable = group[
            score_name
        ].dropna()

        semantic_summary.append(
            {
                "material_group_name": material_group,
                "comparison": score_name,
                "comparable_pairs": len(comparable),
                "mean_similarity": comparable.mean(),
                "exact_match_rate":
                    comparable.eq(1).mean()
            }
        )

semantic_summary = pd.DataFrame(
    semantic_summary
)

display(semantic_summary)

In [ ]:
# Prepare SAP knitting yarn count
sap_yarn_count_knit = (
    fabric_attributes[
        fabric_attributes["field_name"] == "YARNCOUNT1KNITSID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_yarn_count"
        }
    )
)

sap_yarn_count_knit["sap_yarn_count"] = (
    sap_yarn_count_knit["sap_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Prepare PLM knitting yarn count
plm_yarn_count_knit = (
    plm_codes[
        [
            "plm_code",
            "1.İplik numarası örme"
        ]
    ]
    .copy()
    .rename(
        columns={
            "1.İplik numarası örme": "plm_yarn_count"
        }
    )
)

plm_yarn_count_knit["plm_yarn_count"] = (
    plm_yarn_count_knit["plm_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Compare knitting yarn counts for known knitted-fabric mappings
yarn_count_pairs = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020001"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_yarn_count_knit,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_yarn_count_knit,
        on="plm_code",
        how="inner"
    )
)

yarn_count_pairs = yarn_count_pairs[
    yarn_count_pairs["sap_yarn_count"].notna()
    & yarn_count_pairs["plm_yarn_count"].notna()
].copy()

yarn_count_pairs["yarn_count_match"] = (
    yarn_count_pairs["sap_yarn_count"]
    == yarn_count_pairs["plm_yarn_count"]
)

print(
    "Comparable yarn-count pairs:",
    len(yarn_count_pairs)
)

print(
    "Exact match rate:",
    f"{yarn_count_pairs['yarn_count_match'].mean():.2%}"
)

In [ ]:
# Inspect common yarn-count mismatches
display(
    yarn_count_pairs[
        ~yarn_count_pairs["yarn_count_match"]
    ]
    .groupby(
        [
            "sap_yarn_count",
            "plm_yarn_count"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

In [ ]:
import re

# Parse yarn count into structured components
def parse_yarn_count(value):
    if pd.isna(value):
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": pd.NA
        }

    value = str(value).strip().upper()

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value)

    # Detect count system
    unit = pd.NA

    if re.search(r"\bNE\b", value):
        unit = "NE"

    elif re.search(r"\bD\b", value):
        unit = "DENIER"

    # Extract count and optional ply
    match = re.search(
        r"(\d+(?:\.\d+)?)"
        r"(?:\s*/\s*(\d+))?",
        value
    )

    if not match:
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": unit
        }

    count = float(match.group(1))

    ply = (
        float(match.group(2))
        if match.group(2)
        else np.nan
    )

    return {
        "count": count,
        "ply": ply,
        "unit": unit
    }

In [ ]:
# Compare parsed yarn counts conservatively
def calculate_yarn_count_match(sap_value, plm_value):

    sap = parse_yarn_count(sap_value)
    plm = parse_yarn_count(plm_value)

    if pd.isna(sap["count"]) or pd.isna(plm["count"]):
        return np.nan

    # Main yarn count must agree
    if sap["count"] != plm["count"]:
        return False

    # If both units are known, they must agree
    if (
        pd.notna(sap["unit"])
        and pd.notna(plm["unit"])
        and sap["unit"] != plm["unit"]
    ):
        return False

    # If both ply values are known, they must agree
    if (
        pd.notna(sap["ply"])
        and pd.notna(plm["ply"])
        and sap["ply"] != plm["ply"]
    ):
        return False

    # /1 and missing ply are treated as equivalent
    explicit_ply = (
        sap["ply"]
        if pd.notna(sap["ply"])
        else plm["ply"]
    )

    if pd.notna(explicit_ply) and explicit_ply != 1:
        # Do not assume that missing ply equals /2, /3, etc.
        if pd.isna(sap["ply"]) or pd.isna(plm["ply"]):
            return False

    return True

In [ ]:
# Recalculate yarn-count agreement after semantic normalization
yarn_count_pairs["yarn_count_semantic_match"] = (
    yarn_count_pairs.apply(
        lambda row: calculate_yarn_count_match(
            row["sap_yarn_count"],
            row["plm_yarn_count"]
        ),
        axis=1
    )
)

comparable_semantic_yarn_pairs = (
    yarn_count_pairs[
        yarn_count_pairs[
            "yarn_count_semantic_match"
        ].notna()
    ]
)

print(
    "Semantic match rate:",
    f"{comparable_semantic_yarn_pairs['yarn_count_semantic_match'].mean():.2%}"
)

print(
    "Raw exact match rate:",
    f"{yarn_count_pairs['yarn_count_match'].mean():.2%}"
)

In [ ]:
# Prepare SAP knitting yarn type values
sap_yarn_type_knit = (
    fabric_attributes[
        fabric_attributes["field_name"] == "YARN1TYPEKNITSID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_yarn_type_knit["yarn_type"] = (
    sap_yarn_type_knit["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sap_yarn_type_sets = (
    sap_yarn_type_knit
    .groupby("material_id")["yarn_type"]
    .apply(set)
    .rename("sap_yarn_type_set")
    .reset_index()
)

In [ ]:
# Prepare PLM knitting yarn type values
plm_yarn_type_knit = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "YARN1TYPEKNITSID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_yarn_type_knit["yarn_type"] = (
    plm_yarn_type_knit["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_yarn_type_sets = (
    plm_yarn_type_knit
    .groupby("plm_code")["yarn_type"]
    .apply(set)
    .rename("plm_yarn_type_set")
    .reset_index()
)

In [ ]:
# Compare knitting yarn types for known knitted-fabric mappings
yarn_type_pairs = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020001"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_yarn_type_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_yarn_type_sets,
        on="plm_code",
        how="inner"
    )
)

yarn_type_pairs["yarn_type_similarity"] = (
    yarn_type_pairs.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_yarn_type_set"],
            row["plm_yarn_type_set"]
        ),
        axis=1
    )
)

print(
    "Comparable yarn-type pairs:",
    len(yarn_type_pairs)
)

print(
    "Exact yarn-type match:",
    f"{yarn_type_pairs['yarn_type_similarity'].eq(1).mean():.2%}"
)

print(
    "Mean similarity:",
    f"{yarn_type_pairs['yarn_type_similarity'].mean():.3f}"
)

In [ ]:
# Inspect common yarn-type mismatches
yarn_type_mismatches = (
    yarn_type_pairs[
        yarn_type_pairs["yarn_type_similarity"] < 1
    ].copy()
)

yarn_type_mismatches["sap_yarn_type_tuple"] = (
    yarn_type_mismatches["sap_yarn_type_set"]
    .apply(lambda x: tuple(sorted(x)))
)

yarn_type_mismatches["plm_yarn_type_tuple"] = (
    yarn_type_mismatches["plm_yarn_type_set"]
    .apply(lambda x: tuple(sorted(x)))
)

display(
    yarn_type_mismatches[
        [
            "sap_yarn_type_tuple",
            "plm_yarn_type_tuple"
        ]
    ]
    .value_counts()
    .head(30)
)

In [ ]:
# Normalize only clearly equivalent yarn-type labels
yarn_type_mapping = {
    "VORTEX": "VORTEKS",
    "COMBEDCOMPACT": "COMPACTCOMBED"
}


def normalize_yarn_type(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    return yarn_type_mapping.get(
        value,
        value
    )

In [ ]:
# Normalize SAP yarn-type sets
sap_yarn_type_sets_normalized = (
    sap_yarn_type_knit
    .assign(
        yarn_type_normalized=lambda df:
            df["yarn_type"].apply(normalize_yarn_type)
    )
    .groupby("material_id")["yarn_type_normalized"]
    .apply(set)
    .rename("sap_yarn_type_set")
    .reset_index()
)

# Normalize PLM yarn-type sets
plm_yarn_type_sets_normalized = (
    plm_yarn_type_knit
    .assign(
        yarn_type_normalized=lambda df:
            df["yarn_type"].apply(normalize_yarn_type)
    )
    .groupby("plm_code")["yarn_type_normalized"]
    .apply(set)
    .rename("plm_yarn_type_set")
    .reset_index()
)

In [ ]:
# Recalculate yarn-type similarity after safe normalization
yarn_type_pairs_normalized = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020001"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_yarn_type_sets_normalized,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_yarn_type_sets_normalized,
        on="plm_code",
        how="inner"
    )
)

yarn_type_pairs_normalized["yarn_type_similarity"] = (
    yarn_type_pairs_normalized.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_yarn_type_set"],
            row["plm_yarn_type_set"]
        ),
        axis=1
    )
)

print(
    "Comparable yarn-type pairs:",
    len(yarn_type_pairs_normalized)
)

print(
    "Normalized exact match:",
    f"{yarn_type_pairs_normalized['yarn_type_similarity'].eq(1).mean():.2%}"
)

print(
    "Mean similarity:",
    f"{yarn_type_pairs_normalized['yarn_type_similarity'].mean():.3f}"
)

In [ ]:
# Prepare SAP weft yarn count values
sap_weft_yarn_count = (
    fabric_attributes[
        fabric_attributes["field_name"] == "WEFTYARNCOUNT1ID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_weft_yarn_count"
        }
    )
)

sap_weft_yarn_count["sap_weft_yarn_count"] = (
    sap_weft_yarn_count["sap_weft_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Prepare PLM weft yarn count values
plm_weft_yarn_count = (
    plm_codes[
        [
            "plm_code",
            "1.Atkı iplik numarası"
        ]
    ]
    .copy()
    .rename(
        columns={
            "1.Atkı iplik numarası":
            "plm_weft_yarn_count"
        }
    )
)

plm_weft_yarn_count["plm_weft_yarn_count"] = (
    plm_weft_yarn_count["plm_weft_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
# Compare weft yarn counts for known woven-fabric mappings
weft_yarn_count_pairs = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020002"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_weft_yarn_count,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_weft_yarn_count,
        on="plm_code",
        how="inner"
    )
)

weft_yarn_count_pairs = (
    weft_yarn_count_pairs[
        weft_yarn_count_pairs["sap_weft_yarn_count"].notna()
        & weft_yarn_count_pairs["plm_weft_yarn_count"].notna()
    ]
    .copy()
)

In [ ]:
# Calculate raw and semantic yarn-count agreement
weft_yarn_count_pairs["raw_match"] = (
    weft_yarn_count_pairs["sap_weft_yarn_count"]
    == weft_yarn_count_pairs["plm_weft_yarn_count"]
)

weft_yarn_count_pairs["semantic_match"] = (
    weft_yarn_count_pairs.apply(
        lambda row: calculate_yarn_count_match(
            row["sap_weft_yarn_count"],
            row["plm_weft_yarn_count"]
        ),
        axis=1
    )
)

comparable_weft_pairs = (
    weft_yarn_count_pairs[
        weft_yarn_count_pairs["semantic_match"].notna()
    ]
)

print(
    "Comparable weft yarn-count pairs:",
    len(comparable_weft_pairs)
)

print(
    "Raw exact match rate:",
    f"{comparable_weft_pairs['raw_match'].mean():.2%}"
)

print(
    "Semantic match rate:",
    f"{comparable_weft_pairs['semantic_match'].mean():.2%}"
)

In [ ]:
# Inspect semantic mismatches
display(
    comparable_weft_pairs[
        ~comparable_weft_pairs["semantic_match"]
    ]
    .groupby(
        [
            "sap_weft_yarn_count",
            "plm_weft_yarn_count"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

In [ ]:
def parse_yarn_count(value):
    if pd.isna(value):
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": pd.NA
        }

    value = str(value).strip().upper()

    # Normalize decimal separator
    value = value.replace(",", ".")

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value)

    # Detect yarn count system
    unit = pd.NA

    if re.search(r"\bNE\b", value):
        unit = "NE"

    elif re.search(r"\bD\b", value):
        unit = "DENIER"

    # Extract count and optional ply
    match = re.search(
        r"(\d+(?:\.\d+)?)"
        r"(?:\s*/\s*(\d+(?:\.\d+)?))?",
        value
    )

    if not match:
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": unit
        }

    count = float(match.group(1))

    ply = (
        float(match.group(2))
        if match.group(2)
        else np.nan
    )

    return {
        "count": count,
        "ply": ply,
        "unit": unit
    }

In [ ]:
# Build empirical feature-agreement results
agreement_records = []

# WEIGHT
weight_group_map = {
    "ORME": "1020001",
    "DOKUMA": "1020002",
    "DENIM": "1030004"
}

for group_name, group in comparable_weight_pairs.groupby("material_group_name"):
    if group_name not in weight_group_map:
        continue

    agreement_records.append({
        "material_group": weight_group_map[group_name],
        "field_name": "WEIGHT",
        "agreement_rate": (group["weight_relative_error"] <= 0.05).mean(),
        "agreement_metric": "within_5_percent"
    })

# FABRICSTRUCTURETNAME
for group_name, group in fabric_structure_pairs.groupby("material_group_name"):
    agreement_records.append({
        "material_group": weight_group_map[group_name],
        "field_name": "FABRICSTRUCTURETNAME",
        "agreement_rate": group["fabric_structure_match_normalized"].mean(),
        "agreement_metric": "normalized_exact_match"
    })

# COLORINGID
for group_name, group in coloring_pairs.groupby("material_group_name"):
    agreement_records.append({
        "material_group": weight_group_map[group_name],
        "field_name": "COLORINGID",
        "agreement_rate": group["coloring_similarity"].eq(1).mean(),
        "agreement_metric": "exact_set_match"
    })

# ORME - YARNCOUNT1KNITSID
agreement_records.append({
    "material_group": "1020001",
    "field_name": "YARNCOUNT1KNITSID",
    "agreement_rate": yarn_count_pairs[
        "yarn_count_semantic_match"
    ].mean(),
    "agreement_metric": "semantic_match"
})

# ORME - YARN1TYPEKNITSID
agreement_records.append({
    "material_group": "1020001",
    "field_name": "YARN1TYPEKNITSID",
    "agreement_rate": yarn_type_pairs_normalized[
        "yarn_type_similarity"
    ].eq(1).mean(),
    "agreement_metric": "normalized_exact_set_match"
})

# DOKUMA - WEFTYARNCOUNT1ID
agreement_records.append({
    "material_group": "1020002",
    "field_name": "WEFTYARNCOUNT1ID",
    "agreement_rate": comparable_weft_pairs[
        "semantic_match"
    ].mean(),
    "agreement_metric": "semantic_match"
})

feature_agreement = pd.DataFrame(agreement_records)

In [ ]:
# Add previously measured agreement results
previous_agreement = pd.DataFrame([
    {
        "material_group": "1020001",
        "field_name": "FIBERCONTENTLISTID",
        "agreement_rate": 0.9804,
        "agreement_metric": "exact_fiber_set_match_global"
    },
    {
        "material_group": "1020002",
        "field_name": "FIBERCONTENTLISTID",
        "agreement_rate": 0.9804,
        "agreement_metric": "exact_fiber_set_match_global"
    },
    {
        "material_group": "1030004",
        "field_name": "FIBERCONTENTLISTID",
        "agreement_rate": 0.9804,
        "agreement_metric": "exact_fiber_set_match_global"
    },
    {
        "material_group": "1020002",
        "field_name": "WEAVETYPE",
        "agreement_rate": 0.9715,
        "agreement_metric": "exact_match_previous_audit"
    },
    {
        "material_group": "1030004",
        "field_name": "WEAVETYPE",
        "agreement_rate": 0.9715,
        "agreement_metric": "exact_match_previous_audit"
    }
])

feature_agreement = pd.concat(
    [feature_agreement, previous_agreement],
    ignore_index=True
)

In [ ]:
print(feature_policy_audit.columns.tolist())

In [ ]:
# Prepare feature policy audit for merging
final_feature_base = feature_policy_audit.copy()

# Standardize material group column name
if "material_group" not in final_feature_base.columns:
    if "material_group_code" in final_feature_base.columns:
        final_feature_base = final_feature_base.rename(
            columns={
                "material_group_code": "material_group"
            }
        )
    else:
        raise KeyError(
            "Neither 'material_group' nor 'material_group_code' "
            "exists in feature_policy_audit."
        )

# Standardize data types before merge
final_feature_base["material_group"] = (
    final_feature_base["material_group"]
    .astype("string")
)

feature_agreement["material_group"] = (
    feature_agreement["material_group"]
    .astype("string")
)

# Remove old agreement columns if this cell is rerun
columns_to_remove = [
    "agreement_rate",
    "agreement_metric",
    "evidence_score"
]

final_feature_base = final_feature_base.drop(
    columns=[
        col
        for col in columns_to_remove
        if col in final_feature_base.columns
    ],
    errors="ignore"
)

# Combine business policy, coverage and empirical agreement
final_feature_audit = (
    final_feature_base
    .merge(
        feature_agreement,
        on=["material_group", "field_name"],
        how="left"
    )
)

final_feature_audit["evidence_score"] = (
    final_feature_audit["effective_coverage"]
    * final_feature_audit["agreement_rate"]
)

In [ ]:
# Keep only audited features in modeling scope
model_feature_audit = (
    final_feature_audit[
        final_feature_audit["material_group"].isin(
            ["1020001", "1020002", "1030004"]
        )
        & final_feature_audit["agreement_rate"].notna()
    ]
    .copy()
)

display(
    model_feature_audit[
        [
            "material_group",
            "field_name",
            "feature_role",
            "effective_coverage",
            "agreement_rate",
            "agreement_metric",
            "evidence_score"
        ]
    ]
    .sort_values(
        ["material_group", "evidence_score"],
        ascending=[True, False]
    )
)

In [ ]:
# Assign final feature tiers using coverage, agreement and business role
def assign_final_feature_tier(row):
    coverage = row["effective_coverage"]
    agreement = row["agreement_rate"]
    role = row["feature_role"]

    if pd.isna(agreement):
        return "NOT_AUDITED"

    # Strong and broadly available evidence
    if coverage >= 0.80 and agreement >= 0.90:
        return "PRIMARY"

    # Reliable when available
    if coverage >= 0.40 and agreement >= 0.75:
        return "SECONDARY"

    # Business-important fields with moderate agreement
    if (
        role in ["CORE", "MANDATORY"]
        and coverage >= 0.40
        and agreement >= 0.70
    ):
        return "SECONDARY"

    if coverage >= 0.10 and agreement >= 0.70:
        return "OPTIONAL"

    return "EXCLUDE"


model_feature_audit["final_feature_tier"] = (
    model_feature_audit.apply(
        assign_final_feature_tier,
        axis=1
    )
)

display(
    model_feature_audit[
        [
            "material_group",
            "field_name",
            "feature_role",
            "effective_coverage",
            "agreement_rate",
            "evidence_score",
            "final_feature_tier"
        ]
    ]
    .sort_values(
        ["material_group", "final_feature_tier", "evidence_score"],
        ascending=[True, True, False]
    )
)

In [ ]:
import re
import unicodedata

# Normalize text for retrieval
def normalize_retrieval_text(value):
    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    # Normalize whitespace and punctuation
    value = re.sub(r"[^A-ZÇĞİÖŞÜ0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

In [ ]:
# Build SAP retrieval text
sap_candidates_source = (
    fabric_materials[
        [
            "material_id",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması"
        ]
    ]
    .copy()
)

sap_candidates_source["material_group_code"] = (
    sap_candidates_source["Mal grubu"]
    .map({
        "ORME": "1020001",
        "DOKUMA": "1020002",
        "DENIM": "1030004"
    })
)

sap_candidates_source["retrieval_text"] = (
    sap_candidates_source[
        "Türkçe malzeme açıklaması"
    ].fillna("")
    + " "
    + sap_candidates_source[
        "Türkçe malzeme Uzun açıklaması"
    ].fillna("")
)

sap_candidates_source["retrieval_text"] = (
    sap_candidates_source["retrieval_text"]
    .apply(normalize_retrieval_text)
)

In [ ]:
# Build PLM retrieval text
plm_candidates_source = (
    plm_codes[
        [
            "plm_code",
            "material_group_code",
            "Türkçe malzeme açıklaması",
            "Malzeme Türkçe Adı",
            "Malzeme ingilizce adı"
        ]
    ]
    .copy()
)

plm_candidates_source["retrieval_text"] = (
    plm_candidates_source[
        "Türkçe malzeme açıklaması"
    ].fillna("")
    + " "
    + plm_candidates_source[
        "Malzeme Türkçe Adı"
    ].fillna("")
    + " "
    + plm_candidates_source[
        "Malzeme ingilizce adı"
    ].fillna("")
)

plm_candidates_source["retrieval_text"] = (
    plm_candidates_source["retrieval_text"]
    .apply(normalize_retrieval_text)
)

In [ ]:
# Check retrieval-text availability
print(
    "SAP materials:",
    len(sap_candidates_source)
)

print(
    "SAP with retrieval text:",
    (
        sap_candidates_source["retrieval_text"]
        .str.len()
        .gt(0)
        .mean()
    )
)

print(
    "PLM codes:",
    len(plm_candidates_source)
)

print(
    "PLM with retrieval text:",
    (
        plm_candidates_source["retrieval_text"]
        .str.len()
        .gt(0)
        .mean()
    )
)

In [ ]:
display(
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .sample(10, random_state=42)
)

display(
    plm_candidates_source[
        [
            "plm_code",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .sample(10, random_state=42)
)

In [ ]:
# PLM fields useful for candidate retrieval
plm_retrieval_fields = [
    "Kumaş tipi",
    "Dokuma tipi",
    "Örme alt tipi",
    "1.İplik numarası örme",
    "1.Atkı iplik numarası",
    "1.Çözgü iplik numarası",
    "Kumaş ağırlığı"
]


def combine_retrieval_fields(row, fields):
    values = []

    for field in fields:
        value = row.get(field)

        if pd.notna(value):
            value = normalize_retrieval_text(value)

            if value:
                values.append(value)

    return " ".join(values)


plm_structured_text = plm_codes[
    ["plm_code"] + plm_retrieval_fields
].copy()

plm_structured_text["structured_text"] = (
    plm_structured_text.apply(
        lambda row: combine_retrieval_fields(
            row,
            plm_retrieval_fields
        ),
        axis=1
    )
)

In [ ]:
# Characteristics to include in retrieval representation
retrieval_multi_fields = [
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
]

plm_multi_retrieval = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ].isin(retrieval_multi_fields)
    ]
    .copy()
)

plm_multi_retrieval["retrieval_value"] = (
    plm_multi_retrieval[
        "PLM Karakteristik Değeri"
    ]
    .apply(normalize_retrieval_text)
)

plm_multi_text = (
    plm_multi_retrieval
    .groupby("plm_code")["retrieval_value"]
    .apply(
        lambda values:
            " ".join(sorted(set(values)))
    )
    .rename("multi_value_text")
    .reset_index()
)

In [ ]:
# Enrich PLM retrieval text with structured characteristics
plm_candidates_enriched = (
    plm_candidates_source
    .merge(
        plm_structured_text[
            ["plm_code", "structured_text"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_multi_text,
        on="plm_code",
        how="left"
    )
)

plm_candidates_enriched["structured_text"] = (
    plm_candidates_enriched[
        "structured_text"
    ].fillna("")
)

plm_candidates_enriched["multi_value_text"] = (
    plm_candidates_enriched[
        "multi_value_text"
    ].fillna("")
)

plm_candidates_enriched["retrieval_text_enriched"] = (
    plm_candidates_enriched["retrieval_text"]
    + " "
    + plm_candidates_enriched["structured_text"]
    + " "
    + plm_candidates_enriched["multi_value_text"]
).str.strip()

In [ ]:
# Compare original and enriched PLM retrieval coverage
original_coverage = (
    plm_candidates_enriched["retrieval_text"]
    .str.len()
    .gt(0)
    .mean()
)

enriched_coverage = (
    plm_candidates_enriched[
        "retrieval_text_enriched"
    ]
    .str.len()
    .gt(0)
    .mean()
)

print(
    "Original PLM text coverage:",
    f"{original_coverage:.2%}"
)

print(
    "Enriched PLM text coverage:",
    f"{enriched_coverage:.2%}"
)

In [ ]:
display(
    plm_candidates_enriched
    .assign(
        has_retrieval_text=lambda df:
            df["retrieval_text_enriched"]
            .str.len()
            .gt(0)
    )
    .groupby("material_group_code")
    .agg(
        plm_codes=("plm_code", "nunique"),
        retrieval_coverage=(
            "has_retrieval_text",
            "mean"
        )
    )
)

In [ ]:
display(
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "retrieval_text",
            "structured_text",
            "multi_value_text",
            "retrieval_text_enriched"
        ]
    ]
    .sample(10, random_state=42)
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [ ]:
# Generate top-k PLM candidates using character n-gram TF-IDF
def generate_tfidf_candidates(
    sap_df,
    plm_df,
    material_group_code,
    top_k=50
):
    sap_group = (
        sap_df[
            sap_df["material_group_code"] == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    plm_group = (
        plm_df[
            plm_df["material_group_code"] == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Keep only rows with usable retrieval text
    sap_group = sap_group[
        sap_group["retrieval_text"].str.len() > 0
    ].reset_index(drop=True)

    plm_group = plm_group[
        plm_group["retrieval_text_enriched"].str.len() > 0
    ].reset_index(drop=True)

    # Character n-grams are robust to spelling and formatting differences
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        sublinear_tf=True,
        norm="l2"
    )

    plm_matrix = vectorizer.fit_transform(
        plm_group["retrieval_text_enriched"]
    )

    sap_matrix = vectorizer.transform(
        sap_group["retrieval_text"]
    )

    n_neighbors = min(
        top_k,
        len(plm_group)
    )

    nn_model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute"
    )

    nn_model.fit(plm_matrix)

    distances, indices = nn_model.kneighbors(
        sap_matrix
    )

    candidate_records = []

    for sap_idx in range(len(sap_group)):
        for rank, (plm_idx, distance) in enumerate(
            zip(indices[sap_idx], distances[sap_idx]),
            start=1
        ):
            candidate_records.append({
                "material_id":
                    sap_group.loc[sap_idx, "material_id"],
                "material_group_code":
                    material_group_code,
                "candidate_plm_code":
                    plm_group.loc[plm_idx, "plm_code"],
                "candidate_rank":
                    rank,
                "retrieval_similarity":
                    1 - distance
            })

    return pd.DataFrame(candidate_records)

In [ ]:
# Generate candidates for each material group
candidate_tables = []

for material_group_code in [
    "1020001",  # ORME
    "1020002",  # DOKUMA
    "1030004"   # DENIM
]:
    group_candidates = generate_tfidf_candidates(
        sap_candidates_source,
        plm_candidates_enriched,
        material_group_code=material_group_code,
        top_k=50
    )

    candidate_tables.append(
        group_candidates
    )

tfidf_candidates = pd.concat(
    candidate_tables,
    ignore_index=True
)

print(
    "Candidate pairs:",
    len(tfidf_candidates)
)

display(
    tfidf_candidates.head(10)
)

In [ ]:
# Prepare known SAP-PLM mappings for retrieval evaluation
known_retrieval_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_code"
        ]
    ]
    .dropna(subset=["plm_code"])
    .copy()
)

retrieval_evaluation = (
    known_retrieval_pairs
    .merge(
        tfidf_candidates,
        left_on=[
            "material_id",
            "material_group_code",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "material_group_code",
            "candidate_plm_code"
        ],
        how="left"
    )
)

In [ ]:
# Calculate candidate-generation recall
def calculate_recall_at_k(df, k):
    return (
        df["candidate_rank"]
        .le(k)
        .fillna(False)
        .mean()
    )


for k in [1, 3, 5, 10, 20, 50]:
    print(
        f"Recall@{k}:",
        f"{calculate_recall_at_k(retrieval_evaluation, k):.2%}"
    )

In [ ]:
# Evaluate retrieval recall by material group
retrieval_group_summary = []

for group_name, group in (
    retrieval_evaluation
    .merge(
        known_material_groups_model[
            [
                "material_id",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
    .groupby("material_group_name")
):
    retrieval_group_summary.append({
        "material_group_name": group_name,
        "known_pairs": len(group),
        "recall_at_1":
            calculate_recall_at_k(group, 1),
        "recall_at_5":
            calculate_recall_at_k(group, 5),
        "recall_at_10":
            calculate_recall_at_k(group, 10),
        "recall_at_20":
            calculate_recall_at_k(group, 20),
        "recall_at_50":
            calculate_recall_at_k(group, 50)
    })

retrieval_group_summary = pd.DataFrame(
    retrieval_group_summary
)

display(retrieval_group_summary)

In [ ]:
# Add diagnostic flags to retrieval evaluation
materials_with_attributes = set(
    fabric_attributes["material_id"]
    .dropna()
    .astype("string")
)

plm_original_text_lookup = (
    plm_candidates_source[
        ["plm_code", "retrieval_text"]
    ]
    .drop_duplicates("plm_code")
    .set_index("plm_code")["retrieval_text"]
)

retrieval_diagnostic = (
    retrieval_evaluation.copy()
)

retrieval_diagnostic["has_sap_attributes"] = (
    retrieval_diagnostic["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

retrieval_diagnostic["target_plm_has_original_text"] = (
    retrieval_diagnostic["plm_code"]
    .map(plm_original_text_lookup)
    .fillna("")
    .str.len()
    .gt(0)
)

In [ ]:
# Recall by SAP attribute availability
attribute_recall_summary = []

for has_attributes, group in (
    retrieval_diagnostic
    .groupby("has_sap_attributes")
):
    attribute_recall_summary.append({
        "has_sap_attributes": has_attributes,
        "known_pairs": len(group),
        "recall_at_10": calculate_recall_at_k(group, 10),
        "recall_at_20": calculate_recall_at_k(group, 20),
        "recall_at_50": calculate_recall_at_k(group, 50)
    })

display(
    pd.DataFrame(attribute_recall_summary)
)

In [ ]:
# Recall by target PLM description availability
target_text_recall_summary = []

for has_text, group in (
    retrieval_diagnostic
    .groupby("target_plm_has_original_text")
):
    target_text_recall_summary.append({
        "target_plm_has_original_text": has_text,
        "known_pairs": len(group),
        "recall_at_10": calculate_recall_at_k(group, 10),
        "recall_at_20": calculate_recall_at_k(group, 20),
        "recall_at_50": calculate_recall_at_k(group, 50)
    })

display(
    pd.DataFrame(target_text_recall_summary)
)

In [ ]:
# Normalize fabric structure for candidate generation
def normalize_fabric_structure(value):
    if pd.isna(value):
        return pd.NA

    value = (
        str(value)
        .strip()
        .upper()
    )

    mapping = {
        "POLYVISCON": "POLYVISCOSE",
        "BEZAYAĞI": "PLAINWEAVE"
    }

    return mapping.get(value, value)

In [ ]:
# Prepare SAP fabric structures
sap_structure_candidates = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FABRICSTRUCTURETNAME"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .copy()
)

sap_structure_candidates["structure"] = (
    sap_structure_candidates[
        "Karakteristik değeri"
    ]
    .apply(normalize_fabric_structure)
)

In [ ]:
# Prepare PLM fabric structures
plm_structure_candidates = (
    plm_codes[
        [
            "plm_code",
            "material_group_code",
            "Kumaş tipi"
        ]
    ]
    .dropna(subset=["Kumaş tipi"])
    .copy()
)

plm_structure_candidates["structure"] = (
    plm_structure_candidates["Kumaş tipi"]
    .apply(normalize_fabric_structure)
)

In [ ]:
# Generate candidates from exact normalized fabric structure
structure_candidates = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "material_group_code",
                "structure"
            ]
        ],
        on=[
            "material_group_code",
            "structure"
        ],
        how="inner"
    )
    .rename(
        columns={
            "plm_code": "candidate_plm_code"
        }
    )
)

structure_candidates["candidate_source"] = (
    "fabric_structure"
)

print(
    "Structure candidate pairs:",
    len(structure_candidates)
)

print(
    "SAP materials with structure candidates:",
    structure_candidates[
        "material_id"
    ].nunique()
)

In [ ]:
# Evaluate structure-channel candidate recall
structure_recall = (
    known_retrieval_pairs
    .merge(
        structure_candidates[
            [
                "material_id",
                "candidate_plm_code"
            ]
        ].drop_duplicates(),
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

structure_recall["retrieved"] = (
    structure_recall["_merge"]
    == "both"
)

print(
    "Structure channel recall:",
    f"{structure_recall['retrieved'].mean():.2%}"
)

In [ ]:
structure_available = (
    structure_recall["material_id"]
    .isin(
        set(
            sap_structure_candidates[
                "material_id"
            ]
        )
    )
)

print(
    "Structure recall when SAP structure exists:",
    f"{structure_recall.loc[structure_available, 'retrieved'].mean():.2%}"
)

In [ ]:
# Prepare SAP structure + weight representation
sap_structure_weight = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="left"
    )
)

sap_structure_weight = sap_structure_weight[
    sap_structure_weight["sap_weight_raw"].notna()
    & (sap_structure_weight["sap_weight_raw"] > 0)
].copy()

In [ ]:
# Prepare PLM structure + weight representation
plm_structure_weight = (
    plm_structure_candidates[
        [
            "plm_code",
            "material_group_code",
            "structure"
        ]
    ]
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ],
        on="plm_code",
        how="left"
    )
)

plm_structure_weight = plm_structure_weight[
    plm_structure_weight["plm_weight"].notna()
    & (plm_structure_weight["plm_weight"] > 0)
].copy()

In [ ]:
# Build a fast lookup index
plm_structure_index = {
    key: group.reset_index(drop=True)
    for key, group in (
        plm_structure_weight
        .groupby(
            [
                "material_group_code",
                "structure"
            ],
            dropna=False
        )
    )
}

In [ ]:
# Generate candidates that agree on structure
# and are within a weight tolerance
def generate_structure_weight_candidates(
    sap_df,
    plm_index,
    weight_tolerance=0.10
):
    records = []

    for row in sap_df.itertuples(index=False):

        key = (
            row.material_group_code,
            row.structure
        )

        plm_pool = plm_index.get(key)

        if plm_pool is None or plm_pool.empty:
            continue

        sap_weight = row.sap_weight_raw
        sap_unit = row.sap_weight_unit

        plm_weights = (
            plm_pool["plm_weight"]
            .astype(float)
            .to_numpy()
        )

        # Test both observed SAP scale patterns
        same_scale_error = (
            np.abs(sap_weight - plm_weights)
            / plm_weights
        )

        x1000_error = (
            np.abs((sap_weight / 1000) - plm_weights)
            / plm_weights
        )

        best_error = np.minimum(
            same_scale_error,
            x1000_error
        )

        valid = best_error <= weight_tolerance

        # If both units exist, require agreement
        if pd.notna(sap_unit):

            plm_units = (
                plm_pool["plm_weight_unit"]
                .astype("string")
            )

            unit_valid = (
                plm_units.isna()
                | (plm_units == str(sap_unit))
            ).to_numpy()

            valid = valid & unit_valid

        candidate_indices = np.where(valid)[0]

        for idx in candidate_indices:

            records.append(
                (
                    row.material_id,
                    row.material_group_code,
                    plm_pool.iloc[idx]["plm_code"],
                    best_error[idx]
                )
            )

    return pd.DataFrame(
        records,
        columns=[
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    )

In [ ]:
structure_weight_candidates = (
    generate_structure_weight_candidates(
        sap_structure_weight,
        plm_structure_index,
        weight_tolerance=0.10
    )
)

structure_weight_candidates[
    "candidate_source"
] = "structure_weight"

print(
    "Structure + weight candidate pairs:",
    len(structure_weight_candidates)
)

print(
    "SAP materials covered:",
    structure_weight_candidates[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(structure_weight_candidates)
    / structure_weight_candidates["material_id"].nunique()
)

In [ ]:
# Evaluate composite structured channel
structure_weight_recall = (
    known_retrieval_pairs
    .merge(
        structure_weight_candidates[
            [
                "material_id",
                "candidate_plm_code"
            ]
        ].drop_duplicates(),
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

structure_weight_recall["retrieved"] = (
    structure_weight_recall["_merge"]
    == "both"
)

print(
    "Overall structure + weight recall:",
    f"{structure_weight_recall['retrieved'].mean():.2%}"
)

In [ ]:
# Evaluate only materials where both SAP features exist
available_structure_weight_materials = set(
    sap_structure_weight["material_id"]
)

available_mask = (
    structure_weight_recall["material_id"]
    .isin(available_structure_weight_materials)
)

print(
    "Conditional structure + weight recall:",
    f"{structure_weight_recall.loc[available_mask, 'retrieved'].mean():.2%}"
)

In [ ]:
# Combine text and structured candidate channels
text_candidate_pairs = (
    tfidf_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

structured_candidate_pairs = (
    structure_weight_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

combined_candidates = (
    pd.concat(
        [
            text_candidate_pairs,
            structured_candidate_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

combined_recall = (
    known_retrieval_pairs
    .merge(
        combined_candidates,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

print(
    "Text + structure/weight union recall:",
    f"{(combined_recall['_merge'] == 'both').mean():.2%}"
)

In [ ]:
# Evaluate combined candidate recall by SAP attribute availability
combined_recall_diagnostic = (
    known_retrieval_pairs
    .merge(
        combined_candidates,
        left_on=["material_id", "plm_code"],
        right_on=["material_id", "candidate_plm_code"],
        how="left",
        indicator=True
    )
)

combined_recall_diagnostic["retrieved"] = (
    combined_recall_diagnostic["_merge"] == "both"
)

combined_recall_diagnostic["has_sap_attributes"] = (
    combined_recall_diagnostic["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

display(
    combined_recall_diagnostic
    .groupby("has_sap_attributes")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

In [ ]:
# Normalize fiber names for retrieval
def normalize_fiber_for_retrieval(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    # Remove leading percentage / numeric composition
    value = re.sub(
        r"^\s*\d+(?:[.,]\d+)?\s*%?\s*",
        "",
        value
    ).strip()

    if not value or re.fullmatch(r"[\d.,]+", value):
        return pd.NA

    mapping = {
        "POLIESTER": "POLYESTER",
        "ELASTHANE": "ELASTANE",
        "SPANDEX": "ELASTANE",
        "POLYAMIDE6": "POLYAMIDE",
        "POLIAMID6": "POLYAMIDE",
        "TENCEL": "LYOCELL",
        "ACETAT": "ACETATE"
    }

    return mapping.get(value, value)

In [ ]:
# Prepare SAP fiber-name sets
sap_fiber_candidates = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FIBERCONTENTLISTID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .copy()
)

sap_fiber_candidates["fiber_name"] = (
    sap_fiber_candidates["Karakteristik değeri"]
    .apply(normalize_fiber_for_retrieval)
)

sap_fiber_sets = (
    sap_fiber_candidates
    .dropna(subset=["fiber_name"])
    .groupby("material_id")["fiber_name"]
    .apply(lambda x: tuple(sorted(set(x))))
    .rename("fiber_key")
    .reset_index()
)

In [ ]:
# Prepare PLM fiber-name sets
plm_fiber_candidates = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ] == "FIBERCONTENTLISTID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .copy()
)

plm_fiber_candidates["fiber_name"] = (
    plm_fiber_candidates[
        "PLM Karakteristik Değeri"
    ]
    .apply(normalize_fiber_for_retrieval)
)

plm_fiber_sets = (
    plm_fiber_candidates
    .dropna(subset=["fiber_name"])
    .groupby("plm_code")["fiber_name"]
    .apply(lambda x: tuple(sorted(set(x))))
    .rename("fiber_key")
    .reset_index()
)

In [ ]:
# SAP fiber + weight representation
sap_fiber_weight = (
    sap_candidates_source[
        ["material_id", "material_group_code"]
    ]
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

sap_fiber_weight = sap_fiber_weight[
    sap_fiber_weight["sap_weight_raw"].notna()
    & (sap_fiber_weight["sap_weight_raw"] > 0)
].copy()

In [ ]:
# PLM fiber + weight representation
plm_fiber_weight = (
    plm_codes[
        ["plm_code", "material_group_code"]
    ]
    .merge(
        plm_fiber_sets,
        on="plm_code",
        how="inner"
    )
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_fiber_weight = plm_fiber_weight[
    plm_fiber_weight["plm_weight"].notna()
    & (plm_fiber_weight["plm_weight"] > 0)
].copy()

In [ ]:
# Build PLM lookup by material group and fiber set
plm_fiber_index = {
    key: group.reset_index(drop=True)
    for key, group in (
        plm_fiber_weight
        .groupby(
            ["material_group_code", "fiber_key"],
            dropna=False
        )
    )
}

In [ ]:
# Generate exact-fiber + weight candidates
def generate_fiber_weight_candidates(
    sap_df,
    plm_index,
    weight_tolerance=0.10
):
    records = []

    for row in sap_df.itertuples(index=False):

        key = (
            row.material_group_code,
            row.fiber_key
        )

        plm_pool = plm_index.get(key)

        if plm_pool is None or plm_pool.empty:
            continue

        plm_weights = (
            plm_pool["plm_weight"]
            .astype(float)
            .to_numpy()
        )

        same_scale_error = (
            np.abs(row.sap_weight_raw - plm_weights)
            / plm_weights
        )

        x1000_error = (
            np.abs(
                (row.sap_weight_raw / 1000)
                - plm_weights
            )
            / plm_weights
        )

        best_error = np.minimum(
            same_scale_error,
            x1000_error
        )

        valid = best_error <= weight_tolerance

        # Only enforce unit when SAP unit is known
        if pd.notna(row.sap_weight_unit):

            plm_units = (
                plm_pool["plm_weight_unit"]
                .astype("string")
            )

            unit_valid = (
                plm_units.isna()
                | (
                    plm_units
                    == str(row.sap_weight_unit)
                )
            ).to_numpy()

            valid = valid & unit_valid

        for idx in np.where(valid)[0]:

            records.append({
                "material_id": row.material_id,
                "material_group_code":
                    row.material_group_code,
                "candidate_plm_code":
                    plm_pool.iloc[idx]["plm_code"],
                "weight_relative_error":
                    best_error[idx],
                "candidate_source":
                    "fiber_weight"
            })

    return pd.DataFrame(records)

In [ ]:
fiber_weight_candidates = (
    generate_fiber_weight_candidates(
        sap_fiber_weight,
        plm_fiber_index,
        weight_tolerance=0.10
    )
)

print(
    "Fiber + weight candidate pairs:",
    len(fiber_weight_candidates)
)

print(
    "SAP materials covered:",
    fiber_weight_candidates[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(fiber_weight_candidates)
    / fiber_weight_candidates["material_id"].nunique()
)

In [ ]:
fiber_weight_recall = (
    known_retrieval_pairs
    .merge(
        fiber_weight_candidates[
            ["material_id", "candidate_plm_code"]
        ].drop_duplicates(),
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

fiber_weight_recall["retrieved"] = (
    fiber_weight_recall["_merge"] == "both"
)

available_fiber_weight_materials = set(
    sap_fiber_weight["material_id"]
)

available_mask = (
    fiber_weight_recall["material_id"]
    .isin(available_fiber_weight_materials)
)

print(
    "Overall fiber + weight recall:",
    f"{fiber_weight_recall['retrieved'].mean():.2%}"
)

print(
    "Conditional fiber + weight recall:",
    f"{fiber_weight_recall.loc[available_mask, 'retrieved'].mean():.2%}"
)

In [ ]:
# Union of text, structure-weight and fiber-weight channels
all_candidate_pairs = (
    pd.concat(
        [
            tfidf_candidates[
                ["material_id", "candidate_plm_code"]
            ],
            structure_weight_candidates[
                ["material_id", "candidate_plm_code"]
            ],
            fiber_weight_candidates[
                ["material_id", "candidate_plm_code"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

all_candidate_recall = (
    known_retrieval_pairs
    .merge(
        all_candidate_pairs,
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

all_candidate_recall["retrieved"] = (
    all_candidate_recall["_merge"] == "both"
)

print(
    "Combined candidate recall:",
    f"{all_candidate_recall['retrieved'].mean():.2%}"
)

Bu sonuç mimariyi bayağı netleştirdi:

Attribute'u olan SAP'lerde: text + structure/weight ile bile recall %93,20.
fiber + weight tek başına, kullanılabildiği materyallerde %97,70 conditional recall veriyor.
Attribute'u olmayan SAP'lerde: şu an text-only recall sadece %30,54.
Dolayısıyla overall %72,42 düşük görünse de asıl problem structured tarafta değil; attribute'suz grup.

Şu aşamada iki ayrı pipeline düşünmek daha doğru:

A) SAP attributes available
   → structured candidate generation
   → text supplementary channel
   → pairwise reranking

B) SAP attributes unavailable
   → text-only / description-derived retrieval
   → lower-confidence recommendation

In [ ]:
# Evaluate final candidate union by SAP attribute availability
final_candidate_recall = (
    known_retrieval_pairs
    .merge(
        all_candidate_pairs,
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

final_candidate_recall["retrieved"] = (
    final_candidate_recall["_merge"] == "both"
)

final_candidate_recall["has_sap_attributes"] = (
    final_candidate_recall["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

display(
    final_candidate_recall
    .groupby("has_sap_attributes")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

In [ ]:
# Evaluate attribute-rich candidate recall by material group
final_candidate_recall_grouped = (
    final_candidate_recall
    .merge(
        known_material_groups_model[
            [
                "material_id",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
)

display(
    final_candidate_recall_grouped[
        final_candidate_recall_grouped[
            "has_sap_attributes"
        ]
    ]
    .groupby("material_group_name")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

In [ ]:
# Candidate pool size after all retrieval channels
candidate_pool_size = (
    all_candidate_pairs
    .groupby("material_id")
    .size()
    .rename("candidate_count")
)

print(
    candidate_pool_size.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
attribute_candidate_pool_size = (
    candidate_pool_size[
        candidate_pool_size.index.astype("string")
        .isin(materials_with_attributes)
    ]
)

print(
    attribute_candidate_pool_size.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
# Preserve retrieval-source information
text_channel = (
    tfidf_candidates[
        [
            "material_id",
            "candidate_plm_code",
            "retrieval_similarity"
        ]
    ]
    .copy()
)

text_channel["from_text"] = 1

structure_channel = (
    structure_weight_candidates[
        [
            "material_id",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    ]
    .copy()
)

structure_channel["from_structure_weight"] = 1

fiber_channel = (
    fiber_weight_candidates[
        [
            "material_id",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    ]
    .copy()
)

fiber_channel["from_fiber_weight"] = 1

In [ ]:
# Create candidate master table with retrieval-channel flags
candidate_master = (
    all_candidate_pairs
    .merge(
        text_channel,
        on=["material_id", "candidate_plm_code"],
        how="left"
    )
    .merge(
        structure_channel,
        on=["material_id", "candidate_plm_code"],
        how="left",
        suffixes=("", "_structure")
    )
    .merge(
        fiber_channel,
        on=["material_id", "candidate_plm_code"],
        how="left",
        suffixes=("", "_fiber")
    )
)

for column in [
    "from_text",
    "from_structure_weight",
    "from_fiber_weight"
]:
    candidate_master[column] = (
        candidate_master[column]
        .fillna(0)
        .astype(int)
    )

In [ ]:
# Count how many independent retrieval channels selected each pair
candidate_master["retrieval_channel_count"] = (
    candidate_master[
        [
            "from_text",
            "from_structure_weight",
            "from_fiber_weight"
        ]
    ].sum(axis=1)
)

display(
    candidate_master[
        "retrieval_channel_count"
    ].value_counts()
)

In [ ]:
print(candidate_master.columns.tolist())

In [ ]:
# Add material group back to candidate master
material_group_lookup = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .drop_duplicates("material_id")
)

candidate_master = (
    candidate_master
    .drop(
        columns=["material_group_code"],
        errors="ignore"
    )
    .merge(
        material_group_lookup,
        on="material_id",
        how="left"
    )
)

In [ ]:
print(
    candidate_master[
        "material_group_code"
    ].value_counts(dropna=False)
)

print(
    candidate_master.columns.tolist()
)

In [ ]:
# Prepare known mappings with SAP attributes
known_attribute_mappings = (
    known_retrieval_pairs[
        known_retrieval_pairs["material_id"]
        .astype("string")
        .isin(materials_with_attributes)
    ]
    [
        [
            "material_id",
            "plm_code",
            "material_group_code"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "plm_code": "true_plm_code"
        }
    )
    .copy()
)

# Standardize merge-key data types
candidate_master["material_id"] = (
    candidate_master["material_id"]
    .astype("string")
)

candidate_master["material_group_code"] = (
    candidate_master["material_group_code"]
    .astype("string")
)

known_attribute_mappings["material_id"] = (
    known_attribute_mappings["material_id"]
    .astype("string")
)

known_attribute_mappings["material_group_code"] = (
    known_attribute_mappings["material_group_code"]
    .astype("string")
)

In [ ]:
# Build labeled candidate-pair dataset
pairwise_dataset = (
    candidate_master
    .merge(
        known_attribute_mappings,
        on=[
            "material_id",
            "material_group_code"
        ],
        how="inner"
    )
)

pairwise_dataset["is_match"] = (
    pairwise_dataset["candidate_plm_code"]
    == pairwise_dataset["true_plm_code"]
).astype(int)

print(
    "Pairwise rows:",
    len(pairwise_dataset)
)

print(
    "SAP materials:",
    pairwise_dataset["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_dataset["is_match"].sum()
)

print(
    "Positive rate:",
    f"{pairwise_dataset['is_match'].mean():.4%}"
)

In [ ]:
# Compare the material group used by known mappings
# with the material group used during candidate generation

candidate_group_lookup = (
    candidate_master[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group_code":
            "candidate_material_group"
        }
    )
)

group_alignment_audit = (
    known_attribute_mappings
    .merge(
        candidate_group_lookup,
        on="material_id",
        how="left"
    )
)

group_alignment_audit["group_match"] = (
    group_alignment_audit["material_group_code"]
    == group_alignment_audit["candidate_material_group"]
)

print(
    "Known attribute mappings:",
    len(group_alignment_audit)
)

print(
    "Unique known materials:",
    group_alignment_audit[
        "material_id"
    ].nunique()
)

print(
    "Missing from candidate master:",
    group_alignment_audit[
        "candidate_material_group"
    ].isna().sum()
)

print(
    "Material-group mismatches:",
    (~group_alignment_audit["group_match"])
    .sum()
)

In [ ]:
# Inspect material-group inconsistencies
display(
    group_alignment_audit[
        ~group_alignment_audit["group_match"]
    ][
        [
            "material_id",
            "true_plm_code",
            "material_group_code",
            "candidate_material_group"
        ]
    ]
)

In [ ]:
# Check which known mappings were not retrieved
positive_candidate_check = (
    known_attribute_mappings
    .merge(
        candidate_master[
            [
                "material_id",
                "candidate_plm_code"
            ]
        ].drop_duplicates(),
        left_on=[
            "material_id",
            "true_plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

positive_candidate_check["true_plm_retrieved"] = (
    positive_candidate_check[
        "candidate_plm_code"
    ].notna()
)

print(
    "Known mappings:",
    len(positive_candidate_check)
)

print(
    "True PLM retrieved:",
    positive_candidate_check[
        "true_plm_retrieved"
    ].sum()
)

print(
    "True PLM missed:",
    (
        ~positive_candidate_check[
            "true_plm_retrieved"
        ]
    ).sum()
)

In [ ]:
# Add SAP fiber representation
pairwise_dataset = (
    pairwise_dataset
    .merge(
        sap_fiber_sets.rename(
            columns={
                "fiber_key":
                "sap_fiber_key"
            }
        ),
        on="material_id",
        how="left"
    )
)

# Add PLM fiber representation
pairwise_dataset = (
    pairwise_dataset
    .merge(
        plm_fiber_sets.rename(
            columns={
                "plm_code":
                "candidate_plm_code",
                "fiber_key":
                "plm_fiber_key"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

In [ ]:
# Calculate fiber-set similarity
def tuple_jaccard(left, right):
    if not isinstance(left, tuple) or not isinstance(right, tuple):
        return np.nan

    left_set = set(left)
    right_set = set(right)

    if not left_set or not right_set:
        return np.nan

    return (
        len(left_set & right_set)
        / len(left_set | right_set)
    )


pairwise_dataset["fiber_similarity"] = [
    tuple_jaccard(left, right)
    for left, right in zip(
        pairwise_dataset["sap_fiber_key"],
        pairwise_dataset["plm_fiber_key"]
    )
]

In [ ]:
# Add normalized SAP fabric structure
pairwise_dataset = (
    pairwise_dataset
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ].rename(
            columns={
                "structure":
                "sap_structure"
            }
        ),
        on="material_id",
        how="left"
    )
)

# Add normalized PLM fabric structure
pairwise_dataset = (
    pairwise_dataset
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ].rename(
            columns={
                "plm_code":
                "candidate_plm_code",
                "structure":
                "plm_structure"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

pairwise_dataset["fabric_structure_match"] = np.where(
    pairwise_dataset["sap_structure"].notna()
    & pairwise_dataset["plm_structure"].notna(),
    (
        pairwise_dataset["sap_structure"]
        == pairwise_dataset["plm_structure"]
    ).astype(float),
    np.nan
)

In [ ]:
feature_columns = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "fiber_similarity",
    "fabric_structure_match"
]

display(
    pairwise_dataset
    .groupby("is_match")[
        feature_columns
    ]
    .mean()
    .T
    .rename(
        columns={
            0: "non_match_mean",
            1: "match_mean"
        }
    )
)

In [ ]:
display(
    pairwise_dataset[
        pairwise_dataset["is_match"] == 1
    ][feature_columns]
    .notna()
    .mean()
    .rename(
        "positive_feature_coverage"
    )
    .to_frame()
)

In [ ]:
# Audit candidate-generation misses
missed_mappings = (
    positive_candidate_check[
        ~positive_candidate_check["true_plm_retrieved"]
    ][
        [
            "material_id",
            "true_plm_code",
            "material_group_code"
        ]
    ]
    .copy()
)

missed_mappings = (
    missed_mappings
    .merge(
        candidate_group_lookup,
        on="material_id",
        how="left"
    )
)

missed_mappings["miss_reason"] = np.where(
    missed_mappings["material_group_code"]
    != missed_mappings["candidate_material_group"],
    "material_group_mismatch",
    "retrieval_miss_same_group"
)

display(
    missed_mappings[
        "miss_reason"
    ].value_counts()
)

In [ ]:
# Keep only materials whose true PLM reached the candidate pool
retrieved_material_ids = set(
    pairwise_dataset.loc[
        pairwise_dataset["is_match"] == 1,
        "material_id"
    ]
)

pairwise_training = (
    pairwise_dataset[
        pairwise_dataset["material_id"]
        .isin(retrieved_material_ids)
    ]
    .copy()
)

print(
    "Training materials:",
    pairwise_training["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_training["is_match"].sum()
)

print(
    "Materials without a positive:",
    (
        pairwise_training
        .groupby("material_id")["is_match"]
        .sum()
        .eq(0)
        .sum()
    )
)

In [ ]:
# Add SAP weight data to every candidate pair
pairwise_training = (
    pairwise_training
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="left"
    )
)

# Add PLM weight data to every candidate pair
pairwise_training = (
    pairwise_training
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ].rename(
            columns={
                "plm_code": "candidate_plm_code"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

In [ ]:
# Vectorized scale-aware weight similarity
valid_weight = (
    pairwise_training["sap_weight_raw"].notna()
    & pairwise_training["plm_weight"].notna()
    & (pairwise_training["sap_weight_raw"] > 0)
    & (pairwise_training["plm_weight"] > 0)
)

same_scale_error = (
    abs(
        pairwise_training["sap_weight_raw"]
        - pairwise_training["plm_weight"]
    )
    / pairwise_training["plm_weight"]
)

x1000_error = (
    abs(
        pairwise_training["sap_weight_raw"] / 1000
        - pairwise_training["plm_weight"]
    )
    / pairwise_training["plm_weight"]
)

pairwise_training["weight_relative_error_all"] = (
    np.minimum(
        same_scale_error,
        x1000_error
    )
)

# If both units exist and disagree, comparison is invalid
unit_mismatch = (
    pairwise_training["sap_weight_unit"].notna()
    & pairwise_training["plm_weight_unit"].notna()
    & (
        pairwise_training["sap_weight_unit"]
        != pairwise_training["plm_weight_unit"]
    )
)

pairwise_training.loc[
    ~valid_weight | unit_mismatch,
    "weight_relative_error_all"
] = np.nan

pairwise_training["weight_similarity"] = (
    1 / (
        1
        + pairwise_training[
            "weight_relative_error_all"
        ]
    )
)

In [ ]:
feature_columns = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match"
]

display(
    pairwise_training
    .groupby("is_match")[feature_columns]
    .mean()
    .T
    .rename(
        columns={
            0: "non_match_mean",
            1: "match_mean"
        }
    )
)

display(
    pairwise_training[
        pairwise_training["is_match"] == 1
    ][feature_columns]
    .notna()
    .mean()
    .rename("positive_feature_coverage")
    .to_frame()
)

In [ ]:
# Convert coloring sets to canonical tuples
sap_coloring_model = sap_coloring_sets.copy()
plm_coloring_model = plm_coloring_sets.copy()

sap_coloring_model["sap_coloring_key"] = (
    sap_coloring_model["sap_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

plm_coloring_model["plm_coloring_key"] = (
    plm_coloring_model["plm_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

pairwise_training = (
    pairwise_training
    .merge(
        sap_coloring_model[
            ["material_id", "sap_coloring_key"]
        ],
        on="material_id",
        how="left"
    )
    .merge(
        plm_coloring_model[
            ["plm_code", "plm_coloring_key"]
        ].rename(
            columns={
                "plm_code": "candidate_plm_code"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

pairwise_training["coloring_match"] = np.where(
    pairwise_training["sap_coloring_key"].notna()
    & pairwise_training["plm_coloring_key"].notna(),
    (
        pairwise_training["sap_coloring_key"]
        == pairwise_training["plm_coloring_key"]
    ).astype(float),
    np.nan
)

In [ ]:
# Prepare SAP weave type
sap_weave_type = (
    fabric_attributes[
        fabric_attributes["field_name"] == "WEAVETYPE"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna()
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_weave_type"
        }
    )
)

sap_weave_type["sap_weave_type"] = (
    sap_weave_type["sap_weave_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Prepare PLM weave type
plm_weave_type = (
    plm_codes[
        ["plm_code", "Dokuma tipi"]
    ]
    .copy()
    .rename(
        columns={
            "Dokuma tipi": "plm_weave_type"
        }
    )
)

plm_weave_type["plm_weave_type"] = (
    plm_weave_type["plm_weave_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [ ]:
pairwise_training = (
    pairwise_training
    .merge(
        sap_weave_type,
        on="material_id",
        how="left"
    )
    .merge(
        plm_weave_type.rename(
            columns={
                "plm_code": "candidate_plm_code"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

pairwise_training["weave_type_match"] = np.where(
    pairwise_training["sap_weave_type"].notna()
    & pairwise_training["plm_weave_type"].notna(),
    (
        pairwise_training["sap_weave_type"]
        == pairwise_training["plm_weave_type"]
    ).astype(float),
    np.nan
)

In [ ]:
model_features = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match",
    "weave_type_match"
]

display(
    pairwise_training
    .groupby("is_match")[model_features]
    .mean()
    .T
    .rename(
        columns={
            0: "non_match_mean",
            1: "match_mean"
        }
    )
)

In [ ]:
# Build a simple heuristic only for selecting difficult negatives
pairwise_training["hardness_score"] = (
    pairwise_training["fiber_similarity"].fillna(0)
    + pairwise_training["weight_similarity"].fillna(0)
    + pairwise_training["fabric_structure_match"].fillna(0)
    + 0.5 * pairwise_training["coloring_match"].fillna(0)
    + 0.25 * pairwise_training["retrieval_channel_count"]
    + 0.25 * pairwise_training["retrieval_similarity"].fillna(0)
)

In [ ]:
positive_pairs = (
    pairwise_training[
        pairwise_training["is_match"] == 1
    ]
    .copy()
)

negative_pairs = (
    pairwise_training[
        pairwise_training["is_match"] == 0
    ]
    .copy()
)

In [ ]:
# Keep the most confusing negative candidates per SAP material
hard_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(20)
)

In [ ]:
# Add random negatives for broader coverage
negative_pairs["random_score"] = np.random.default_rng(
    42
).random(len(negative_pairs))

random_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

In [ ]:
pairwise_sample = (
    pd.concat(
        [
            positive_pairs,
            hard_negatives,
            random_negatives
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=[
            "material_id",
            "candidate_plm_code"
        ]
    )
)

print(
    "Sample rows:",
    len(pairwise_sample)
)

print(
    "Materials:",
    pairwise_sample["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_sample["is_match"].sum()
)

print(
    "Positive rate:",
    f"{pairwise_sample['is_match'].mean():.2%}"
)

In [ ]:
from sklearn.model_selection import train_test_split

material_split = (
    pairwise_sample[
        ["material_id", "material_group_code"]
    ]
    .drop_duplicates("material_id")
)

train_materials, test_materials = train_test_split(
    material_split,
    test_size=0.20,
    random_state=42,
    stratify=material_split["material_group_code"]
)

train_ids = set(train_materials["material_id"])
test_ids = set(test_materials["material_id"])

train_df = pairwise_sample[
    pairwise_sample["material_id"].isin(train_ids)
].copy()

test_df = pairwise_sample[
    pairwise_sample["material_id"].isin(test_ids)
].copy()

print("Train materials:", len(train_ids))
print("Test materials:", len(test_ids))

In [ ]:
model_features = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "from_text",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match",
    "weave_type_match"
]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Build an interpretable baseline reranker
logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])


In [ ]:
X_train = train_df[model_features]
y_train = train_df["is_match"]

logistic_model.fit(
    X_train,
    y_train
)

print("Model trained.")

In [ ]:
# Evaluate on all candidates, not only sampled negatives
test_full = (
    pairwise_training[
        pairwise_training["material_id"]
        .isin(test_ids)
    ]
    .copy()
)

print(
    "Test materials:",
    test_full["material_id"].nunique()
)

print(
    "Full test candidate pairs:",
    len(test_full)
)

print(
    "Test positives:",
    test_full["is_match"].sum()
)

In [ ]:
# Predict match probabilities for every candidate
test_full["match_probability"] = (
    logistic_model.predict_proba(
        test_full[model_features]
    )[:, 1]
)

In [ ]:
# Rank PLM candidates within each SAP material
test_full["predicted_rank"] = (
    test_full
    .groupby("material_id")[
        "match_probability"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

In [ ]:
# Keep the true PLM row for each test material
true_match_ranks = (
    test_full[
        test_full["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "true_plm_code",
            "predicted_rank",
            "match_probability"
        ]
    ]
    .copy()
)

def top_k_accuracy(df, k):
    return (
        df["predicted_rank"] <= k
    ).mean()

print(
    "Top-1:",
    f"{top_k_accuracy(true_match_ranks, 1):.2%}"
)

print(
    "Top-3:",
    f"{top_k_accuracy(true_match_ranks, 3):.2%}"
)

print(
    "Top-5:",
    f"{top_k_accuracy(true_match_ranks, 5):.2%}"
)

print(
    "Top-10:",
    f"{top_k_accuracy(true_match_ranks, 10):.2%}"
)

mrr = (
    1 / true_match_ranks["predicted_rank"]
).mean()

print(
    "MRR:",
    f"{mrr:.4f}"
)

In [ ]:
# Ranking performance by material group
group_ranking_summary = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, group in (
    true_match_ranks
    .groupby("material_group_code")
):
    group_ranking_summary.append({
        "material_group":
            group_name_mapping.get(
                group_code,
                group_code
            ),
        "test_materials":
            len(group),
        "top_1":
            top_k_accuracy(group, 1),
        "top_3":
            top_k_accuracy(group, 3),
        "top_5":
            top_k_accuracy(group, 5),
        "top_10":
            top_k_accuracy(group, 10),
        "mrr":
            (1 / group["predicted_rank"]).mean()
    })

display(
    pd.DataFrame(
        group_ranking_summary
    )
)

In [ ]:
print(
    true_match_ranks[
        "predicted_rank"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
# Inspect Logistic Regression feature coefficients
imputer = logistic_model.named_steps["imputer"]
classifier = logistic_model.named_steps["model"]

feature_names = imputer.get_feature_names_out(
    model_features
)

coefficient_table = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": classifier.coef_[0]
    })
    .sort_values(
        "coefficient",
        ascending=False
    )
)

display(coefficient_table)

In [ ]:
group_ranking_summary = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, group in (
    true_match_ranks
    .groupby("material_group_code")
):
    group_ranking_summary.append({
        "material_group":
            group_name_mapping.get(group_code, group_code),
        "test_materials": len(group),
        "top_1":
            (group["predicted_rank"] <= 1).mean(),
        "top_3":
            (group["predicted_rank"] <= 3).mean(),
        "top_5":
            (group["predicted_rank"] <= 5).mean(),
        "top_10":
            (group["predicted_rank"] <= 10).mean(),
        "mrr":
            (1 / group["predicted_rank"]).mean()
    })

display(
    pd.DataFrame(group_ranking_summary)
)

In [ ]:
# Initialize group-specific yarn features
pairwise_training["knit_yarn_count_match"] = np.nan
pairwise_training["knit_yarn_type_similarity"] = np.nan
pairwise_training["weft_yarn_count_match"] = np.nan

In [ ]:
# Build knitting yarn-count lookups
sap_knit_yarn_count_lookup = (
    sap_yarn_count_knit
    .set_index("material_id")["sap_yarn_count"]
    .to_dict()
)

plm_knit_yarn_count_lookup = (
    plm_yarn_count_knit
    .set_index("plm_code")["plm_yarn_count"]
    .to_dict()
)

orme_mask = (
    pairwise_training["material_group_code"]
    == "1020001"
)

pairwise_training.loc[
    orme_mask,
    "knit_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_knit_yarn_count_lookup.get(material_id),
        plm_knit_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training.loc[orme_mask, "material_id"],
        pairwise_training.loc[orme_mask, "candidate_plm_code"]
    )
]

In [ ]:
# Build normalized knitting yarn-type lookups
sap_knit_yarn_type_lookup = (
    sap_yarn_type_sets_normalized
    .set_index("material_id")["sap_yarn_type_set"]
    .to_dict()
)

plm_knit_yarn_type_lookup = (
    plm_yarn_type_sets_normalized
    .set_index("plm_code")["plm_yarn_type_set"]
    .to_dict()
)


def safe_set_jaccard(left, right):
    if not isinstance(left, set) or not isinstance(right, set):
        return np.nan

    if not left or not right:
        return np.nan

    return len(left & right) / len(left | right)


pairwise_training.loc[
    orme_mask,
    "knit_yarn_type_similarity"
] = [
    safe_set_jaccard(
        sap_knit_yarn_type_lookup.get(material_id),
        plm_knit_yarn_type_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training.loc[orme_mask, "material_id"],
        pairwise_training.loc[orme_mask, "candidate_plm_code"]
    )
]

In [ ]:
# Build woven weft yarn-count lookups
sap_weft_yarn_count_lookup = (
    sap_weft_yarn_count
    .set_index("material_id")["sap_weft_yarn_count"]
    .to_dict()
)

plm_weft_yarn_count_lookup = (
    plm_weft_yarn_count
    .set_index("plm_code")["plm_weft_yarn_count"]
    .to_dict()
)

dokuma_mask = (
    pairwise_training["material_group_code"]
    == "1020002"
)

pairwise_training.loc[
    dokuma_mask,
    "weft_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_weft_yarn_count_lookup.get(material_id),
        plm_weft_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training.loc[dokuma_mask, "material_id"],
        pairwise_training.loc[dokuma_mask, "candidate_plm_code"]
    )
]

In [ ]:
# Rebuild sampled dataset with newly added features
sample_keys = (
    pairwise_sample[
        ["material_id", "candidate_plm_code"]
    ]
    .drop_duplicates()
)

pairwise_sample_v2 = (
    sample_keys
    .merge(
        pairwise_training,
        on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="inner"
    )
)

print("Sample rows:", len(pairwise_sample_v2))
print(
    "Materials:",
    pairwise_sample_v2["material_id"].nunique()
)

In [ ]:
# Common non-redundant features
common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

group_features = {
    "1020001": common_features + [
        "knit_yarn_count_match",
        "knit_yarn_type_similarity"
    ],

    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

group_models = {}
group_results = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, features in group_features.items():

    train_group = pairwise_sample_v2[
        pairwise_sample_v2["material_id"].isin(train_ids)
        & (
            pairwise_sample_v2["material_group_code"]
            == group_code
        )
    ].copy()

    # Full candidate pool for unbiased ranking evaluation
    test_group = pairwise_training[
        pairwise_training["material_id"].isin(test_ids)
        & (
            pairwise_training["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    test_group["match_probability"] = (
        model.predict_proba(
            test_group[features]
        )[:, 1]
    )

    test_group["predicted_rank"] = (
        test_group
        .groupby("material_id")[
            "match_probability"
        ]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

    true_ranks = test_group[
        test_group["is_match"] == 1
    ].copy()

    group_models[group_code] = model

    group_results.append({
        "material_group":
            group_name_mapping[group_code],
        "test_materials":
            len(true_ranks),
        "top_1":
            (true_ranks["predicted_rank"] <= 1).mean(),
        "top_3":
            (true_ranks["predicted_rank"] <= 3).mean(),
        "top_5":
            (true_ranks["predicted_rank"] <= 5).mean(),
        "top_10":
            (true_ranks["predicted_rank"] <= 10).mean(),
        "mrr":
            (
                1 / true_ranks["predicted_rank"]
            ).mean()
    })

group_results = pd.DataFrame(group_results)

display(group_results)


In [ ]:
# Weighted overall ranking performance
total_test_materials = (
    group_results["test_materials"].sum()
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    weighted_metric = (
        (
            group_results[metric]
            * group_results["test_materials"]
        ).sum()
        / total_test_materials
    )

    print(
        f"{metric}:",
        f"{weighted_metric:.4f}"
    )

In [ ]:
# Check duplicate SAP-PLM candidate pairs
duplicate_pair_summary = (
    pairwise_training
    .groupby(
        ["material_id", "candidate_plm_code"]
    )
    .size()
    .reset_index(name="row_count")
)

print(
    "Unique candidate pairs:",
    len(duplicate_pair_summary)
)

print(
    "Duplicated candidate pairs:",
    (duplicate_pair_summary["row_count"] > 1).sum()
)

print(
    "Maximum duplicate count:",
    duplicate_pair_summary["row_count"].max()
)

display(
    duplicate_pair_summary[
        duplicate_pair_summary["row_count"] > 1
    ]
    .sort_values(
        "row_count",
        ascending=False
    )
    .head(20)
)

In [ ]:
print(
    "Pairwise training rows:",
    len(pairwise_training)
)

print(
    "Unique SAP-PLM pairs:",
    pairwise_training[
        ["material_id", "candidate_plm_code"]
    ].drop_duplicates().shape[0]
)

In [ ]:
# Check whether duplication already exists in candidate master
candidate_master_duplicates = (
    candidate_master
    .groupby(
        ["material_id", "candidate_plm_code"]
    )
    .size()
)

print(
    "Candidate master duplicated pairs:",
    (candidate_master_duplicates > 1).sum()
)

print(
    "Candidate master maximum duplicate count:",
    candidate_master_duplicates.max()
)

In [ ]:
# Aggregate text channel to one row per SAP-PLM pair
text_channel_clean = (
    tfidf_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        retrieval_similarity=(
            "retrieval_similarity",
            "max"
        )
    )
)

text_channel_clean["from_text"] = 1

In [ ]:
# Aggregate structure-weight channel
structure_channel_clean = (
    structure_weight_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        structure_weight_error=(
            "weight_relative_error",
            "min"
        )
    )
)

structure_channel_clean[
    "from_structure_weight"
] = 1

In [ ]:
# Aggregate fiber-weight channel
fiber_channel_clean = (
    fiber_weight_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        fiber_weight_error=(
            "weight_relative_error",
            "min"
        )
    )
)

fiber_channel_clean[
    "from_fiber_weight"
] = 1

In [ ]:
# Rebuild a strictly one-row-per-pair candidate master
candidate_master_clean = (
    all_candidate_pairs[
        ["material_id", "candidate_plm_code"]
    ]
    .drop_duplicates()
    .merge(
        text_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        structure_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        fiber_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        material_group_lookup,
        on="material_id",
        how="left",
        validate="many_to_one"
    )
)

for column in [
    "from_text",
    "from_structure_weight",
    "from_fiber_weight"
]:
    candidate_master_clean[column] = (
        candidate_master_clean[column]
        .fillna(0)
        .astype(int)
    )

candidate_master_clean["retrieval_channel_count"] = (
    candidate_master_clean[
        [
            "from_text",
            "from_structure_weight",
            "from_fiber_weight"
        ]
    ].sum(axis=1)
)

In [ ]:
# Candidate pair must be unique
assert not candidate_master_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

print(
    "Clean candidate pairs:",
    len(candidate_master_clean)
)

In [ ]:
# Define the unique pair key
pair_key = [
    "material_id",
    "candidate_plm_code"
]

assert not candidate_master_clean.duplicated(pair_key).any()

# Rebuild labeled candidate pairs from the clean candidate master
pairwise_dataset_clean = (
    candidate_master_clean
    .merge(
        known_attribute_mappings,
        on=[
            "material_id",
            "material_group_code"
        ],
        how="inner",
        validate="many_to_one"
    )
)

pairwise_dataset_clean["is_match"] = (
    pairwise_dataset_clean["candidate_plm_code"]
    == pairwise_dataset_clean["true_plm_code"]
).astype(int)

print(
    "Pairwise rows:",
    len(pairwise_dataset_clean)
)

print(
    "Unique pairs:",
    pairwise_dataset_clean[pair_key]
    .drop_duplicates()
    .shape[0]
)

assert not pairwise_dataset_clean.duplicated(pair_key).any()

In [ ]:
# Keep only materials with a positive candidate
retrieved_material_ids = set(
    pairwise_dataset_clean.loc[
        pairwise_dataset_clean["is_match"] == 1,
        "material_id"
    ]
)

pairwise_training_clean = (
    pairwise_dataset_clean[
        pairwise_dataset_clean["material_id"]
        .isin(retrieved_material_ids)
    ]
    .copy()
)

print(
    "Training materials:",
    pairwise_training_clean[
        "material_id"
    ].nunique()
)

print(
    "Positive pairs:",
    pairwise_training_clean[
        "is_match"
    ].sum()
)

print(
    "Duplicate pairs:",
    pairwise_training_clean
    .duplicated(pair_key)
    .sum()
)

In [ ]:
# Build scalar lookup dictionaries
sap_structure_lookup = (
    sap_structure_candidates
    .drop_duplicates("material_id")
    .set_index("material_id")["structure"]
    .to_dict()
)

plm_structure_lookup = (
    plm_structure_candidates
    .drop_duplicates("plm_code")
    .set_index("plm_code")["structure"]
    .to_dict()
)

sap_weight_raw_lookup = (
    sap_weight_audit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weight_raw"]
    .to_dict()
)

sap_weight_unit_lookup = (
    sap_weight_audit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weight_unit"]
    .to_dict()
)

plm_weight_value_lookup = (
    plm_weight_audit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weight"]
    .to_dict()
)

plm_weight_unit_lookup = (
    plm_weight_audit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weight_unit"]
    .to_dict()
)

In [ ]:
sap_fiber_lookup = (
    sap_fiber_sets
    .set_index("material_id")["fiber_key"]
    .to_dict()
)

plm_fiber_lookup = (
    plm_fiber_sets
    .set_index("plm_code")["fiber_key"]
    .to_dict()
)

sap_coloring_lookup = (
    sap_coloring_model
    .set_index("material_id")["sap_coloring_key"]
    .to_dict()
)

plm_coloring_lookup = (
    plm_coloring_model
    .set_index("plm_code")["plm_coloring_key"]
    .to_dict()
)

sap_weave_lookup = (
    sap_weave_type
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weave_type"]
    .to_dict()
)

plm_weave_lookup = (
    plm_weave_type
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weave_type"]
    .to_dict()
)

In [ ]:
# Fiber similarity
pairwise_training_clean["fiber_similarity"] = [
    tuple_jaccard(
        sap_fiber_lookup.get(material_id),
        plm_fiber_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Fabric structure match
pairwise_training_clean["fabric_structure_match"] = [
    (
        float(
            sap_structure_lookup.get(material_id)
            == plm_structure_lookup.get(plm_code)
        )
        if (
            sap_structure_lookup.get(material_id) is not None
            and plm_structure_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Coloring match
pairwise_training_clean["coloring_match"] = [
    (
        float(
            sap_coloring_lookup.get(material_id)
            == plm_coloring_lookup.get(plm_code)
        )
        if (
            sap_coloring_lookup.get(material_id) is not None
            and plm_coloring_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Weave type match
pairwise_training_clean["weave_type_match"] = [
    (
        float(
            sap_weave_lookup.get(material_id)
            == plm_weave_lookup.get(plm_code)
        )
        if (
            sap_weave_lookup.get(material_id) is not None
            and plm_weave_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

In [ ]:
# Map weight values without changing row count
pairwise_training_clean["sap_weight_raw"] = (
    pairwise_training_clean["material_id"]
    .map(sap_weight_raw_lookup)
)

pairwise_training_clean["sap_weight_unit"] = (
    pairwise_training_clean["material_id"]
    .map(sap_weight_unit_lookup)
)

pairwise_training_clean["plm_weight"] = (
    pairwise_training_clean["candidate_plm_code"]
    .map(plm_weight_value_lookup)
)

pairwise_training_clean["plm_weight_unit"] = (
    pairwise_training_clean["candidate_plm_code"]
    .map(plm_weight_unit_lookup)
)

valid_weight = (
    pairwise_training_clean["sap_weight_raw"].notna()
    & pairwise_training_clean["plm_weight"].notna()
    & (pairwise_training_clean["sap_weight_raw"] > 0)
    & (pairwise_training_clean["plm_weight"] > 0)
)

same_scale_error = (
    abs(
        pairwise_training_clean["sap_weight_raw"]
        - pairwise_training_clean["plm_weight"]
    )
    / pairwise_training_clean["plm_weight"]
)

x1000_error = (
    abs(
        pairwise_training_clean["sap_weight_raw"] / 1000
        - pairwise_training_clean["plm_weight"]
    )
    / pairwise_training_clean["plm_weight"]
)

weight_error = np.minimum(
    same_scale_error,
    x1000_error
)

unit_mismatch = (
    pairwise_training_clean["sap_weight_unit"].notna()
    & pairwise_training_clean["plm_weight_unit"].notna()
    & (
        pairwise_training_clean["sap_weight_unit"]
        != pairwise_training_clean["plm_weight_unit"]
    )
)

weight_error[
    ~valid_weight | unit_mismatch
] = np.nan

pairwise_training_clean["weight_similarity"] = (
    1 / (1 + weight_error)
)

In [ ]:
assert not pairwise_training_clean.duplicated(
    pair_key
).any()

print(
    "Clean pairwise rows:",
    len(pairwise_training_clean)
)

print(
    "Clean unique pairs:",
    pairwise_training_clean[
        pair_key
    ].drop_duplicates().shape[0]
)

In [ ]:
# Final integrity checks
print(
    "Training materials:",
    pairwise_training_clean["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_training_clean["is_match"].sum()
)

print(
    "Materials without a positive:",
    (
        pairwise_training_clean
        .groupby("material_id")["is_match"]
        .sum()
        .eq(0)
        .sum()
    )
)

assert not pairwise_training_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

In [ ]:
# Build yarn-count lookup dictionaries
sap_knit_yarn_count_lookup = (
    sap_yarn_count_knit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_yarn_count"]
    .to_dict()
)

plm_knit_yarn_count_lookup = (
    plm_yarn_count_knit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_yarn_count"]
    .to_dict()
)

sap_weft_yarn_count_lookup = (
    sap_weft_yarn_count
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weft_yarn_count"]
    .to_dict()
)

plm_weft_yarn_count_lookup = (
    plm_weft_yarn_count
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weft_yarn_count"]
    .to_dict()
)

In [ ]:
# Build normalized knitting yarn-type lookups
sap_knit_yarn_type_lookup = (
    sap_yarn_type_sets_normalized
    .set_index("material_id")["sap_yarn_type_set"]
    .to_dict()
)

plm_knit_yarn_type_lookup = (
    plm_yarn_type_sets_normalized
    .set_index("plm_code")["plm_yarn_type_set"]
    .to_dict()
)


def safe_set_jaccard(left, right):
    if not isinstance(left, set) or not isinstance(right, set):
        return np.nan

    if not left or not right:
        return np.nan

    return len(left & right) / len(left | right)

In [ ]:
pairwise_training_clean[
    "knit_yarn_count_match"
] = np.nan

pairwise_training_clean[
    "knit_yarn_type_similarity"
] = np.nan

pairwise_training_clean[
    "weft_yarn_count_match"
] = np.nan

In [ ]:
orme_mask = (
    pairwise_training_clean["material_group_code"]
    == "1020001"
)

pairwise_training_clean.loc[
    orme_mask,
    "knit_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_knit_yarn_count_lookup.get(material_id),
        plm_knit_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            orme_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            orme_mask,
            "candidate_plm_code"
        ]
    )
]

pairwise_training_clean.loc[
    orme_mask,
    "knit_yarn_type_similarity"
] = [
    safe_set_jaccard(
        sap_knit_yarn_type_lookup.get(material_id),
        plm_knit_yarn_type_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            orme_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            orme_mask,
            "candidate_plm_code"
        ]
    )
]

In [ ]:
dokuma_mask = (
    pairwise_training_clean["material_group_code"]
    == "1020002"
)

pairwise_training_clean.loc[
    dokuma_mask,
    "weft_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_weft_yarn_count_lookup.get(material_id),
        plm_weft_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            dokuma_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            dokuma_mask,
            "candidate_plm_code"
        ]
    )
]

In [ ]:
assert not pairwise_training_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

In [ ]:
# Heuristic score used only for negative sampling
pairwise_training_clean["hardness_score"] = (
    pairwise_training_clean[
        "fiber_similarity"
    ].fillna(0)
    + pairwise_training_clean[
        "weight_similarity"
    ].fillna(0)
    + pairwise_training_clean[
        "fabric_structure_match"
    ].fillna(0)
    + 0.5
    * pairwise_training_clean[
        "coloring_match"
    ].fillna(0)
    + 0.25
    * pairwise_training_clean[
        "retrieval_similarity"
    ].fillna(0)
)

positive_pairs_clean = (
    pairwise_training_clean[
        pairwise_training_clean["is_match"] == 1
    ]
    .copy()
)

negative_pairs_clean = (
    pairwise_training_clean[
        pairwise_training_clean["is_match"] == 0
    ]
    .copy()
)

In [ ]:
hard_negatives_clean = (
    negative_pairs_clean
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(20)
)

In [ ]:
rng = np.random.default_rng(42)

negative_pairs_clean["random_score"] = (
    rng.random(len(negative_pairs_clean))
)

random_negatives_clean = (
    negative_pairs_clean
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

In [ ]:
pairwise_sample_clean = (
    pd.concat(
        [
            positive_pairs_clean,
            hard_negatives_clean,
            random_negatives_clean
        ],
        ignore_index=True
    )
    .drop_duplicates(
        [
            "material_id",
            "candidate_plm_code"
        ]
    )
)

print(
    "Sample rows:",
    len(pairwise_sample_clean)
)

print(
    "Materials:",
    pairwise_sample_clean[
        "material_id"
    ].nunique()
)

print(
    "Positive pairs:",
    pairwise_sample_clean[
        "is_match"
    ].sum()
)

In [ ]:
common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

baseline_group_features = {
    "1020001": common_features,
    "1020002": common_features + [
        "weave_type_match"
    ],
    "1030004": common_features + [
        "weave_type_match"
    ]
}

yarn_group_features = {
    "1020001": common_features + [
        "knit_yarn_count_match",
        "knit_yarn_type_similarity"
    ],
    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],
    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def evaluate_group_models(
    feature_config,
    sample_df,
    full_df,
    train_ids,
    test_ids
):
    results = []

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        train_group = sample_df[
            sample_df["material_id"].isin(train_ids)
            & (
                sample_df["material_group_code"]
                == group_code
            )
        ].copy()

        test_group = full_df[
            full_df["material_id"].isin(test_ids)
            & (
                full_df["material_group_code"]
                == group_code
            )
        ].copy()

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ])

        model.fit(
            train_group[features],
            train_group["is_match"]
        )

        test_group["score"] = (
            model.predict_proba(
                test_group[features]
            )[:, 1]
        )

        test_group["rank"] = (
            test_group
            .groupby("material_id")["score"]
            .rank(
                ascending=False,
                method="first"
            )
        )

        true_rows = (
            test_group[
                test_group["is_match"] == 1
            ]
        )

        results.append({
            "material_group":
                group_name_mapping[group_code],
            "test_materials":
                len(true_rows),
            "top_1":
                (true_rows["rank"] <= 1).mean(),
            "top_3":
                (true_rows["rank"] <= 3).mean(),
            "top_5":
                (true_rows["rank"] <= 5).mean(),
            "top_10":
                (true_rows["rank"] <= 10).mean(),
            "mrr":
                (1 / true_rows["rank"]).mean()
        })

    return pd.DataFrame(results)


In [ ]:
baseline_results = evaluate_group_models(
    baseline_group_features,
    pairwise_sample_clean,
    pairwise_training_clean,
    train_ids,
    test_ids
)

display(baseline_results)

In [ ]:
yarn_results = evaluate_group_models(
    yarn_group_features,
    pairwise_sample_clean,
    pairwise_training_clean,
    train_ids,
    test_ids
)

display(yarn_results)

In [ ]:
comparison = (
    baseline_results
    .merge(
        yarn_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_baseline",
            "_with_yarn"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    comparison[
        f"{metric}_delta"
    ] = (
        comparison[
            f"{metric}_with_yarn"
        ]
        - comparison[
            f"{metric}_baseline"
        ]
    )

display(comparison)

In [ ]:
# ORME feature ablation
orme_feature_configs = {
    "baseline": common_features,

    "yarn_count_only": common_features + [
        "knit_yarn_count_match"
    ],

    "yarn_type_only": common_features + [
        "knit_yarn_type_similarity"
    ],

    "both_yarn_features": common_features + [
        "knit_yarn_count_match",
        "knit_yarn_type_similarity"
    ]
}

In [ ]:
# Evaluate each ORME feature configuration
orme_ablation_results = []

for experiment_name, features in orme_feature_configs.items():

    result = evaluate_group_models(
        {"1020001": features},
        pairwise_sample_clean,
        pairwise_training_clean,
        train_ids,
        test_ids
    )

    result["experiment"] = experiment_name

    orme_ablation_results.append(result)

orme_ablation_results = pd.concat(
    orme_ablation_results,
    ignore_index=True
)

display(
    orme_ablation_results[
        [
            "experiment",
            "test_materials",
            "top_1",
            "top_3",
            "top_5",
            "top_10",
            "mrr"
        ]
    ]
)

In [ ]:
# Compare every experiment with the ORME baseline
baseline_row = (
    orme_ablation_results[
        orme_ablation_results["experiment"] == "baseline"
    ]
    .iloc[0]
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    orme_ablation_results[
        f"{metric}_delta"
    ] = (
        orme_ablation_results[metric]
        - baseline_row[metric]
    )

display(
    orme_ablation_results[
        [
            "experiment",
            "top_1_delta",
            "top_3_delta",
            "top_5_delta",
            "top_10_delta",
            "mrr_delta"
        ]
    ]
)

In [ ]:
# Final group-specific feature configuration
final_group_features = {
    "1020001": common_features + [
        "knit_yarn_count_match"
    ],

    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
def evaluate_random_forest_models(
    feature_config,
    sample_df,
    full_df,
    train_ids,
    test_ids
):
    results = []
    models = {}

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        train_group = sample_df[
            sample_df["material_id"].isin(train_ids)
            & (
                sample_df["material_group_code"]
                == group_code
            )
        ].copy()

        test_group = full_df[
            full_df["material_id"].isin(test_ids)
            & (
                full_df["material_group_code"]
                == group_code
            )
        ].copy()

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=12,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=42
                )
            )
        ])

        model.fit(
            train_group[features],
            train_group["is_match"]
        )

        test_group["score"] = (
            model.predict_proba(
                test_group[features]
            )[:, 1]
        )

        test_group["rank"] = (
            test_group
            .groupby("material_id")["score"]
            .rank(
                ascending=False,
                method="first"
            )
        )

        true_rows = (
            test_group[
                test_group["is_match"] == 1
            ]
            .copy()
        )

        results.append({
            "material_group":
                group_name_mapping[group_code],
            "test_materials":
                len(true_rows),
            "top_1":
                (true_rows["rank"] <= 1).mean(),
            "top_3":
                (true_rows["rank"] <= 3).mean(),
            "top_5":
                (true_rows["rank"] <= 5).mean(),
            "top_10":
                (true_rows["rank"] <= 10).mean(),
            "mrr":
                (1 / true_rows["rank"]).mean()
        })

        models[group_code] = model

    return pd.DataFrame(results), models


In [ ]:
rf_results, rf_models = (
    evaluate_random_forest_models(
        final_group_features,
        pairwise_sample_clean,
        pairwise_training_clean,
        train_ids,
        test_ids
    )
)

display(rf_results)

In [ ]:
logistic_final_results = evaluate_group_models(
    final_group_features,
    pairwise_sample_clean,
    pairwise_training_clean,
    train_ids,
    test_ids
)

display(logistic_final_results)

In [ ]:
model_comparison = (
    logistic_final_results
    .merge(
        rf_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_logistic",
            "_random_forest"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    model_comparison[
        f"{metric}_delta"
    ] = (
        model_comparison[
            f"{metric}_random_forest"
        ]
        - model_comparison[
            f"{metric}_logistic"
        ]
    )

display(model_comparison)

In [ ]:
# Current split has been used for model/feature selection,
# so it should be treated as validation data
validation_ids = test_ids
development_train_ids = train_ids

In [ ]:
from sklearn.model_selection import train_test_split

material_table = (
    pairwise_training_clean[
        ["material_id", "material_group_code"]
    ]
    .drop_duplicates("material_id")
)

# First reserve 20% as untouched holdout test
development_materials, holdout_materials = train_test_split(
    material_table,
    test_size=0.20,
    random_state=123,
    stratify=material_table["material_group_code"]
)

# Split development portion into train and validation
train_materials, validation_materials = train_test_split(
    development_materials,
    test_size=0.20,
    random_state=123,
    stratify=development_materials["material_group_code"]
)

train_ids_final = set(train_materials["material_id"])
validation_ids_final = set(validation_materials["material_id"])
holdout_ids = set(holdout_materials["material_id"])

print("Train materials:", len(train_ids_final))
print("Validation materials:", len(validation_ids_final))
print("Holdout materials:", len(holdout_ids))

In [ ]:
for name, df in {
    "Train": train_materials,
    "Validation": validation_materials,
    "Holdout": holdout_materials
}.items():
    print(f"\n{name}")
    print(
        df["material_group_code"]
        .value_counts()
        .sort_index()
    )

In [ ]:
def build_training_sample(
    full_df,
    train_ids,
    hard_negatives_per_material=20,
    random_negatives_per_material=20,
    random_state=42
):
    train_pool = full_df[
        full_df["material_id"].isin(train_ids)
    ].copy()

    positive_pairs = train_pool[
        train_pool["is_match"] == 1
    ].copy()

    negative_pairs = train_pool[
        train_pool["is_match"] == 0
    ].copy()

    # Heuristic used only to identify difficult negatives
    negative_pairs["hardness_score"] = (
        negative_pairs["fiber_similarity"].fillna(0)
        + negative_pairs["weight_similarity"].fillna(0)
        + negative_pairs["fabric_structure_match"].fillna(0)
        + 0.5 * negative_pairs["coloring_match"].fillna(0)
        + 0.25 * negative_pairs["retrieval_similarity"].fillna(0)
    )

    hard_negatives = (
        negative_pairs
        .sort_values(
            ["material_id", "hardness_score"],
            ascending=[True, False]
        )
        .groupby("material_id")
        .head(hard_negatives_per_material)
    )

    rng = np.random.default_rng(random_state)

    negative_pairs["random_score"] = (
        rng.random(len(negative_pairs))
    )

    random_negatives = (
        negative_pairs
        .sort_values(
            ["material_id", "random_score"]
        )
        .groupby("material_id")
        .head(random_negatives_per_material)
    )

    training_sample = (
        pd.concat(
            [
                positive_pairs,
                hard_negatives,
                random_negatives
            ],
            ignore_index=True
        )
        .drop_duplicates(
            ["material_id", "candidate_plm_code"]
        )
    )

    return training_sample

In [ ]:
training_sample_final = build_training_sample(
    pairwise_training_clean,
    train_ids_final
)

print(
    "Training sample rows:",
    len(training_sample_final)
)

print(
    "Training materials:",
    training_sample_final["material_id"].nunique()
)

print(
    "Positive pairs:",
    training_sample_final["is_match"].sum()
)

print(
    "Positive rate:",
    f"{training_sample_final['is_match'].mean():.2%}"
)

In [ ]:
common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

final_group_features = {
    # ORME
    "1020001": common_features + [
        "knit_yarn_count_match"
    ],

    # DOKUMA
    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    # DENIM
    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [ ]:
logistic_validation_results = evaluate_group_models(
    final_group_features,
    training_sample_final,
    pairwise_training_clean,
    train_ids_final,
    validation_ids_final
)

display(logistic_validation_results)

In [ ]:
rf_validation_results, rf_validation_models = (
    evaluate_random_forest_models(
        final_group_features,
        training_sample_final,
        pairwise_training_clean,
        train_ids_final,
        validation_ids_final
    )
)

display(rf_validation_results)

In [ ]:
validation_model_comparison = (
    logistic_validation_results
    .merge(
        rf_validation_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_logistic",
            "_random_forest"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    validation_model_comparison[
        f"{metric}_delta"
    ] = (
        validation_model_comparison[
            f"{metric}_random_forest"
        ]
        - validation_model_comparison[
            f"{metric}_logistic"
        ]
    )

display(validation_model_comparison)

In [ ]:
from sklearn.model_selection import KFold

development_ids = set(
    development_materials["material_id"]
)

development_pairwise = (
    pairwise_training_clean[
        pairwise_training_clean["material_id"]
        .isin(development_ids)
    ]
    .copy()
)

In [ ]:
def cross_validate_rerankers(
    full_df,
    feature_config,
    n_splits=5,
    random_state=42
):
    cv_results = []

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        material_ids = (
            full_df.loc[
                full_df["material_group_code"] == group_code,
                "material_id"
            ]
            .drop_duplicates()
            .to_numpy()
        )

        kfold = KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state
        )

        for fold, (train_idx, val_idx) in enumerate(
            kfold.split(material_ids),
            start=1
        ):
            fold_train_ids = set(
                material_ids[train_idx]
            )

            fold_val_ids = set(
                material_ids[val_idx]
            )

            # Negative sampling only from fold training materials
            training_sample = build_training_sample(
                full_df,
                fold_train_ids,
                random_state=42 + fold
            )

            validation_full = (
                full_df[
                    full_df["material_id"]
                    .isin(fold_val_ids)
                ]
                .copy()
            )

            models = {
                "logistic": Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    ),
                    (
                        "model",
                        LogisticRegression(
                            max_iter=2000,
                            class_weight="balanced",
                            random_state=42
                        )
                    )
                ]),

                "random_forest":
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
                        ),
                        (
                            "model",
                            RandomForestClassifier(
                                n_estimators=300,
                                max_depth=12,
                                min_samples_leaf=2,
                                class_weight=
                                    "balanced_subsample",
                                n_jobs=-1,
                                random_state=42
                            )
                        )
                    ])
            }

            for model_name, model in models.items():

                model.fit(
                    training_sample[features],
                    training_sample["is_match"]
                )

                scored = validation_full.copy()

                scored["score"] = (
                    model.predict_proba(
                        scored[features]
                    )[:, 1]
                )

                scored["rank"] = (
                    scored
                    .groupby("material_id")["score"]
                    .rank(
                        ascending=False,
                        method="first"
                    )
                )

                true_rows = (
                    scored[
                        scored["is_match"] == 1
                    ]
                )

                cv_results.append({
                    "material_group":
                        group_name_mapping[group_code],
                    "fold": fold,
                    "model": model_name,
                    "test_materials":
                        len(true_rows),

                    "top_1":
                        (
                            true_rows["rank"] <= 1
                        ).mean(),

                    "top_3":
                        (
                            true_rows["rank"] <= 3
                        ).mean(),

                    "top_5":
                        (
                            true_rows["rank"] <= 5
                        ).mean(),

                    "top_10":
                        (
                            true_rows["rank"] <= 10
                        ).mean(),

                    "mrr":
                        (
                            1 / true_rows["rank"]
                        ).mean()
                })

    return pd.DataFrame(cv_results)


In [ ]:
cv_results = cross_validate_rerankers(
    development_pairwise,
    final_group_features,
    n_splits=5,
    random_state=42
)

In [ ]:
cv_summary = (
    cv_results
    .groupby(
        ["material_group", "model"]
    )
    .agg(
        folds=("fold", "count"),
        top_1_mean=("top_1", "mean"),
        top_1_std=("top_1", "std"),
        top_3_mean=("top_3", "mean"),
        top_5_mean=("top_5", "mean"),
        top_10_mean=("top_10", "mean"),
        mrr_mean=("mrr", "mean"),
        mrr_std=("mrr", "std")
    )
    .reset_index()
)

display(cv_summary)

In [ ]:
cv_comparison = (
    cv_summary
    .pivot(
        index="material_group",
        columns="model",
        values=[
            "top_1_mean",
            "top_5_mean",
            "mrr_mean"
        ]
    )
)

display(cv_comparison)

In [ ]:
# Combine train and validation for final development training
development_ids_final = (
    train_ids_final
    | validation_ids_final
)

final_training_sample = build_training_sample(
    pairwise_training_clean,
    development_ids_final,
    hard_negatives_per_material=20,
    random_negatives_per_material=20,
    random_state=42
)

print(
    "Development training materials:",
    final_training_sample["material_id"].nunique()
)

print(
    "Training sample rows:",
    len(final_training_sample)
)

print(
    "Positive pairs:",
    final_training_sample["is_match"].sum()
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

final_rf_models = {}
holdout_results = []
holdout_scored_parts = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, features in final_group_features.items():

    train_group = final_training_sample[
        final_training_sample["material_group_code"]
        == group_code
    ].copy()

    holdout_group = pairwise_training_clean[
        pairwise_training_clean["material_id"].isin(holdout_ids)
        & (
            pairwise_training_clean["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    holdout_group["match_probability"] = (
        model.predict_proba(
            holdout_group[features]
        )[:, 1]
    )

    holdout_group["predicted_rank"] = (
        holdout_group
        .groupby("material_id")["match_probability"]
        .rank(
            ascending=False,
            method="first"
        )
    )

    true_rows = holdout_group[
        holdout_group["is_match"] == 1
    ].copy()

    holdout_results.append({
        "material_group":
            group_name_mapping[group_code],

        "holdout_materials":
            len(true_rows),

        "top_1":
            (true_rows["predicted_rank"] <= 1).mean(),

        "top_3":
            (true_rows["predicted_rank"] <= 3).mean(),

        "top_5":
            (true_rows["predicted_rank"] <= 5).mean(),

        "top_10":
            (true_rows["predicted_rank"] <= 10).mean(),

        "mrr":
            (
                1 / true_rows["predicted_rank"]
            ).mean()
    })

    final_rf_models[group_code] = model
    holdout_scored_parts.append(holdout_group)

holdout_results = pd.DataFrame(
    holdout_results
)

holdout_scored = pd.concat(
    holdout_scored_parts,
    ignore_index=True
)

display(holdout_results)


In [ ]:
total_holdout = (
    holdout_results["holdout_materials"].sum()
)

overall_holdout = {}

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    overall_holdout[metric] = (
        (
            holdout_results[metric]
            * holdout_results["holdout_materials"]
        ).sum()
        / total_holdout
    )

print("Reranker holdout performance")

for metric, value in overall_holdout.items():
    print(
        f"{metric}: {value:.4f}"
    )

In [ ]:
holdout_true_ranks = (
    holdout_scored[
        holdout_scored["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "true_plm_code",
            "predicted_rank",
            "match_probability"
        ]
    ]
    .copy()
)

display(
    holdout_true_ranks[
        "predicted_rank"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
# Score validation candidates using the selected Random Forest models
validation_scored_parts = []

for group_code, features in final_group_features.items():

    train_group = training_sample_final[
        training_sample_final["material_group_code"]
        == group_code
    ].copy()

    validation_group = pairwise_training_clean[
        pairwise_training_clean["material_id"]
        .isin(validation_ids_final)
        & (
            pairwise_training_clean["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    validation_group["score"] = (
        model.predict_proba(
            validation_group[features]
        )[:, 1]
    )

    validation_scored_parts.append(
        validation_group
    )

validation_scored = pd.concat(
    validation_scored_parts,
    ignore_index=True
)


In [ ]:
# Rank validation candidates
validation_scored["rank"] = (
    validation_scored
    .groupby("material_id")["score"]
    .rank(
        ascending=False,
        method="first"
    )
)

top_two = (
    validation_scored[
        validation_scored["rank"] <= 2
    ]
    .sort_values(
        ["material_id", "rank"]
    )
)

confidence_table = (
    top_two
    .pivot(
        index="material_id",
        columns="rank",
        values="score"
    )
    .rename(
        columns={
            1.0: "top1_score",
            2.0: "top2_score"
        }
    )
    .reset_index()
)

confidence_table["score_margin"] = (
    confidence_table["top1_score"]
    - confidence_table["top2_score"]
)

In [ ]:
top1_predictions = (
    validation_scored[
        validation_scored["rank"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "true_plm_code",
            "score"
        ]
    ]
    .copy()
)

top1_predictions["is_top1_correct"] = (
    top1_predictions["candidate_plm_code"]
    == top1_predictions["true_plm_code"]
)

confidence_audit = (
    top1_predictions
    .merge(
        confidence_table,
        on="material_id",
        how="left"
    )
)

In [ ]:
confidence_audit["margin_band"] = pd.cut(
    confidence_audit["score_margin"],
    bins=[
        -np.inf,
        0.05,
        0.10,
        0.20,
        0.30,
        np.inf
    ],
    labels=[
        "<=0.05",
        "0.05-0.10",
        "0.10-0.20",
        "0.20-0.30",
        ">0.30"
    ]
)

display(
    confidence_audit
    .groupby(
        "margin_band",
        observed=True
    )
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_top1_score=("top1_score", "mean"),
        mean_margin=("score_margin", "mean")
    )
)

In [ ]:
display(
    confidence_audit
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_top1_score=("top1_score", "mean"),
        mean_margin=("score_margin", "mean")
    )
)

In [ ]:
# Evaluate cumulative confidence thresholds
margin_thresholds = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

threshold_results = []

total_materials = len(confidence_audit)

for threshold in margin_thresholds:

    selected = confidence_audit[
        confidence_audit["score_margin"] >= threshold
    ]

    if len(selected) == 0:
        continue

    threshold_results.append({
        "margin_threshold": threshold,
        "selected_materials": len(selected),
        "coverage": len(selected) / total_materials,
        "precision": selected["is_top1_correct"].mean()
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

In [ ]:
# Evaluate confidence thresholds by material group
group_threshold_results = []

for group_code, group in confidence_audit.groupby(
    "material_group_code"
):

    group_total = len(group)

    for threshold in margin_thresholds:

        selected = group[
            group["score_margin"] >= threshold
        ]

        if len(selected) == 0:
            continue

        group_threshold_results.append({
            "material_group_code": group_code,
            "margin_threshold": threshold,
            "selected_materials": len(selected),
            "coverage": len(selected) / group_total,
            "precision": selected[
                "is_top1_correct"
            ].mean()
        })

group_threshold_results = pd.DataFrame(
    group_threshold_results
)

display(group_threshold_results)

In [ ]:
# Find the highest-coverage threshold
# satisfying a target precision
def find_best_threshold(
    results,
    target_precision
):
    eligible = results[
        results["precision"] >= target_precision
    ].copy()

    if eligible.empty:
        return None

    return (
        eligible
        .sort_values(
            [
                "coverage",
                "margin_threshold"
            ],
            ascending=[False, True]
        )
        .iloc[0]
    )


for target in [0.90, 0.95]:
    result = find_best_threshold(
        threshold_results,
        target
    )

    print(
        f"\nTarget precision: {target:.0%}"
    )

    if result is None:
        print("No threshold reached the target.")
    else:
        print(
            f"Margin threshold: "
            f"{result['margin_threshold']:.2f}"
        )
        print(
            f"Coverage: "
            f"{result['coverage']:.2%}"
        )
        print(
            f"Observed precision: "
            f"{result['precision']:.2%}"
        )

In [ ]:
# Confidence thresholds selected using validation data only
auto_margin_thresholds = {
    "1020001": 0.15,  # ORME
    "1020002": 0.30,  # DOKUMA
    "1030004": 0.40   # DENIM
}

review_margin_threshold = 0.05

In [ ]:
# Extract top two candidates from holdout results
holdout_top_two = (
    holdout_scored[
        holdout_scored["predicted_rank"] <= 2
    ]
    .sort_values(
        ["material_id", "predicted_rank"]
    )
)

holdout_confidence = (
    holdout_top_two
    .pivot(
        index="material_id",
        columns="predicted_rank",
        values="match_probability"
    )
    .rename(
        columns={
            1.0: "top1_score",
            2.0: "top2_score"
        }
    )
    .reset_index()
)

holdout_confidence["score_margin"] = (
    holdout_confidence["top1_score"]
    - holdout_confidence["top2_score"]
)

In [ ]:
# Add top-1 prediction and correctness
holdout_top1 = (
    holdout_scored[
        holdout_scored["predicted_rank"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "true_plm_code",
            "match_probability"
        ]
    ]
    .copy()
)

holdout_top1["is_top1_correct"] = (
    holdout_top1["candidate_plm_code"]
    == holdout_top1["true_plm_code"]
)

holdout_decisions = (
    holdout_top1
    .merge(
        holdout_confidence,
        on="material_id",
        how="left",
        validate="one_to_one"
    )
)

In [ ]:
# Assign operational decision based on fixed validation thresholds
def assign_confidence_decision(row):

    auto_threshold = auto_margin_thresholds[
        row["material_group_code"]
    ]

    if row["score_margin"] >= auto_threshold:
        return "AUTO"

    if row["score_margin"] >= review_margin_threshold:
        return "REVIEW"

    return "LOW_CONFIDENCE"


holdout_decisions["decision"] = (
    holdout_decisions.apply(
        assign_confidence_decision,
        axis=1
    )
)

In [ ]:
# Evaluate decision-layer performance
decision_summary = (
    holdout_decisions
    .groupby("decision")
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

decision_summary["coverage"] = (
    decision_summary["materials"]
    / len(holdout_decisions)
)

display(decision_summary)

In [ ]:
# Audit automatic recommendations by material group
auto_holdout_summary = (
    holdout_decisions[
        holdout_decisions["decision"] == "AUTO"
    ]
    .groupby("material_group_code")
    .agg(
        auto_materials=("material_id", "size"),
        auto_precision=("is_top1_correct", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

group_holdout_counts = (
    holdout_decisions
    .groupby("material_group_code")
    ["material_id"]
    .size()
    .rename("total_materials")
    .reset_index()
)

auto_holdout_summary = (
    auto_holdout_summary
    .merge(
        group_holdout_counts,
        on="material_group_code",
        how="left"
    )
)

auto_holdout_summary["auto_coverage"] = (
    auto_holdout_summary["auto_materials"]
    / auto_holdout_summary["total_materials"]
)

display(auto_holdout_summary)

In [ ]:
# Overall automatic-decision performance
auto_predictions = (
    holdout_decisions[
        holdout_decisions["decision"] == "AUTO"
    ]
)

print(
    "AUTO coverage:",
    f"{len(auto_predictions) / len(holdout_decisions):.2%}"
)

print(
    "AUTO precision:",
    f"{auto_predictions['is_top1_correct'].mean():.2%}"
)

In [ ]:
# Add true-PLM presence within Top-K to the holdout decision table
holdout_rank_lookup = (
    holdout_scored[
        holdout_scored["is_match"] == 1
    ][
        [
            "material_id",
            "predicted_rank"
        ]
    ]
    .rename(
        columns={
            "predicted_rank": "true_plm_rank"
        }
    )
)

holdout_decisions = (
    holdout_decisions
    .merge(
        holdout_rank_lookup,
        on="material_id",
        how="left",
        validate="one_to_one"
    )
)

holdout_decisions["true_in_top3"] = (
    holdout_decisions["true_plm_rank"] <= 3
)

holdout_decisions["true_in_top5"] = (
    holdout_decisions["true_plm_rank"] <= 5
)

In [ ]:
review_holdout = (
    holdout_decisions[
        holdout_decisions["decision"] == "REVIEW"
    ]
)

print(
    "REVIEW coverage:",
    f"{len(review_holdout) / len(holdout_decisions):.2%}"
)

print(
    "REVIEW Top-1 accuracy:",
    f"{review_holdout['is_top1_correct'].mean():.2%}"
)

print(
    "REVIEW Top-3 success:",
    f"{review_holdout['true_in_top3'].mean():.2%}"
)

print(
    "REVIEW Top-5 success:",
    f"{review_holdout['true_in_top5'].mean():.2%}"
)

In [ ]:
review_group_summary = (
    review_holdout
    .groupby("material_group_code")
    .agg(
        review_materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        top3_success=("true_in_top3", "mean"),
        top5_success=("true_in_top5", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

display(review_group_summary)

In [ ]:
# Operational success under the fixed decision policy
holdout_decisions["workflow_success"] = np.select(
    [
        holdout_decisions["decision"] == "AUTO",
        holdout_decisions["decision"] == "REVIEW"
    ],
    [
        holdout_decisions["is_top1_correct"],
        holdout_decisions["true_in_top3"]
    ],
    default=False
)

actionable = (
    holdout_decisions["decision"]
    .isin(["AUTO", "REVIEW"])
)

print(
    "AUTO + REVIEW coverage:",
    f"{actionable.mean():.2%}"
)

print(
    "Success among actionable materials:",
    f"{holdout_decisions.loc[actionable, 'workflow_success'].mean():.2%}"
)

In [ ]:
# Prepare known mappings without SAP attributes
known_no_attribute_mappings = (
    known_retrieval_pairs[
        ~known_retrieval_pairs["material_id"]
        .astype("string")
        .isin(materials_with_attributes)
    ]
    .copy()
)

print(
    "Known mappings without SAP attributes:",
    len(known_no_attribute_mappings)
)

In [ ]:
# Add the SAP-side material group used for blocking
known_no_attribute_audit = (
    known_no_attribute_mappings
    .merge(
        material_group_lookup.rename(
            columns={
                "material_group_code":
                "sap_material_group_code"
            }
        ),
        on="material_id",
        how="left"
    )
)

known_no_attribute_audit["group_match"] = (
    known_no_attribute_audit["material_group_code"]
    == known_no_attribute_audit["sap_material_group_code"]
)

print(
    "Material-group mismatches:",
    (~known_no_attribute_audit["group_match"]).sum()
)

display(
    known_no_attribute_audit[
        ~known_no_attribute_audit["group_match"]
    ].head(20)
)

In [ ]:
from sklearn.model_selection import train_test_split

no_attribute_materials = (
    known_no_attribute_audit[
        [
            "material_id",
            "sap_material_group_code"
        ]
    ]
    .drop_duplicates("material_id")
    .rename(
        columns={
            "sap_material_group_code":
            "material_group_code"
        }
    )
)

no_attr_development, no_attr_holdout = train_test_split(
    no_attribute_materials,
    test_size=0.20,
    random_state=321,
    stratify=no_attribute_materials[
        "material_group_code"
    ]
)

no_attr_dev_ids = set(
    no_attr_development["material_id"]
)

no_attr_holdout_ids = set(
    no_attr_holdout["material_id"]
)

print(
    "Development materials:",
    len(no_attr_dev_ids)
)

print(
    "Untouched holdout materials:",
    len(no_attr_holdout_ids)
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


def generate_text_channel_candidates(
    sap_df,
    plm_df,
    material_group_code,
    plm_text_column,
    channel_name,
    top_k=50,
    analyzer="char_wb",
    ngram_range=(3, 5)
):
    sap_group = (
        sap_df[
            sap_df["material_group_code"]
            == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    plm_group = (
        plm_df[
            plm_df["material_group_code"]
            == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    sap_group = sap_group[
        sap_group["retrieval_text"].str.len() > 0
    ].reset_index(drop=True)

    plm_group = plm_group[
        plm_group[plm_text_column]
        .fillna("")
        .str.len() > 0
    ].reset_index(drop=True)

    if sap_group.empty or plm_group.empty:
        return pd.DataFrame()

    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        sublinear_tf=True,
        norm="l2"
    )

    plm_matrix = vectorizer.fit_transform(
        plm_group[plm_text_column]
    )

    sap_matrix = vectorizer.transform(
        sap_group["retrieval_text"]
    )

    n_neighbors = min(
        top_k,
        len(plm_group)
    )

    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute"
    )

    model.fit(plm_matrix)

    distances, indices = model.kneighbors(
        sap_matrix
    )

    records = []

    for sap_idx in range(len(sap_group)):
        for rank, (plm_idx, distance) in enumerate(
            zip(
                indices[sap_idx],
                distances[sap_idx]
            ),
            start=1
        ):
            records.append({
                "material_id":
                    sap_group.loc[
                        sap_idx,
                        "material_id"
                    ],
                "candidate_plm_code":
                    plm_group.loc[
                        plm_idx,
                        "plm_code"
                    ],
                "channel":
                    channel_name,
                "channel_rank":
                    rank,
                "channel_similarity":
                    1 - distance
            })

    return pd.DataFrame(records)

In [ ]:
sap_no_attr_dev = (
    sap_candidates_source[
        sap_candidates_source["material_id"]
        .isin(no_attr_dev_ids)
    ]
    .copy()
)

In [ ]:
text_channel_tables = []

channel_config = {
    "original_text": "retrieval_text",
    "structured_text": "structured_text",
    "multi_value_text": "multi_value_text",
    "enriched_text": "retrieval_text_enriched"
}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    for channel_name, text_column in (
        channel_config.items()
    ):
        result = generate_text_channel_candidates(
            sap_no_attr_dev,
            plm_candidates_enriched,
            material_group_code=group_code,
            plm_text_column=text_column,
            channel_name=channel_name,
            top_k=50
        )

        if not result.empty:
            text_channel_tables.append(result)

multi_channel_candidates = pd.concat(
    text_channel_tables,
    ignore_index=True
)

In [ ]:
multi_channel_union = (
    multi_channel_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Candidate pairs:",
    len(multi_channel_union)
)

print(
    "Average candidates per SAP:",
    len(multi_channel_union)
    / multi_channel_union["material_id"].nunique()
)

In [ ]:
no_attr_dev_truth = (
    known_no_attribute_audit[
        known_no_attribute_audit["material_id"]
        .isin(no_attr_dev_ids)
    ][
        [
            "material_id",
            "plm_code"
        ]
    ]
    .copy()
)

no_attr_dev_recall = (
    no_attr_dev_truth
    .merge(
        multi_channel_union,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_dev_recall["retrieved"] = (
    no_attr_dev_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Multi-channel development recall:",
    f"{no_attr_dev_recall['retrieved'].mean():.2%}"
)

In [ ]:
channel_recall_summary = []

for channel_name in (
    multi_channel_candidates["channel"]
    .unique()
):
    channel_candidates = (
        multi_channel_candidates[
            multi_channel_candidates["channel"]
            == channel_name
        ][
            [
                "material_id",
                "candidate_plm_code"
            ]
        ]
        .drop_duplicates()
    )

    evaluation = (
        no_attr_dev_truth
        .merge(
            channel_candidates,
            left_on=[
                "material_id",
                "plm_code"
            ],
            right_on=[
                "material_id",
                "candidate_plm_code"
            ],
            how="left"
        )
    )

    channel_recall_summary.append({
        "channel": channel_name,
        "recall": (
            evaluation[
                "candidate_plm_code"
            ].notna().mean()
        )
    })

display(
    pd.DataFrame(
        channel_recall_summary
    ).sort_values(
        "recall",
        ascending=False
    )
)

In [ ]:
# Evaluate multi-channel recall by material group
no_attr_dev_recall_grouped = (
    no_attr_dev_recall
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
)

display(
    no_attr_dev_recall_grouped
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

In [ ]:
# Build training data for description-to-structure prediction
structure_prediction_data = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_structure_candidates[
            [
                "material_id",
                "structure"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

structure_prediction_data = (
    structure_prediction_data[
        structure_prediction_data["retrieval_text"].str.len() > 0
    ]
    .dropna(
        subset=[
            "material_group_code",
            "structure"
        ]
    )
    .copy()
)

display(
    structure_prediction_data
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "nunique"),
        structure_classes=("structure", "nunique")
    )
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [ ]:
# Evaluate description-based fabric-structure prediction
structure_model_results = []
structure_models = {}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    group_data = (
        structure_prediction_data[
            structure_prediction_data["material_group_code"]
            == group_code
        ]
        .copy()
    )

    # Very rare labels are not suitable for stratified classification
    label_counts = group_data["structure"].value_counts()

    valid_labels = label_counts[
        label_counts >= 2
    ].index

    group_data = group_data[
        group_data["structure"].isin(valid_labels)
    ].copy()

    train_data, test_data = train_test_split(
        group_data,
        test_size=0.20,
        random_state=42,
        stratify=group_data["structure"]
    )

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        train_data["retrieval_text"],
        train_data["structure"]
    )

    predicted = model.predict(
        test_data["retrieval_text"]
    )

    probabilities = model.predict_proba(
        test_data["retrieval_text"]
    )

    classes = model.named_steps[
        "model"
    ].classes_

    structure_model_results.append({
        "material_group_code": group_code,
        "test_materials": len(test_data),
        "classes": len(classes),
        "top_1_accuracy":
            accuracy_score(
                test_data["structure"],
                predicted
            ),
        "top_3_accuracy":
            top_k_accuracy_score(
                test_data["structure"],
                probabilities,
                k=min(3, len(classes)),
                labels=classes
            )
    })

    structure_models[group_code] = model

display(
    pd.DataFrame(structure_model_results)
)

In [ ]:
# Retrain structure classifiers using all available labeled SAP materials
final_structure_models = {}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    group_data = (
        structure_prediction_data[
            structure_prediction_data["material_group_code"]
            == group_code
        ]
        .copy()
    )

    # Keep labels with at least two observations
    label_counts = group_data["structure"].value_counts()

    valid_labels = label_counts[
        label_counts >= 2
    ].index

    group_data = group_data[
        group_data["structure"].isin(valid_labels)
    ].copy()

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        group_data["retrieval_text"],
        group_data["structure"]
    )

    final_structure_models[group_code] = model

In [ ]:
# Predict Top-3 fabric structures for SAP materials without attributes
predicted_structure_records = []

for group_code, model in final_structure_models.items():

    sap_group = (
        sap_no_attr_dev[
            sap_no_attr_dev["material_group_code"]
            == group_code
        ]
        .copy()
    )

    if sap_group.empty:
        continue

    probabilities = model.predict_proba(
        sap_group["retrieval_text"]
    )

    classes = model.named_steps[
        "model"
    ].classes_

    top_k = min(3, len(classes))

    top_indices = np.argsort(
        probabilities,
        axis=1
    )[:, -top_k:][:, ::-1]

    for row_idx, material_id in enumerate(
        sap_group["material_id"]
    ):
        for rank, class_idx in enumerate(
            top_indices[row_idx],
            start=1
        ):
            predicted_structure_records.append({
                "material_id": material_id,
                "material_group_code": group_code,
                "predicted_structure":
                    classes[class_idx],
                "structure_rank": rank,
                "structure_probability":
                    probabilities[row_idx, class_idx]
            })

predicted_structures = pd.DataFrame(
    predicted_structure_records
)

display(predicted_structures.head(10))

In [ ]:
# Prepare PLM structure-aware retrieval table
plm_structure_retrieval = (
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "structured_text"
        ]
    ]
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "structure"
            ]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_structure_retrieval = (
    plm_structure_retrieval[
        plm_structure_retrieval[
            "structured_text"
        ].str.len() > 0
    ]
    .copy()
)

In [ ]:
def generate_predicted_structure_candidates(
    sap_df,
    predicted_structures,
    plm_df,
    top_k_per_structure=30
):
    candidate_records = []

    for group_code in predicted_structures[
        "material_group_code"
    ].unique():

        sap_group = (
            sap_df[
                sap_df["material_group_code"]
                == group_code
            ]
            .set_index("material_id")
        )

        predictions_group = (
            predicted_structures[
                predicted_structures[
                    "material_group_code"
                ] == group_code
            ]
        )

        for structure_value, structure_predictions in (
            predictions_group.groupby(
                "predicted_structure"
            )
        ):

            plm_pool = (
                plm_df[
                    (plm_df["material_group_code"]
                     == group_code)
                    & (
                        plm_df["structure"]
                        == structure_value
                    )
                ]
                .reset_index(drop=True)
            )

            if plm_pool.empty:
                continue

            vectorizer = TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                norm="l2"
            )

            plm_matrix = vectorizer.fit_transform(
                plm_pool["structured_text"]
            )

            n_neighbors = min(
                top_k_per_structure,
                len(plm_pool)
            )

            nn_model = NearestNeighbors(
                n_neighbors=n_neighbors,
                metric="cosine",
                algorithm="brute"
            )

            nn_model.fit(plm_matrix)

            for pred in structure_predictions.itertuples(
                index=False
            ):

                if pred.material_id not in sap_group.index:
                    continue

                sap_text = sap_group.loc[
                    pred.material_id,
                    "retrieval_text"
                ]

                sap_vector = vectorizer.transform(
                    [sap_text]
                )

                distances, indices = (
                    nn_model.kneighbors(
                        sap_vector
                    )
                )

                for retrieval_rank, (
                    plm_idx,
                    distance
                ) in enumerate(
                    zip(
                        indices[0],
                        distances[0]
                    ),
                    start=1
                ):
                    candidate_records.append({
                        "material_id":
                            pred.material_id,
                        "candidate_plm_code":
                            plm_pool.loc[
                                plm_idx,
                                "plm_code"
                            ],
                        "predicted_structure":
                            structure_value,
                        "structure_rank":
                            pred.structure_rank,
                        "structure_probability":
                            pred.structure_probability,
                        "structure_retrieval_rank":
                            retrieval_rank,
                        "structure_retrieval_similarity":
                            1 - distance
                    })

    return pd.DataFrame(candidate_records)

In [ ]:
pseudo_structure_candidates = (
    generate_predicted_structure_candidates(
        sap_no_attr_dev,
        predicted_structures,
        plm_structure_retrieval,
        top_k_per_structure=30
    )
)

pseudo_structure_pairs = (
    pseudo_structure_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Pseudo-structure candidate pairs:",
    len(pseudo_structure_pairs)
)

print(
    "Average candidates per SAP:",
    len(pseudo_structure_pairs)
    / pseudo_structure_pairs[
        "material_id"
    ].nunique()
)

In [ ]:
pseudo_structure_recall = (
    no_attr_dev_truth
    .merge(
        pseudo_structure_pairs,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

pseudo_structure_recall["retrieved"] = (
    pseudo_structure_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Pseudo-structure recall:",
    f"{pseudo_structure_recall['retrieved'].mean():.2%}"
)

In [ ]:
# Union text channels with predicted-structure retrieval
no_attr_combined_candidates = (
    pd.concat(
        [
            multi_channel_union,
            pseudo_structure_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

no_attr_combined_recall = (
    no_attr_dev_truth
    .merge(
        no_attr_combined_candidates,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_combined_recall["retrieved"] = (
    no_attr_combined_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Combined no-attribute recall:",
    f"{no_attr_combined_recall['retrieved'].mean():.2%}"
)

In [ ]:
no_attr_combined_grouped = (
    no_attr_combined_recall
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
)

display(
    no_attr_combined_grouped
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

In [ ]:
# Convert yarn count into a retrieval-friendly canonical label
def canonicalize_yarn_count(value):
    parsed = parse_yarn_count(value)

    if pd.isna(parsed["count"]):
        return pd.NA

    count = parsed["count"]

    if float(count).is_integer():
        count_text = str(int(count))
    else:
        count_text = str(count)

    ply = parsed["ply"]

    # Missing ply and /1 are treated as equivalent for candidate retrieval
    if pd.isna(ply) or ply == 1:
        return count_text

    if float(ply).is_integer():
        ply_text = str(int(ply))
    else:
        ply_text = str(ply)

    return f"{count_text}/{ply_text}"

In [ ]:
# Build description-to-yarn-count training data for knitted fabrics
yarn_count_prediction_data = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_yarn_count_knit[
            [
                "material_id",
                "sap_yarn_count"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

yarn_count_prediction_data = (
    yarn_count_prediction_data[
        yarn_count_prediction_data["material_group_code"]
        == "1020001"
    ]
    .copy()
)

yarn_count_prediction_data["yarn_count_class"] = (
    yarn_count_prediction_data["sap_yarn_count"]
    .apply(canonicalize_yarn_count)
)

yarn_count_prediction_data = (
    yarn_count_prediction_data
    .dropna(subset=["yarn_count_class"])
)

print(
    "ORME materials:",
    len(yarn_count_prediction_data)
)

print(
    "Yarn-count classes:",
    yarn_count_prediction_data[
        "yarn_count_class"
    ].nunique()
)

display(
    yarn_count_prediction_data[
        "yarn_count_class"
    ].value_counts().head(20)
)

In [ ]:
# Keep classes that have enough observations for stratified evaluation
class_counts = (
    yarn_count_prediction_data[
        "yarn_count_class"
    ].value_counts()
)

valid_classes = class_counts[
    class_counts >= 2
].index

yarn_count_model_data = (
    yarn_count_prediction_data[
        yarn_count_prediction_data[
            "yarn_count_class"
        ].isin(valid_classes)
    ]
    .copy()
)

train_yarn, test_yarn = train_test_split(
    yarn_count_model_data,
    test_size=0.20,
    random_state=42,
    stratify=yarn_count_model_data[
        "yarn_count_class"
    ]
)

yarn_count_text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

yarn_count_text_model.fit(
    train_yarn["retrieval_text"],
    train_yarn["yarn_count_class"]
)

In [ ]:
yarn_predictions = (
    yarn_count_text_model.predict(
        test_yarn["retrieval_text"]
    )
)

yarn_probabilities = (
    yarn_count_text_model.predict_proba(
        test_yarn["retrieval_text"]
    )
)

yarn_classes = (
    yarn_count_text_model
    .named_steps["model"]
    .classes_
)

print(
    "Top-1 yarn-count accuracy:",
    f"{accuracy_score(test_yarn['yarn_count_class'], yarn_predictions):.2%}"
)

print(
    "Top-3 yarn-count accuracy:",
    f"{top_k_accuracy_score(
        test_yarn['yarn_count_class'],
        yarn_probabilities,
        k=min(3, len(yarn_classes)),
        labels=yarn_classes
    ):.2%}"
)

In [ ]:
print(
    "Materials used for evaluation:",
    len(yarn_count_model_data)
)

print(
    "Materials excluded due to rare class:",
    len(yarn_count_prediction_data)
    - len(yarn_count_model_data)
)

In [ ]:
# Retrain the yarn-count classifier using all available labeled data
final_yarn_count_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

final_yarn_count_model.fit(
    yarn_count_model_data["retrieval_text"],
    yarn_count_model_data["yarn_count_class"]
)

In [ ]:
# Predict Top-3 yarn counts for no-attribute knitted fabrics
sap_no_attr_orme = (
    sap_no_attr_dev[
        sap_no_attr_dev["material_group_code"] == "1020001"
    ]
    .copy()
)

yarn_probabilities = (
    final_yarn_count_model.predict_proba(
        sap_no_attr_orme["retrieval_text"]
    )
)

yarn_classes = (
    final_yarn_count_model
    .named_steps["model"]
    .classes_
)

top_indices = np.argsort(
    yarn_probabilities,
    axis=1
)[:, -3:][:, ::-1]

predicted_yarn_records = []

for row_idx, material_id in enumerate(
    sap_no_attr_orme["material_id"]
):
    for rank, class_idx in enumerate(
        top_indices[row_idx],
        start=1
    ):
        predicted_yarn_records.append({
            "material_id": material_id,
            "predicted_yarn_count":
                yarn_classes[class_idx],
            "yarn_rank": rank,
            "yarn_probability":
                yarn_probabilities[row_idx, class_idx]
        })

predicted_yarn_counts = pd.DataFrame(
    predicted_yarn_records
)

display(predicted_yarn_counts.head(10))

In [ ]:
# Prepare canonical PLM knitting yarn counts
plm_yarn_count_retrieval = (
    plm_yarn_count_knit[
        ["plm_code", "plm_yarn_count"]
    ]
    .copy()
)

plm_yarn_count_retrieval["yarn_count_class"] = (
    plm_yarn_count_retrieval["plm_yarn_count"]
    .apply(canonicalize_yarn_count)
)

plm_yarn_count_retrieval = (
    plm_yarn_count_retrieval
    .dropna(subset=["yarn_count_class"])
    .drop_duplicates("plm_code")
)

In [ ]:
# Build PLM pool indexed by structure and yarn count
plm_orme_structure_yarn = (
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "structured_text"
        ]
    ]
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="inner"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_orme_structure_yarn = (
    plm_orme_structure_yarn[
        plm_orme_structure_yarn[
            "material_group_code"
        ] == "1020001"
    ]
    .copy()
)

In [ ]:
# Combine predicted structures and yarn counts
orme_structure_predictions = (
    predicted_structures[
        predicted_structures["material_group_code"]
        == "1020001"
    ][
        [
            "material_id",
            "predicted_structure",
            "structure_rank",
            "structure_probability"
        ]
    ]
)

prediction_combinations = (
    orme_structure_predictions
    .merge(
        predicted_yarn_counts,
        on="material_id",
        how="inner"
    )
)

print(
    "Prediction combinations:",
    len(prediction_combinations)
)

In [ ]:
def generate_structure_yarn_candidates(
    sap_df,
    prediction_df,
    plm_df,
    top_k_per_combination=20
):
    records = []

    sap_lookup = (
        sap_df
        .set_index("material_id")["retrieval_text"]
        .to_dict()
    )

    for (
        structure_value,
        yarn_value
    ), predictions in prediction_df.groupby(
        [
            "predicted_structure",
            "predicted_yarn_count"
        ]
    ):

        plm_pool = (
            plm_df[
                (plm_df["structure"] == structure_value)
                & (
                    plm_df["yarn_count_class"]
                    == yarn_value
                )
            ]
            .reset_index(drop=True)
        )

        if plm_pool.empty:
            continue

        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            norm="l2"
        )

        plm_matrix = vectorizer.fit_transform(
            plm_pool["structured_text"]
        )

        n_neighbors = min(
            top_k_per_combination,
            len(plm_pool)
        )

        nn_model = NearestNeighbors(
            n_neighbors=n_neighbors,
            metric="cosine",
            algorithm="brute"
        )

        nn_model.fit(plm_matrix)

        material_ids = [
            material_id
            for material_id in predictions["material_id"]
            if material_id in sap_lookup
        ]

        if not material_ids:
            continue

        sap_texts = [
            sap_lookup[material_id]
            for material_id in material_ids
        ]

        sap_matrix = vectorizer.transform(
            sap_texts
        )

        distances, indices = nn_model.kneighbors(
            sap_matrix
        )

        for row_idx, material_id in enumerate(
            material_ids
        ):
            prediction_row = (
                predictions[
                    predictions["material_id"]
                    == material_id
                ]
                .iloc[0]
            )

            for retrieval_rank, (
                plm_idx,
                distance
            ) in enumerate(
                zip(
                    indices[row_idx],
                    distances[row_idx]
                ),
                start=1
            ):
                records.append({
                    "material_id": material_id,
                    "candidate_plm_code":
                        plm_pool.loc[
                            plm_idx,
                            "plm_code"
                        ],
                    "predicted_structure":
                        structure_value,
                    "predicted_yarn_count":
                        yarn_value,
                    "structure_rank":
                        prediction_row[
                            "structure_rank"
                        ],
                    "yarn_rank":
                        prediction_row[
                            "yarn_rank"
                        ],
                    "retrieval_rank":
                        retrieval_rank,
                    "retrieval_similarity":
                        1 - distance
                })

    return pd.DataFrame(records)

In [ ]:
pseudo_structure_yarn_candidates = (
    generate_structure_yarn_candidates(
        sap_no_attr_orme,
        prediction_combinations,
        plm_orme_structure_yarn,
        top_k_per_combination=20
    )
)

pseudo_structure_yarn_pairs = (
    pseudo_structure_yarn_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Structure + yarn candidate pairs:",
    len(pseudo_structure_yarn_pairs)
)

print(
    "Average candidates per covered SAP:",
    len(pseudo_structure_yarn_pairs)
    / pseudo_structure_yarn_pairs[
        "material_id"
    ].nunique()
)

In [ ]:
orme_dev_truth = (
    no_attr_dev_truth[
        no_attr_dev_truth["material_id"]
        .isin(set(sap_no_attr_orme["material_id"]))
    ]
)

orme_structure_yarn_recall = (
    orme_dev_truth
    .merge(
        pseudo_structure_yarn_pairs,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

print(
    "Structure + yarn ORME recall:",
    f"{orme_structure_yarn_recall['candidate_plm_code'].notna().mean():.2%}"
)

In [ ]:
# Add structure+yarn channel to the existing no-attribute candidate pool
no_attr_candidates_v2 = (
    pd.concat(
        [
            no_attr_combined_candidates,
            pseudo_structure_yarn_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

no_attr_recall_v2 = (
    no_attr_dev_truth
    .merge(
        no_attr_candidates_v2,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_recall_v2["retrieved"] = (
    no_attr_recall_v2[
        "candidate_plm_code"
    ].notna()
)

print(
    "Overall no-attribute recall v2:",
    f"{no_attr_recall_v2['retrieved'].mean():.2%}"
)

In [ ]:
display(
    no_attr_recall_v2
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

In [ ]:
# Audit yarn-count availability on true PLM codes
orme_truth_yarn_audit = (
    orme_dev_truth
    .merge(
        plm_yarn_count_retrieval[
            [
                "plm_code",
                "yarn_count_class"
            ]
        ],
        on="plm_code",
        how="left"
    )
)

print(
    "ORME development materials:",
    len(orme_truth_yarn_audit)
)

print(
    "True PLM with yarn count:",
    f"{orme_truth_yarn_audit['yarn_count_class'].notna().mean():.2%}"
)

In [ ]:
# Check whether predicted Top-3 yarn counts contain the true PLM yarn count
predicted_yarn_sets = (
    predicted_yarn_counts
    .groupby("material_id")[
        "predicted_yarn_count"
    ]
    .apply(set)
    .rename("predicted_yarn_set")
    .reset_index()
)

orme_yarn_prediction_audit = (
    orme_truth_yarn_audit
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
)

orme_yarn_prediction_audit[
    "true_yarn_in_top3"
] = (
    orme_yarn_prediction_audit.apply(
        lambda row:
            (
                row["yarn_count_class"]
                in row["predicted_yarn_set"]
            )
            if (
                pd.notna(row["yarn_count_class"])
                and isinstance(
                    row["predicted_yarn_set"],
                    set
                )
            )
            else np.nan,
        axis=1
    )
)

valid_yarn_audit = (
    orme_yarn_prediction_audit[
        orme_yarn_prediction_audit[
            "true_yarn_in_top3"
        ].notna()
    ]
)

print(
    "Comparable materials:",
    len(valid_yarn_audit)
)

print(
    "Predicted yarn Top-3 agreement with true PLM:",
    f"{valid_yarn_audit['true_yarn_in_top3'].mean():.2%}"
)

In [ ]:
# Get true PLM fabric structure
orme_truth_structure_audit = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "structure"
            ]
        ].rename(
            columns={
                "structure":
                "true_plm_structure"
            }
        ),
        on="plm_code",
        how="left"
    )
)

predicted_structure_sets = (
    predicted_structures[
        predicted_structures[
            "material_group_code"
        ] == "1020001"
    ]
    .groupby("material_id")[
        "predicted_structure"
    ]
    .apply(set)
    .rename("predicted_structure_set")
    .reset_index()
)

orme_structure_prediction_audit = (
    orme_truth_structure_audit
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
)

orme_structure_prediction_audit[
    "true_structure_in_top3"
] = (
    orme_structure_prediction_audit.apply(
        lambda row:
            (
                row["true_plm_structure"]
                in row["predicted_structure_set"]
            )
            if (
                pd.notna(row["true_plm_structure"])
                and isinstance(
                    row["predicted_structure_set"],
                    set
                )
            )
            else np.nan,
        axis=1
    )
)

valid_structure_audit = (
    orme_structure_prediction_audit[
        orme_structure_prediction_audit[
            "true_structure_in_top3"
        ].notna()
    ]
)

print(
    "Predicted structure Top-3 agreement with true PLM:",
    f"{valid_structure_audit['true_structure_in_top3'].mean():.2%}"
)

In [ ]:
# Audit whether both true PLM structure and yarn count
# are contained in the predicted Top-3 sets
orme_joint_audit = (
    orme_truth_yarn_audit[
        [
            "material_id",
            "plm_code",
            "yarn_count_class"
        ]
    ]
    .merge(
        orme_truth_structure_audit[
            [
                "material_id",
                "true_plm_structure"
            ]
        ],
        on="material_id",
        how="left"
    )
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
)

orme_joint_audit["true_yarn_in_top3"] = (
    orme_joint_audit.apply(
        lambda row:
            row["yarn_count_class"] in row["predicted_yarn_set"]
            if (
                pd.notna(row["yarn_count_class"])
                and isinstance(row["predicted_yarn_set"], set)
            )
            else np.nan,
        axis=1
    )
)

orme_joint_audit["true_structure_in_top3"] = (
    orme_joint_audit.apply(
        lambda row:
            row["true_plm_structure"] in row["predicted_structure_set"]
            if (
                pd.notna(row["true_plm_structure"])
                and isinstance(row["predicted_structure_set"], set)
            )
            else np.nan,
        axis=1
    )
)

comparable_joint = orme_joint_audit[
    orme_joint_audit["true_yarn_in_top3"].notna()
    & orme_joint_audit["true_structure_in_top3"].notna()
].copy()

comparable_joint["both_in_top3"] = (
    comparable_joint["true_yarn_in_top3"]
    & comparable_joint["true_structure_in_top3"]
)

print(
    "Comparable ORME materials:",
    len(comparable_joint)
)

print(
    "Both true structure and yarn in predicted Top-3:",
    f"{comparable_joint['both_in_top3'].mean():.2%}"
)

In [ ]:
# Test candidate-pool recall before text ranking
true_plm_attributes = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
)

true_plm_attributes["eligible_by_pseudo_attributes"] = (
    true_plm_attributes.apply(
        lambda row:
            (
                row["structure"]
                in row["predicted_structure_set"]
            )
            and (
                row["yarn_count_class"]
                in row["predicted_yarn_set"]
            )
            if (
                pd.notna(row["structure"])
                and pd.notna(row["yarn_count_class"])
                and isinstance(
                    row["predicted_structure_set"],
                    set
                )
                and isinstance(
                    row["predicted_yarn_set"],
                    set
                )
            )
            else False,
        axis=1
    )
)

print(
    "Pseudo-attribute pool recall:",
    f"{true_plm_attributes['eligible_by_pseudo_attributes'].mean():.2%}"
)

In [ ]:
# Build the full pseudo-attribute candidate pool
pseudo_attribute_candidates = (
    prediction_combinations[
        [
            "material_id",
            "predicted_structure",
            "structure_rank",
            "structure_probability",
            "predicted_yarn_count",
            "yarn_rank",
            "yarn_probability"
        ]
    ]
    .merge(
        plm_orme_structure_yarn[
            [
                "plm_code",
                "structure",
                "yarn_count_class"
            ]
        ],
        left_on=[
            "predicted_structure",
            "predicted_yarn_count"
        ],
        right_on=[
            "structure",
            "yarn_count_class"
        ],
        how="inner"
    )
    .rename(
        columns={
            "plm_code": "candidate_plm_code"
        }
    )
)

In [ ]:
# Keep one row per SAP-PLM pair
pseudo_attribute_candidates_clean = (
    pseudo_attribute_candidates
    .groupby(
        [
            "material_id",
            "candidate_plm_code"
        ],
        as_index=False
    )
    .agg(
        structure_probability=(
            "structure_probability",
            "max"
        ),
        yarn_probability=(
            "yarn_probability",
            "max"
        ),
        best_structure_rank=(
            "structure_rank",
            "min"
        ),
        best_yarn_rank=(
            "yarn_rank",
            "min"
        )
    )
)

pseudo_attribute_candidates_clean[
    "pseudo_attribute_score"
] = (
    pseudo_attribute_candidates_clean[
        "structure_probability"
    ]
    * pseudo_attribute_candidates_clean[
        "yarn_probability"
    ]
)

In [ ]:
print(
    "Pseudo-attribute candidate pairs:",
    len(pseudo_attribute_candidates_clean)
)

print(
    "Covered SAP materials:",
    pseudo_attribute_candidates_clean[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(pseudo_attribute_candidates_clean)
    / pseudo_attribute_candidates_clean[
        "material_id"
    ].nunique()
)

In [ ]:
# Combine text retrieval and pseudo-attribute candidate generation
no_attr_orme_candidate_union = (
    pd.concat(
        [
            multi_channel_union[
                multi_channel_union["material_id"]
                .isin(set(sap_no_attr_orme["material_id"]))
            ],
            pseudo_attribute_candidates_clean[
                [
                    "material_id",
                    "candidate_plm_code"
                ]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

In [ ]:
orme_union_recall = (
    orme_dev_truth
    .merge(
        no_attr_orme_candidate_union,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

orme_union_recall["retrieved"] = (
    orme_union_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "ORME candidate union recall:",
    f"{orme_union_recall['retrieved'].mean():.2%}"
)

print(
    "Average union candidates per SAP:",
    len(no_attr_orme_candidate_union)
    / no_attr_orme_candidate_union[
        "material_id"
    ].nunique()
)

In [ ]:
# Final no-attribute candidate union for development
no_attr_candidate_union_v3 = (
    pd.concat(
        [
            no_attr_combined_candidates,
            pseudo_attribute_candidates_clean[
                ["material_id", "candidate_plm_code"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

print(
    "Final no-attribute candidate pairs:",
    len(no_attr_candidate_union_v3)
)

print(
    "Average candidates per SAP:",
    len(no_attr_candidate_union_v3)
    / no_attr_candidate_union_v3["material_id"].nunique()
)

In [ ]:
# Aggregate text retrieval features
text_pair_features = (
    multi_channel_candidates
    .groupby(
        [
            "material_id",
            "candidate_plm_code",
            "channel"
        ],
        as_index=False
    )
    .agg(
        similarity=("channel_similarity", "max"),
        rank=("channel_rank", "min")
    )
)

text_similarity_wide = (
    text_pair_features
    .pivot(
        index=["material_id", "candidate_plm_code"],
        columns="channel",
        values="similarity"
    )
    .add_suffix("_similarity")
    .reset_index()
)

text_rank_wide = (
    text_pair_features
    .pivot(
        index=["material_id", "candidate_plm_code"],
        columns="channel",
        values="rank"
    )
    .add_suffix("_rank")
    .reset_index()
)

text_channel_count = (
    text_pair_features
    .groupby(
        ["material_id", "candidate_plm_code"]
    )["channel"]
    .nunique()
    .rename("text_channel_count")
    .reset_index()
)

In [ ]:
# Aggregate pseudo-structure retrieval features
pseudo_structure_features = (
    pseudo_structure_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        structure_probability=(
            "structure_probability",
            "max"
        ),
        best_structure_rank=(
            "structure_rank",
            "min"
        ),
        structure_retrieval_similarity=(
            "structure_retrieval_similarity",
            "max"
        ),
        structure_retrieval_rank=(
            "structure_retrieval_rank",
            "min"
        )
    )
)

pseudo_structure_features[
    "from_pseudo_structure"
] = 1

In [ ]:
pseudo_attribute_features = (
    pseudo_attribute_candidates_clean.copy()
)

pseudo_attribute_features[
    "from_structure_yarn"
] = 1

In [ ]:
# Build one-row-per-SAP-PLM candidate master
no_attr_candidate_master = (
    no_attr_candidate_union_v3
    .merge(
        text_similarity_wide,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        text_rank_wide,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        text_channel_count,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        pseudo_structure_features,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        pseudo_attribute_features,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
)

for column in [
    "text_channel_count",
    "from_pseudo_structure",
    "from_structure_yarn"
]:
    no_attr_candidate_master[column] = (
        no_attr_candidate_master[column]
        .fillna(0)
    )

assert not no_attr_candidate_master.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

In [ ]:
# Add material group
no_attr_candidate_master = (
    no_attr_candidate_master
    .merge(
        material_group_lookup,
        on="material_id",
        how="left",
        validate="many_to_one"
    )
)

# Add known target PLM for development materials only
no_attr_dev_truth_labeled = (
    no_attr_dev_truth
    .rename(
        columns={
            "plm_code": "true_plm_code"
        }
    )
)

no_attr_dev_pairs = (
    no_attr_candidate_master[
        no_attr_candidate_master["material_id"]
        .isin(no_attr_dev_ids)
    ]
    .merge(
        no_attr_dev_truth_labeled,
        on="material_id",
        how="inner",
        validate="many_to_one"
    )
)

no_attr_dev_pairs["is_match"] = (
    no_attr_dev_pairs["candidate_plm_code"]
    == no_attr_dev_pairs["true_plm_code"]
).astype(int)

In [ ]:
print(
    "Development SAP materials:",
    no_attr_dev_pairs["material_id"].nunique()
)

print(
    "Candidate pairs:",
    len(no_attr_dev_pairs)
)

print(
    "Positive pairs retrieved:",
    no_attr_dev_pairs["is_match"].sum()
)

print(
    "Candidate recall:",
    f"{(
        no_attr_dev_pairs
        .groupby('material_id')['is_match']
        .max()
        .mean()
    ):.2%}"
)

In [ ]:
candidate_feature_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if (
        column.endswith("_similarity")
        or column.endswith("_rank")
        or column in [
            "text_channel_count",
            "structure_probability",
            "yarn_probability",
            "pseudo_attribute_score",
            "from_pseudo_structure",
            "from_structure_yarn"
        ]
    )
]

print(candidate_feature_columns)

In [ ]:
# Inspect duplicated pseudo-structure feature names
print([
    column
    for column in no_attr_dev_pairs.columns
    if "structure_probability" in column
    or "best_structure_rank" in column
])

In [ ]:
# Consolidate structure probability from multiple candidate channels
structure_probability_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if column.startswith("structure_probability")
]

structure_rank_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if column.startswith("best_structure_rank")
]

no_attr_dev_pairs["structure_probability"] = (
    no_attr_dev_pairs[
        structure_probability_columns
    ]
    .max(axis=1)
)

no_attr_dev_pairs["best_structure_rank"] = (
    no_attr_dev_pairs[
        structure_rank_columns
    ]
    .min(axis=1)
)

In [ ]:
# Add material-group indicators
for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    no_attr_dev_pairs[
        f"group_{group_code}"
    ] = (
        no_attr_dev_pairs[
            "material_group_code"
        ] == group_code
    ).astype(int)

In [ ]:
no_attr_features = [
    "structured_text_similarity",
    "enriched_text_similarity",
    "multi_value_text_similarity",
    "original_text_similarity",

    "structure_retrieval_similarity",
    "structure_probability",
    "best_structure_rank",

    "yarn_probability",
    "best_yarn_rank",
    "pseudo_attribute_score",

    "text_channel_count",
    "from_pseudo_structure",
    "from_structure_yarn",

    "group_1020001",
    "group_1020002",
    "group_1030004"
]

In [ ]:
# Create material-level development split
no_attr_dev_material_table = (
    no_attr_development[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .copy()
)

no_attr_train_materials, no_attr_validation_materials = (
    train_test_split(
        no_attr_dev_material_table,
        test_size=0.20,
        random_state=777,
        stratify=no_attr_dev_material_table[
            "material_group_code"
        ]
    )
)

no_attr_train_ids = set(
    no_attr_train_materials["material_id"]
)

no_attr_validation_ids = set(
    no_attr_validation_materials["material_id"]
)

print(
    "Train materials:",
    len(no_attr_train_ids)
)

print(
    "Validation materials:",
    len(no_attr_validation_ids)
)

In [ ]:
# Identify train materials with a retrieved positive candidate
train_pair_pool = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_train_ids)
    ]
    .copy()
)

retrieved_train_ids = set(
    train_pair_pool
    .groupby("material_id")["is_match"]
    .max()
    .loc[lambda x: x == 1]
    .index
)

train_pair_pool = (
    train_pair_pool[
        train_pair_pool["material_id"]
        .isin(retrieved_train_ids)
    ]
    .copy()
)

print(
    "Train materials with true PLM retrieved:",
    len(retrieved_train_ids)
)

In [ ]:
# Score used only for hard-negative selection
train_pair_pool["hardness_score"] = (
    train_pair_pool[
        "structured_text_similarity"
    ].fillna(0)

    + train_pair_pool[
        "enriched_text_similarity"
    ].fillna(0)

    + train_pair_pool[
        "structure_retrieval_similarity"
    ].fillna(0)

    + train_pair_pool[
        "pseudo_attribute_score"
    ].fillna(0)

    + 0.25 * train_pair_pool[
        "multi_value_text_similarity"
    ].fillna(0)
)

positive_pairs = (
    train_pair_pool[
        train_pair_pool["is_match"] == 1
    ]
    .copy()
)

negative_pairs = (
    train_pair_pool[
        train_pair_pool["is_match"] == 0
    ]
    .copy()
)

hard_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(30)
)

rng = np.random.default_rng(42)

negative_pairs["random_score"] = (
    rng.random(len(negative_pairs))
)

random_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

no_attr_training_sample = (
    pd.concat(
        [
            positive_pairs,
            hard_negatives,
            random_negatives
        ],
        ignore_index=True
    )
    .drop_duplicates(
        ["material_id", "candidate_plm_code"]
    )
)

print(
    "Training rows:",
    len(no_attr_training_sample)
)

print(
    "Training positives:",
    no_attr_training_sample[
        "is_match"
    ].sum()
)

In [ ]:
# Train first no-attribute reranker
no_attr_rf_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True,
                    keep_empty_features=True
                )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        )
    )
])

no_attr_rf_model.fit(
    no_attr_training_sample[
        no_attr_features
    ],
    no_attr_training_sample[
        "is_match"
    ]
)


In [ ]:
# Score the full validation candidate pool
no_attr_validation_pairs = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_validation_ids)
    ]
    .copy()
)

no_attr_validation_pairs["score"] = (
    no_attr_rf_model.predict_proba(
        no_attr_validation_pairs[
            no_attr_features
        ]
    )[:, 1]
)

no_attr_validation_pairs["rank"] = (
    no_attr_validation_pairs
    .groupby("material_id")["score"]
    .rank(
        ascending=False,
        method="first"
    )
)

In [ ]:
validation_true_ranks = (
    no_attr_validation_pairs[
        no_attr_validation_pairs["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "rank"
        ]
    ]
    .copy()
)

print(
    "Conditional Top-1:",
    f"{(validation_true_ranks['rank'] <= 1).mean():.2%}"
)

print(
    "Conditional Top-3:",
    f"{(validation_true_ranks['rank'] <= 3).mean():.2%}"
)

print(
    "Conditional Top-5:",
    f"{(validation_true_ranks['rank'] <= 5).mean():.2%}"
)

print(
    "Conditional Top-10:",
    f"{(validation_true_ranks['rank'] <= 10).mean():.2%}"
)

print(
    "Conditional MRR:",
    f"{(1 / validation_true_ranks['rank']).mean():.4f}"
)

In [ ]:
# End-to-end validation evaluation
validation_rank_table = (
    no_attr_validation_materials[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        validation_true_ranks[
            [
                "material_id",
                "rank"
            ]
        ],
        on="material_id",
        how="left"
    )
)

print(
    "Validation candidate recall:",
    f"{validation_rank_table['rank'].notna().mean():.2%}"
)

for k in [1, 3, 5, 10]:
    success = (
        validation_rank_table["rank"]
        .le(k)
        .fillna(False)
        .mean()
    )

    print(
        f"End-to-end Top-{k}:",
        f"{success:.2%}"
    )

In [ ]:
# Evaluate no-attribute reranker by material group
group_results = []

for group_code, group in validation_rank_table.groupby(
    "material_group_code"
):
    retrieved = group[group["rank"].notna()]

    group_results.append({
        "material_group_code": group_code,
        "materials": len(group),

        "candidate_recall":
            group["rank"].notna().mean(),

        "conditional_top1":
            (retrieved["rank"] <= 1).mean(),

        "conditional_top3":
            (retrieved["rank"] <= 3).mean(),

        "conditional_top5":
            (retrieved["rank"] <= 5).mean(),

        "end_to_end_top5":
            group["rank"]
            .le(5)
            .fillna(False)
            .mean()
    })

display(pd.DataFrame(group_results))

In [ ]:
# Inspect feature importance
imputer = no_attr_rf_model.named_steps["imputer"]
rf_model = no_attr_rf_model.named_steps["model"]

feature_names = imputer.get_feature_names_out(
    no_attr_features
)

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_model.feature_importances_
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

display(feature_importance.head(25))

In [ ]:
# Build a canonical fiber lexicon
fiber_alias_mapping = {
    "PAMUK": "COTTON",
    "COTTON": "COTTON",

    "PES": "POLYESTER",
    "POLYESTER": "POLYESTER",
    "POLIESTER": "POLYESTER",

    "VIS": "VISCOSE",
    "VISCOSE": "VISCOSE",

    "EA": "ELASTANE",
    "ELASTAN": "ELASTANE",
    "ELASTANE": "ELASTANE",
    "ELASTHANE": "ELASTANE",
    "SPANDEX": "ELASTANE",

    "PA": "POLYAMIDE",
    "POLYAMIDE": "POLYAMIDE",
    "POLYAMIDE6": "POLYAMIDE",

    "LYOCELL": "LYOCELL",
    "TENCEL": "LYOCELL",

    "ACRYLIC": "ACRYLIC",
    "AKRILIK": "ACRYLIC",

    "LINEN": "LINEN",
    "KETEN": "LINEN",

    "WOOL": "WOOL",
    "YUN": "WOOL",
    "YÜN": "WOOL"
}

In [ ]:
import re

def extract_fiber_names_from_text(text):
    if pd.isna(text):
        return set()

    text = str(text).upper()

    found = set()

    for alias, canonical in fiber_alias_mapping.items():

        # Short abbreviations need token boundaries
        pattern = rf"(?<![A-ZÇĞİÖŞÜ]){re.escape(alias)}(?![A-ZÇĞİÖŞÜ])"

        if re.search(pattern, text):
            found.add(canonical)

    return found

In [ ]:
# Compare description-derived fibers with actual SAP fiber attributes
fiber_extraction_audit = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="inner"
    )
)

fiber_extraction_audit[
    "predicted_fiber_set"
] = (
    fiber_extraction_audit[
        "retrieval_text"
    ]
    .apply(extract_fiber_names_from_text)
)

fiber_extraction_audit[
    "actual_fiber_set"
] = (
    fiber_extraction_audit[
        "fiber_key"
    ]
    .apply(set)
)

fiber_extraction_audit[
    "fiber_set_match"
] = (
    fiber_extraction_audit[
        "predicted_fiber_set"
    ]
    == fiber_extraction_audit[
        "actual_fiber_set"
    ]
)

fiber_extraction_audit[
    "fiber_jaccard"
] = [
    (
        len(pred & actual)
        / len(pred | actual)
        if pred and actual
        else np.nan
    )
    for pred, actual in zip(
        fiber_extraction_audit[
            "predicted_fiber_set"
        ],
        fiber_extraction_audit[
            "actual_fiber_set"
        ]
    )
]

In [ ]:
print(
    "Materials:",
    len(fiber_extraction_audit)
)

print(
    "Description with at least one extracted fiber:",
    f"{fiber_extraction_audit['predicted_fiber_set'].apply(bool).mean():.2%}"
)

print(
    "Exact fiber-set match:",
    f"{fiber_extraction_audit['fiber_set_match'].mean():.2%}"
)

print(
    "Mean fiber Jaccard:",
    f"{fiber_extraction_audit['fiber_jaccard'].mean():.3f}"
)

In [ ]:
display(
    fiber_extraction_audit
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        extraction_coverage=(
            "predicted_fiber_set",
            lambda x: x.apply(bool).mean()
        ),
        exact_match=(
            "fiber_set_match",
            "mean"
        ),
        mean_jaccard=(
            "fiber_jaccard",
            "mean"
        )
    )
)

In [ ]:
# Inspect validation candidate-pool size by material group
validation_candidate_stats = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_validation_ids)
    ]
    .groupby(
        ["material_id", "material_group_code"],
        as_index=False
    )
    .agg(
        candidate_count=("candidate_plm_code", "size"),
        true_retrieved=("is_match", "max")
    )
)

candidate_pool_summary = (
    validation_candidate_stats
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        candidate_recall=("true_retrieved", "mean"),
        mean_candidates=("candidate_count", "mean"),
        median_candidates=("candidate_count", "median"),
        p90_candidates=(
            "candidate_count",
            lambda x: x.quantile(0.90)
        )
    )
)

display(candidate_pool_summary)

In [ ]:
group_specific_features = [
    feature
    for feature in no_attr_features
    if not feature.startswith("group_")
]

print(group_specific_features)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


def build_no_attr_lr():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])


def build_no_attr_rf():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])


In [ ]:
def evaluate_group_reranker(
    group_code,
    model,
    model_name,
    features
):
    train_group = (
        no_attr_training_sample[
            (
                no_attr_training_sample["material_id"]
                .isin(no_attr_train_ids)
            )
            & (
                no_attr_training_sample[
                    "material_group_code"
                ] == group_code
            )
        ]
        .copy()
    )

    validation_group = (
        no_attr_dev_pairs[
            (
                no_attr_dev_pairs["material_id"]
                .isin(no_attr_validation_ids)
            )
            & (
                no_attr_dev_pairs[
                    "material_group_code"
                ] == group_code
            )
        ]
        .copy()
    )

    validation_materials_group = (
        no_attr_validation_materials[
            no_attr_validation_materials[
                "material_group_code"
            ] == group_code
        ][["material_id"]]
        .copy()
    )

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    validation_group["score"] = (
        model.predict_proba(
            validation_group[features]
        )[:, 1]
    )

    validation_group["rank"] = (
        validation_group
        .groupby("material_id")["score"]
        .rank(
            ascending=False,
            method="first"
        )
    )

    true_ranks = (
        validation_group[
            validation_group["is_match"] == 1
        ][
            ["material_id", "rank"]
        ]
        .copy()
    )

    evaluation_table = (
        validation_materials_group
        .merge(
            true_ranks,
            on="material_id",
            how="left",
            validate="one_to_one"
        )
    )

    retrieved = (
        evaluation_table[
            evaluation_table["rank"].notna()
        ]
    )

    result = {
        "material_group_code": group_code,
        "model": model_name,

        "validation_materials":
            len(evaluation_table),

        "candidate_recall":
            evaluation_table["rank"]
            .notna()
            .mean(),

        "conditional_top1":
            (retrieved["rank"] <= 1).mean(),

        "conditional_top3":
            (retrieved["rank"] <= 3).mean(),

        "conditional_top5":
            (retrieved["rank"] <= 5).mean(),

        "conditional_top10":
            (retrieved["rank"] <= 10).mean(),

        "conditional_mrr":
            (1 / retrieved["rank"]).mean(),

        "end_to_end_top1":
            evaluation_table["rank"]
            .le(1)
            .fillna(False)
            .mean(),

        "end_to_end_top3":
            evaluation_table["rank"]
            .le(3)
            .fillna(False)
            .mean(),

        "end_to_end_top5":
            evaluation_table["rank"]
            .le(5)
            .fillna(False)
            .mean(),

        "end_to_end_top10":
            evaluation_table["rank"]
            .le(10)
            .fillna(False)
            .mean()
    }

    return result, model, validation_group

In [ ]:
group_model_results = []

trained_group_models = {}
scored_group_validation = {}

group_codes = [
    "1020001",  # ORME
    "1020002"   # DOKUMA
]

for group_code in group_codes:

    lr_result, lr_model, lr_scored = (
        evaluate_group_reranker(
            group_code=group_code,
            model=build_no_attr_lr(),
            model_name="Logistic Regression",
            features=group_specific_features
        )
    )

    group_model_results.append(lr_result)

    trained_group_models[
        (group_code, "LR")
    ] = lr_model

    scored_group_validation[
        (group_code, "LR")
    ] = lr_scored


    rf_result, rf_model, rf_scored = (
        evaluate_group_reranker(
            group_code=group_code,
            model=build_no_attr_rf(),
            model_name="Random Forest",
            features=group_specific_features
        )
    )

    group_model_results.append(rf_result)

    trained_group_models[
        (group_code, "RF")
    ] = rf_model

    scored_group_validation[
        (group_code, "RF")
    ] = rf_scored

In [ ]:
group_model_results_df = (
    pd.DataFrame(group_model_results)
    .sort_values(
        [
            "material_group_code",
            "conditional_mrr"
        ],
        ascending=[True, False]
    )
)

display(group_model_results_df)

In [ ]:
rank_distribution_results = []

for (
    group_code,
    model_name
), scored_data in scored_group_validation.items():

    true_ranks = (
        scored_data[
            scored_data["is_match"] == 1
        ]["rank"]
    )

    rank_distribution_results.append({
        "material_group_code":
            group_code,

        "model":
            model_name,

        "median_rank":
            true_ranks.median(),

        "p75_rank":
            true_ranks.quantile(0.75),

        "p90_rank":
            true_ranks.quantile(0.90),

        "mean_rank":
            true_ranks.mean(),

        "max_rank":
            true_ranks.max()
    })

display(
    pd.DataFrame(
        rank_distribution_results
    )
)

In [ ]:
# Build true PLM structure + yarn attributes for ORME development materials
orme_oracle_truth = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="left"
    )
)

# Count how many PLM codes share the exact same structure + yarn combination
orme_pool_sizes = (
    plm_orme_structure_yarn
    .groupby(
        ["structure", "yarn_count_class"]
    )
    .size()
    .rename("oracle_pool_size")
    .reset_index()
)

orme_oracle_truth = (
    orme_oracle_truth
    .merge(
        orme_pool_sizes,
        on=["structure", "yarn_count_class"],
        how="left"
    )
)

display(
    orme_oracle_truth["oracle_pool_size"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

In [ ]:
print(
    "Mean oracle pool:",
    orme_oracle_truth["oracle_pool_size"].mean()
)

print(
    "Median oracle pool:",
    orme_oracle_truth["oracle_pool_size"].median()
)

print(
    "P90 oracle pool:",
    orme_oracle_truth["oracle_pool_size"].quantile(0.90)
)

In [ ]:
# Audit exact description ambiguity in no-attribute ORME
orme_description_truth = (
    sap_no_attr_dev[
        sap_no_attr_dev["material_group_code"] == "1020001"
    ][
        ["material_id", "retrieval_text"]
    ]
    .merge(
        orme_dev_truth[
            ["material_id", "plm_code"]
        ],
        on="material_id",
        how="inner"
    )
)

description_ambiguity = (
    orme_description_truth
    .groupby("retrieval_text")
    .agg(
        sap_materials=("material_id", "nunique"),
        mapped_plms=("plm_code", "nunique")
    )
    .reset_index()
)

print(
    "Unique descriptions:",
    len(description_ambiguity)
)

print(
    "Descriptions mapping to >1 PLM:",
    (
        description_ambiguity["mapped_plms"] > 1
    ).sum()
)

ambiguous_descriptions = set(
    description_ambiguity.loc[
        description_ambiguity["mapped_plms"] > 1,
        "retrieval_text"
    ]
)

materials_inside_ambiguous_descriptions = (
    orme_description_truth[
        orme_description_truth[
            "retrieval_text"
        ].isin(ambiguous_descriptions)
    ]["material_id"]
    .nunique()
)

print(
    "Materials inside ambiguous descriptions:",
    materials_inside_ambiguous_descriptions
)

In [ ]:
# How unique is the PLM structured representation?
orme_plm_text_ambiguity = (
    plm_candidates_enriched[
        plm_candidates_enriched[
            "material_group_code"
        ] == "1020001"
    ]
    .groupby("structured_text")
    .agg(
        plm_codes=("plm_code", "nunique")
    )
    .reset_index()
)

print(
    "ORME PLM codes:",
    plm_candidates_enriched[
        plm_candidates_enriched[
            "material_group_code"
        ] == "1020001"
    ]["plm_code"].nunique()
)

print(
    "Unique structured texts:",
    len(orme_plm_text_ambiguity)
)

print(
    "Structured texts shared by multiple PLMs:",
    (
        orme_plm_text_ambiguity["plm_codes"] > 1
    ).sum()
)

display(
    orme_plm_text_ambiguity[
        orme_plm_text_ambiguity["plm_codes"] > 1
    ]
    .sort_values(
        "plm_codes",
        ascending=False
    )
    .head(20)
)

In [ ]:
# Audit PLM master columns before duplicate detection
plm_column_audit = pd.DataFrame({
    "column": plm_codes.columns,
    "dtype": [
        str(plm_codes[column].dtype)
        for column in plm_codes.columns
    ],
    "non_null": [
        plm_codes[column].notna().sum()
        for column in plm_codes.columns
    ],
    "unique_values": [
        plm_codes[column].nunique(dropna=True)
        for column in plm_codes.columns
    ]
})

display(plm_column_audit)

In [ ]:
import re
import numpy as np
import pandas as pd


def normalize_duplicate_text(series):
    return (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .fillna("<MISSING>")
        .replace("", "<MISSING>")
    )


def normalize_duplicate_column(series):
    # Preserve numeric equivalence such as 150 and 150.0
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(
            series,
            errors="coerce"
        )

        return (
            numeric
            .round(8)
            .astype("string")
            .fillna("<MISSING>")
        )

    return normalize_duplicate_text(series)

In [ ]:
# Inspect PLM-code collisions introduced by normalization
plm_code_collision_audit = (
    plm_codes[
        ["PLM Kodu"]
    ]
    .assign(
        normalized_plm_code=(
            plm_codes["PLM Kodu"]
            .astype("string")
            .str.strip()
            .str.upper()
            .str.replace(r"\.0$", "", regex=True)
        )
    )
)

collision_codes = (
    plm_code_collision_audit
    .groupby("normalized_plm_code")
    .agg(
        raw_code_count=("PLM Kodu", "nunique"),
        raw_codes=(
            "PLM Kodu",
            lambda x: list(x.astype(str).unique())
        )
    )
    .query("raw_code_count > 1")
)

display(collision_codes)

In [ ]:
def normalize_plm_code_identity(series):
    values = (
        series
        .astype("string")
        .str.strip()
    )

    # Normalize Excel numeric rendering (e.g. 12345.0 -> 12345)
    # without changing case for alphanumeric identifiers.
    return values.str.replace(
        r"^([+-]?\d+)\.0+$",
        r"\1",
        regex=True
    )


In [ ]:
plm_duplicate_base = plm_codes.copy()

plm_duplicate_base["plm_code"] = (
    normalize_plm_code_identity(
        plm_duplicate_base["PLM Kodu"]
    )
)

print(
    "Rows:",
    len(plm_duplicate_base)
)

print(
    "Unique PLM codes:",
    plm_duplicate_base["plm_code"].nunique()
)

assert plm_duplicate_base["plm_code"].is_unique

In [ ]:
mv_duplicate = plm_multi_value_valid.copy()

mv_duplicate["plm_code"] = (
    normalize_plm_code_identity(
        mv_duplicate["plm_code"]
    )
)

mv_duplicate["characteristic"] = (
    normalize_duplicate_text(
        mv_duplicate[
            "PLM Karakteristik Tanımı"
        ]
    )
)

mv_duplicate["value"] = (
    normalize_duplicate_text(
        mv_duplicate[
            "PLM Karakteristik Değeri"
        ]
    )
)

In [ ]:
master_codes = set(
    plm_duplicate_base["plm_code"]
)

multi_value_codes = set(
    mv_duplicate["plm_code"]
)

print(
    "Master PLM codes:",
    len(master_codes)
)

print(
    "Multi-value PLM codes:",
    len(multi_value_codes)
)

print(
    "Multi-value codes found in master:",
    len(
        multi_value_codes
        & master_codes
    )
)

print(
    "Multi-value codes NOT found in master:",
    len(
        multi_value_codes
        - master_codes
    )
)

In [ ]:
duplicate_excluded_columns = {
    # Identity
    "PLM Kodu",
    "plm_code",

    # Descriptive fields
    "Türkçe malzeme açıklaması",
    "Malzeme Türkçe Adı",
    "Malzeme ingilizce adı",

    # Administrative / derived fields
    "Malzeme statüs",
    "material_group_code",
    "material_group_name",
    "material_family"
}

structured_duplicate_features = [
    column
    for column in plm_duplicate_base.columns
    if column not in duplicate_excluded_columns
]

print(
    "Structured duplicate features:",
    len(structured_duplicate_features)
)

print(structured_duplicate_features)

In [ ]:
structured_signature = (
    plm_duplicate_base[
        ["plm_code"] + structured_duplicate_features
    ]
    .copy()
)

for column in structured_duplicate_features:
    structured_signature[column] = (
        normalize_duplicate_column(
            structured_signature[column]
        )
    )

assert structured_signature["plm_code"].is_unique

print(
    "Structured PLM rows:",
    len(structured_signature)
)

In [ ]:
mv_characteristic_sets = (
    mv_duplicate
    .groupby(
        [
            "plm_code",
            "characteristic"
        ]
    )["value"]
    .agg(
        lambda values:
            " || ".join(
                sorted(set(values))
            )
    )
    .reset_index()
)

mv_signature_wide = (
    mv_characteristic_sets
    .pivot(
        index="plm_code",
        columns="characteristic",
        values="value"
    )
    .fillna("<ABSENT>")
    .reset_index()
)

mv_feature_columns = [
    column
    for column in mv_signature_wide.columns
    if column != "plm_code"
]

mv_signature_wide = (
    mv_signature_wide.rename(
        columns={
            column: f"MV__{column}"
            for column in mv_feature_columns
        }
    )
)

assert mv_signature_wide["plm_code"].is_unique

print(
    "Multi-value PLM profiles:",
    len(mv_signature_wide)
)

print(
    "Multi-value characteristics:",
    len(mv_feature_columns)
)

In [ ]:
plm_duplicate_master = (
    structured_signature
    .merge(
        mv_signature_wide,
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

mv_columns = [
    column
    for column in plm_duplicate_master.columns
    if column.startswith("MV__")
]

plm_duplicate_master[mv_columns] = (
    plm_duplicate_master[mv_columns]
    .fillna("<ABSENT>")
)

assert plm_duplicate_master["plm_code"].is_unique
assert len(plm_duplicate_master) == len(structured_signature)

print(
    "Merged PLM rows:",
    len(plm_duplicate_master)
)

print(
    "Structured features:",
    len(structured_duplicate_features)
)

print(
    "Multi-value features:",
    len(mv_columns)
)


In [ ]:
duplicate_feature_columns = (
    structured_duplicate_features
    + mv_columns
)

plm_duplicate_master[
    "material_profile_hash"
] = (
    pd.util.hash_pandas_object(
        plm_duplicate_master[
            duplicate_feature_columns
        ],
        index=False
    )
    .astype("uint64")
)

In [ ]:
profile_counts = (
    plm_duplicate_master[
        "material_profile_hash"
    ]
    .value_counts()
)

duplicate_hashes = set(
    profile_counts[
        profile_counts > 1
    ].index
)

exact_duplicate_plms = (
    plm_duplicate_master[
        plm_duplicate_master[
            "material_profile_hash"
        ].isin(duplicate_hashes)
    ]
    .copy()
)

print(
    "Total PLM codes:",
    len(plm_duplicate_master)
)

print(
    "Unique technical profiles:",
    plm_duplicate_master[
        "material_profile_hash"
    ].nunique()
)

print(
    "Exact duplicate groups:",
    len(duplicate_hashes)
)

print(
    "PLM codes inside exact duplicate groups:",
    len(exact_duplicate_plms)
)

print(
    "Duplicate PLM rate:",
    f"{len(exact_duplicate_plms) / len(plm_duplicate_master):.2%}"
)

In [ ]:
# Verify that every hash group really contains a single normalized feature profile
duplicate_verification = (
    exact_duplicate_plms
    .groupby("material_profile_hash")[
        duplicate_feature_columns
    ]
    .nunique(dropna=False)
    .max(axis=1)
)

print(
    "Groups with more than one distinct feature value:",
    (duplicate_verification > 1).sum()
)

assert (duplicate_verification == 1).all()


In [ ]:
duplicate_hash_order = (
    exact_duplicate_plms[
        "material_profile_hash"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

duplicate_group_mapping = {
    profile_hash: f"PLM_DUP_{index:05d}"
    for index, profile_hash in enumerate(
        duplicate_hash_order,
        start=1
    )
}

exact_duplicate_plms[
    "duplicate_group_id"
] = (
    exact_duplicate_plms[
        "material_profile_hash"
    ]
    .map(duplicate_group_mapping)
)

exact_duplicate_plms[
    "duplicate_group_size"
] = (
    exact_duplicate_plms
    .groupby("duplicate_group_id")[
        "plm_code"
    ]
    .transform("size")
)

In [ ]:
duplicate_review = (
    exact_duplicate_plms[
        [
            "duplicate_group_id",
            "duplicate_group_size",
            "plm_code",
            "material_profile_hash"
        ]
    ]
    .merge(
        plm_duplicate_base[
            [
                "plm_code",
                "Mal grubu",
                "Türkçe malzeme açıklaması",
                "Malzeme Türkçe Adı",
                "Malzeme ingilizce adı",
                "Malzeme statüs"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        [
            "duplicate_group_size",
            "duplicate_group_id",
            "plm_code"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

display(
    duplicate_review.head(100)
)

In [ ]:
duplicate_impact_by_group = (
    plm_duplicate_base[
        [
            "plm_code",
            "Mal grubu"
        ]
    ]
    .merge(
        exact_duplicate_plms[
            [
                "plm_code",
                "duplicate_group_id"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .groupby("Mal grubu")
    .agg(
        plm_codes=(
            "plm_code",
            "nunique"
        ),
        duplicate_plm_codes=(
            "duplicate_group_id",
            lambda x: x.notna().sum()
        ),
        duplicate_groups=(
            "duplicate_group_id",
            "nunique"
        )
    )
)

duplicate_impact_by_group[
    "duplicate_plm_rate"
] = (
    duplicate_impact_by_group[
        "duplicate_plm_codes"
    ]
    / duplicate_impact_by_group[
        "plm_codes"
    ]
)

display(
    duplicate_impact_by_group
    .sort_values(
        "duplicate_plm_rate",
        ascending=False
    )
)

In [ ]:
duplicate_group_summary = (
    exact_duplicate_plms
    .groupby("duplicate_group_id")
    .agg(
        duplicate_group_size=(
            "plm_code",
            "nunique"
        )
    )
    .reset_index()
)

display(
    duplicate_group_summary[
        "duplicate_group_size"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
# Audit whether PLM codes used by SAP exist in the PLM master
sap_used_plm = (
    sap_plm_mapping[
        ["PLM Kodu"]
    ]
    .dropna()
    .copy()
)

sap_used_plm["plm_code"] = (
    normalize_plm_code_identity(
        sap_used_plm["PLM Kodu"]
    )
)

sap_used_unique = (
    sap_used_plm[
        ["plm_code"]
    ]
    .drop_duplicates()
    .merge(
        plm_duplicate_base[
            [
                "plm_code",
                "Mal grubu"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)


In [ ]:
print(
    "Unique SAP-used PLM codes:",
    len(sap_used_unique)
)

print(
    "Found in PLM master:",
    sap_used_unique["Mal grubu"]
    .notna()
    .sum()
)

print(
    "Not found in PLM master:",
    sap_used_unique["Mal grubu"]
    .isna()
    .sum()
)